# GCDD HGCTM-S workflow

This notebook contains the code workflow for the **HGCTM-S analysis of the Global Copper Deposit Database (GCDD)**.

## Notes for public GitHub use
- This version is cleaned for sharing.
- Cell outputs and execution counts were removed.
- Several cells still use **Kaggle-specific paths** such as `/kaggle/input/...` and `/kaggle/working/...`.
- Before running outside Kaggle, replace those paths with your own local or repository paths.
- External data sources such as **Macrostrat** should be documented in the repository README for reproducibility.

## Suggested GitHub companion files
- `README.md`
- `requirements.txt`
- `data/README.md` explaining which input data are required and where they come from


## Environment setup
Imports core libraries and prints available Kaggle input files.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## CELL 0 — Data load

Loads the raw GCDD master spreadsheet and prints the full
column inventory. This is the starting point for Stage 0
(dataset freeze) and Stage 1 (data viability) of the HGCTM
experimental protocol.

INPUT:  Global_Copper_Deposit_Main.xlsx (Kaggle input)
OUTPUT: df (in memory) — raw GCDD deposit table

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

RAW_SRC = "/kaggle/input/datasets/katerynaglin/global-copper-deposit-dataset-main"
MASTER_XLSX = f"{RAW_SRC}/Global_Copper_Deposit_Main.xlsx"

df = pd.read_excel(MASTER_XLSX, sheet_name="Sheet1")
df["Deposit_type"] = df["Deposit_type"].replace("Porphyry Cu-Mo", "Porphyry")

print(f"Shape: {df.shape[0]} deposits × {df.shape[1]} columns")
print(f"\nColumns and dtypes:")
for c in df.columns:
    n_miss = df[c].isna().sum()
    pct = 100 * n_miss / len(df)
    print(f"  {c:<30s}  {str(df[c].dtype):<12s}  missing: {n_miss:>5d} ({pct:5.1f}%)")

## CELL 1 — Deposit types

Checks class balance across the five copper deposit types.
This directly addresses HGCTM protocol Stage 1: are all
target classes viable for topic modelling and downstream
classification? Reports counts, proportions, and flags
any class with fewer than 30 deposits.

INPUT:  df (from Cell 0)
OUTPUT: console summary

In [ ]:
print("=== Deposit-type distribution ===\n")

dtype_counts = (
    df["Deposit_type"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "type", "Deposit_type": "type", "count": "n"})
)
# Handle both old and new pandas value_counts output format
if "type" not in dtype_counts.columns:
    dtype_counts.columns = ["type", "n"]

dtype_counts["pct"] = (100 * dtype_counts["n"] / dtype_counts["n"].sum()).round(1)
dtype_counts["viable"] = dtype_counts["n"].apply(
    lambda x: "OK" if x >= 30 else "WARN: <30"
)

print(dtype_counts.to_string(index=False))
print(f"\nTotal deposits: {dtype_counts['n'].sum()}")
print(f"Number of types: {len(dtype_counts)}")

n_small = (dtype_counts["n"] < 30).sum()
if n_small > 0:
    print(f"\n⚠ {n_small} class(es) have <30 deposits — may need merging")
else:
    print("\n✓ All classes have ≥30 deposits")


## CELL 2 — Assemblage parsing

Parses the Mineral_assemblage field into per-deposit mineral
lists. Reports the distribution of list lengths (number of
minerals per deposit), the fraction of empty records, and
the total vocabulary size. This is a critical viability check:
if most deposits have very short lists, topic modelling will
struggle. Also identifies the sparsity structure that
motivates the Dirichlet-Multinomial likelihood in the HGCTM.

INPUT:  df (from Cell 0)
OUTPUT: df gains column "mineral_list" and "n_minerals"

In [ ]:
def parse_minerals(x):
    """Split comma-separated assemblage string into list of mineral names."""
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(",") if t.strip()]

df["mineral_list"] = df["Mineral_assemblage"].apply(parse_minerals)
df["n_minerals"] = df["mineral_list"].apply(len)

print("=== Mineral assemblage length distribution ===\n")
desc = df["n_minerals"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
for k, v in desc.items():
    print(f"  {k:<8s}: {v:8.1f}")

n_empty = (df["n_minerals"] == 0).sum()
n_short = (df["n_minerals"] < 3).sum()
n_ok = (df["n_minerals"] >= 5).sum()

print(f"\n  Empty assemblages (0 minerals):  {n_empty:>5d}  ({100*n_empty/len(df):.1f}%)")
print(f"  Short assemblages (<3 minerals): {n_short:>5d}  ({100*n_short/len(df):.1f}%)")
print(f"  Usable assemblages (≥5 minerals):{n_ok:>5d}  ({100*n_ok/len(df):.1f}%)")

# Unique species
from collections import Counter
all_minerals = []
for ml in df["mineral_list"]:
    all_minerals.extend(ml)
mineral_counts = Counter(all_minerals)

print(f"\n  Total mineral mentions: {len(all_minerals):,}")
print(f"  Unique mineral species: {len(mineral_counts):,}")

# Frequency tiers
n_rare = sum(1 for m, c in mineral_counts.items() if c < 5)
n_common = sum(1 for m, c in mineral_counts.items() if c >= 50)
print(f"  Species with <5 occurrences:   {n_rare:>4d}")
print(f"  Species with ≥50 occurrences:  {n_common:>4d}")

## CELL 3 — Assemblage by type

Cross-tabulates assemblage length by deposit type. This
checks whether some deposit types are systematically more
data-sparse than others — a key concern for the HGCTM
because the DM likelihood must handle heterogeneous
completeness. Also shows whether the ≥5-mineral filter
would disproportionately remove certain types.

INPUT:  df (from Cell 2)
OUTPUT: console summary table + histogram

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("=== Assemblage length by deposit type ===\n")

type_stats = (
    df.groupby("Deposit_type")["n_minerals"]
    .agg(["count", "mean", "median", "min", "max",
          lambda x: (x < 3).sum(),
          lambda x: (x >= 5).sum()])
)
type_stats.columns = ["n_deposits", "mean_len", "median_len",
                       "min_len", "max_len", "n_short(<3)", "n_usable(≥5)"]
type_stats["pct_usable"] = (
    100 * type_stats["n_usable(≥5)"] / type_stats["n_deposits"]
).round(1)
print(type_stats.to_string())

# Histogram of assemblage length by type
fig, ax = plt.subplots(figsize=(10, 5))
types = df["Deposit_type"].dropna().unique()
for t in sorted(types):
    subset = df.loc[df["Deposit_type"] == t, "n_minerals"]
    ax.hist(subset, bins=range(0, 120, 3), alpha=0.5, label=t)
ax.set_xlabel("Number of minerals per deposit")
ax.set_ylabel("Count")
ax.set_title("Assemblage length distribution by deposit type")
ax.legend(fontsize=8)
ax.set_xlim(0, 120)
fig.tight_layout()
plt.show()


## CELL 4 — Top minerals

Lists the 40 most frequent mineral species globally and
by deposit type (top 15 per type). This gives a quick
geological sanity check: do the dominant minerals match
expectations (chalcopyrite, pyrite, quartz for porphyry;
pentlandite, pyrrhotite for magmatic sulfide; etc.)?
Also reveals the heavy-tail sparsity that motivates
family-level aggregation for the HGCTM vocabulary.

INPUT:  df (from Cell 2), mineral_counts (from Cell 2)
OUTPUT: console tables

In [ ]:
print("=== Top 40 minerals (global) ===\n")
top40 = pd.DataFrame(
    mineral_counts.most_common(40),
    columns=["mineral", "count"]
)
top40["pct_deposits"] = (100 * top40["count"] / len(df)).round(1)
print(top40.to_string(index=False))

print("\n=== Top 15 minerals per deposit type ===\n")
for dtype in sorted(df["Deposit_type"].dropna().unique()):
    sub = df[df["Deposit_type"] == dtype]
    mins = []
    for ml in sub["mineral_list"]:
        mins.extend(ml)
    c = Counter(mins)
    top = c.most_common(15)
    pct = [(m, cnt, round(100 * cnt / len(sub), 1)) for m, cnt in top]
    print(f"\n--- {dtype} (n={len(sub)}) ---")
    for m, cnt, p in pct:
        print(f"  {m:<25s}  {cnt:>5d}  ({p}%)")


## CELL 5 — Age and grade fields

Checks coverage and distribution of Max_age, Min_age,
Tonnage, and Cu_grade fields. These are not direct inputs
to the HGCTM but are needed for: (a) the Macrostrat
age-category covariate, (b) validation analyses, and
(c) understanding which deposits have rich vs sparse
metadata. Reports missingness and basic statistics by
deposit type.

INPUT:  df (from Cell 0)
OUTPUT: console summary

In [ ]:
numeric_fields = {
    "Max_age(Ma)": "Max age (Ma)",
    "Min_age(Ma)": "Min age (Ma)",
}

# Check which tonnage/grade columns exist
possible_tg = ["Tonnage(Mt)", "Cu_grade(%)", "Tonnage", "Cu_grade",
               "tonnage_mt", "cu_pct"]
for c in df.columns:
    if "tonnage" in c.lower() or "grade" in c.lower() or "cu_" in c.lower():
        if c not in numeric_fields:
            numeric_fields[c] = c

print("=== Numeric field coverage ===\n")
for col, label in numeric_fields.items():
    if col not in df.columns:
        print(f"  {label:<25s}  COLUMN NOT FOUND")
        continue
    vals = pd.to_numeric(df[col], errors="coerce")
    n_valid = vals.notna().sum()
    n_miss = vals.isna().sum()
    pct_valid = 100 * n_valid / len(df)
    print(f"  {label:<25s}  valid: {n_valid:>5d} ({pct_valid:5.1f}%)  "
          f"missing: {n_miss:>5d}  "
          f"median: {vals.median():>10.2f}  "
          f"range: [{vals.min():.1f}, {vals.max():.1f}]")

# Age coverage by type
print("\n=== Age coverage by deposit type ===\n")
if "Max_age(Ma)" in df.columns:
    age_cov = (
        df.groupby("Deposit_type")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "has_age": g["Max_age(Ma)"].notna().sum(),
            "pct_age": round(100 * g["Max_age(Ma)"].notna().mean(), 1),
        }))
    )
    print(age_cov.to_string())
else:
    print("  Max_age(Ma) column not found — skip")


## CELL 6 — Geographic coverage

Maps deposit locations and summarises geographic spread.
Reports deposits per continent (approximated from lat/lon)
and checks whether continent-level holdout groups are
viable for Stage 8 of the HGCTM protocol. A continent
needs ≥30 deposits to serve as a meaningful holdout set.

INPUT:  df (from Cell 0)
OUTPUT: console summary + scatter map

In [ ]:
print("=== Geographic coverage ===\n")

# Simple continent assignment from coordinates
def assign_continent(lat, lon):
    """Rough continent assignment for holdout design."""
    if pd.isna(lat) or pd.isna(lon):
        return "Unknown"
    if lat > 15 and -170 < lon < -50:
        return "North_America"
    if lat <= 15 and -120 < lon < -30:
        return "South_America"
    if lat > 35 and -25 < lon < 60:
        return "Europe"
    if lat > 0 and 60 < lon < 180:
        return "Asia"
    if lat <= 0 and 95 < lon < 180:
        return "Oceania"
    if -40 < lat < 40 and -25 < lon < 55:
        return "Africa"
    if lat <= -10 and 110 < lon < 180:
        return "Oceania"
    return "Other"

df["continent_approx"] = df.apply(
    lambda r: assign_continent(r.get("Latitude"), r.get("Longitude")), axis=1
)

cont_counts = (
    df.groupby("continent_approx")
    .agg(
        n_deposits=("Deposit_type", "count"),
        types=("Deposit_type", "nunique"),
        median_minerals=("n_minerals", "median"),
    )
    .sort_values("n_deposits", ascending=False)
)
cont_counts["holdout_viable"] = cont_counts["n_deposits"].apply(
    lambda x: "OK" if x >= 30 else "WARN"
)
print(cont_counts.to_string())

n_no_coords = df[["Latitude", "Longitude"]].isna().any(axis=1).sum()
print(f"\nDeposits missing coordinates: {n_no_coords}")

# Scatter map
fig, ax = plt.subplots(figsize=(14, 6))
colors = {
    "Porphyry": "#1b9e77", "VMS": "#d95f02",
    "Sediment-Hosted": "#7570b3", "Magmatic sulfide": "#e7298a",
    "IOCG": "#66a61e"
}
for dtype, grp in df.groupby("Deposit_type"):
    ax.scatter(
        grp["Longitude"], grp["Latitude"],
        s=12, alpha=0.6, label=dtype,
        color=colors.get(dtype, "gray")
    )
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("GCDD deposit locations by type")
ax.legend(fontsize=8, markerscale=2)
ax.grid(True, linewidth=0.3, alpha=0.5)
fig.tight_layout()
plt.show()


## CELL 7 — Viability summary

Prints a compact go/no-go summary for HGCTM protocol
Stages 0–1. Checks five preconditions: (1) dataset size,
(2) class balance, (3) assemblage depth, (4) covariate
availability, (5) holdout group viability. Each condition
is flagged as PASS or FAIL. If all pass, the data supports
proceeding to family mapping and model fitting.

INPUT:  df (from previous cells)
OUTPUT: console go/no-go report

In [ ]:
print("=" * 60)
print("  HGCTM PROTOCOL — STAGE 0/1 VIABILITY SUMMARY")
print("=" * 60)

checks = []

# 1. Dataset size
n = len(df)
ok = n >= 500
checks.append(("Dataset size", f"{n} deposits", "PASS" if ok else "FAIL"))

# 2. Class balance
min_class = df["Deposit_type"].value_counts().min()
n_types = df["Deposit_type"].nunique()
ok = min_class >= 20 and n_types >= 4
checks.append(("Class balance",
               f"{n_types} types, smallest={min_class}",
               "PASS" if ok else "FAIL"))

# 3. Assemblage depth
med_minerals = df["n_minerals"].median()
pct_usable = 100 * (df["n_minerals"] >= 5).mean()
ok = med_minerals >= 5 and pct_usable >= 50
checks.append(("Assemblage depth",
               f"median={med_minerals:.0f}, ≥5 minerals: {pct_usable:.0f}%",
               "PASS" if ok else "FAIL"))

# 4. Age field availability
if "Max_age(Ma)" in df.columns:
    pct_age = 100 * df["Max_age(Ma)"].notna().mean()
else:
    pct_age = 0
ok_age = pct_age >= 40
checks.append(("Age coverage",
               f"{pct_age:.0f}% have Max_age",
               "PASS" if ok_age else "WARN"))

# 5. Holdout viability
n_viable_continents = (cont_counts["n_deposits"] >= 30).sum()
ok = n_viable_continents >= 3
checks.append(("Holdout groups",
               f"{n_viable_continents} continents with ≥30 deposits",
               "PASS" if ok else "FAIL"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<22s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print()
if all_pass:
    print("  → ALL CHECKS PASSED. Proceed to family mapping and model fitting.")
else:
    print("  → ONE OR MORE CHECKS FAILED. Address issues before proceeding.")
print("=" * 60)

## CELL 8 — Family mapping

Builds the complete species-to-family mapping from scratch
using RRUFF/IMA chemistry. Produces two family columns:
family_primary  (~10 broad mineralogical classes)
family_copper   (~36 copper-geology-specific groups)
Plus process/element tags for later validation layers.

Requires RRUFF dataset added to Kaggle:
lsind18/ima-database-of-mineral-properties

INPUT:  df (from Cell 0–2, needs mineral_list and mineral_counts)
RRUFF_Export_20230608_052338.csv
OUTPUT: /kaggle/working/mineral_to_family.csv

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# ── Paths ─────────────────────────────────────────────────────
RRUFF_PATH = "/kaggle/input/datasets/lsind18/ima-database-of-mineral-properties/RRUFF_Export_20230608_052338.csv"
OUT_MAP = "/kaggle/working/mineral_to_family.csv"

# ── Collect unique minerals from GCDD ─────────────────────────
all_mins = []
for ml in df["mineral_list"]:
    all_mins.extend(ml)
mineral_counts = Counter(all_mins)

gcdd_minerals = (
    pd.DataFrame(mineral_counts.items(), columns=["mineral_raw", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
print(f"Unique mineral species in GCDD: {len(gcdd_minerals)}")

# ── Name normalisation ────────────────────────────────────────
def base_name(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"-\([A-Za-z0-9]+\)$", "", s).strip()
    return s.strip('"').strip("'")

def mineral_key(s):
    return base_name(s).lower()

gcdd_minerals["mineral_base"] = gcdd_minerals["mineral_raw"].map(base_name)
gcdd_minerals["key"] = gcdd_minerals["mineral_base"].map(mineral_key)

# ── Load RRUFF ────────────────────────────────────────────────
rr = pd.read_csv(RRUFF_PATH)
req_rr = ["Mineral Name", "IMA Chemistry (plain)",
           "RRUFF Chemistry (plain)", "Chemistry Elements"]
missing = [c for c in req_rr if c not in rr.columns]
if missing:
    raise ValueError(f"RRUFF missing columns: {missing}")

rr["mineral_base_ref"] = rr["Mineral Name"].map(base_name)
rr["key"] = rr["mineral_base_ref"].map(mineral_key)
rru = rr.drop_duplicates(subset=["key"]).copy()
rru["chem_use"] = rru["IMA Chemistry (plain)"].where(
    rru["IMA Chemistry (plain)"].notna(),
    rru["RRUFF Chemistry (plain)"]
)

# ── Chemistry-based family_primary classifier ─────────────────
HAL = {"F", "CL", "BR", "I"}
REE = set("LA CE PR ND SM EU GD TB DY HO ER TM YB LU Y".split())

def family_from_chemistry(formula, elements_str):
    f = "" if pd.isna(formula) else str(formula)
    e = "" if pd.isna(elements_str) else str(elements_str)
    fU = f.upper().replace("·", ".")
    elems = set(e.upper().split())

    # Native elements
    if len(elems) == 1:
        el = next(iter(elems))
        if el not in {"O", "H", "C", "N", "S"}:
            return "Native_elements"

    # Organic
    if re.search(r"C\d{2,}", fU) and ("H" in elems) and \
       all(x not in fU for x in ["CO3","SO4","PO4","ASO4","VO4","SIO"]):
        return "Organic_compounds"

    # Halides
    if (elems & HAL) and not any(g in fU for g in ["CO3","SO4","PO4","ASO4","VO4"]):
        return "Halides"

    # Oxyanion classes
    if "CO3" in fU or "NO3" in fU:
        return "Carbonates_nitrates"
    if any(g in fU for g in ["SO4","CRO4","MOO4","WO4"]):
        return "Sulfates_chromates_molybdates_tungstates"
    if any(g in fU for g in ["PO4","ASO4","VO4"]):
        return "Phosphates_arsenates_vanadates"

    # Borates
    if ("B" in elems) and ("O" in elems) and ("SI" not in elems) and \
       not any(g in fU for g in ["CO3","SO4","PO4","ASO4","VO4"]):
        return "Borates"

    # Silicates
    if ("SI" in elems) and ("O" in elems) and \
       not any(g in fU for g in ["CO3","SO4","PO4","ASO4","VO4"]):
        return "Silicates"

    # Oxides/hydroxides
    if ("O" in elems) and not any(x in elems for x in ["S","SE","TE"]) and \
       not any(g in fU for g in ["CO3","SO4","PO4","ASO4","VO4"]) and ("SI" not in elems):
        return "Oxides_hydroxides"

    # Sulfides + analogues / sulfosalts
    if any(x in elems for x in ["S","SE","TE"]) or \
       (("AS" in elems or "SB" in elems or "BI" in elems) and ("O" not in elems)):
        if any(x in elems for x in ["S","SE","TE"]) and \
           any(x in elems for x in ["AS","SB","BI"]) and ("O" not in elems):
            return "Sulfosalts"
        return "Sulfides_and_analogues"

    return "Other_unknown"

rru["family_primary"] = [
    family_from_chemistry(f, e)
    for f, e in zip(rru["chem_use"], rru["Chemistry Elements"])
]

# ── Refined family_copper classifier (~36 groups) ─────────────
FE_SULF = {"pyrite", "pyrrhotite", "marcasite"}

SKARN_MINERALS = {"garnet", "diopside", "wollastonite",
                  "vesuvianite", "epidote", "actinolite"}

def family_copper_refined(mineral_base, family_primary, elems_str):
    nb = (mineral_base or "").lower()
    elems = set(str(elems_str).upper().split()) if pd.notna(elems_str) else set()

    # Quartz dedicated
    if nb == "quartz":
        return "Quartz"

    # Native elements
    if family_primary == "Native_elements":
        if nb in ["gold", "silver", "electrum"]:
            return "Au_Ag_native"
        if "PT" in elems or "PD" in elems or nb.endswith("platinum") or "pallad" in nb:
            return "PGE_alloys_intermetallics"
        return "Native_elements_misc"

    # Sulfides / sulfosalts
    if family_primary in ["Sulfides_and_analogues", "Sulfosalts"]:
        if nb in FE_SULF:
            return "Fe_sulfides"
        if "CU" in elems:
            if nb in ["chalcocite", "covellite", "digenite"]:
                return "Cu_sulfides_supergene"
            if nb in ["chalcopyrite", "bornite"]:
                return "Cu_sulfides_primary"
            return "Cu_sulfides_other"
        if nb == "molybdenite" or "MO" in elems:
            return "Mo_sulfides"
        if nb == "galena" or "PB" in elems:
            return "Pb_sulfides"
        if nb == "sphalerite" or "ZN" in elems:
            return "Zn_sulfides"
        if ("NI" in elems) or ("CO" in elems) or ("AS" in elems and "O" not in elems):
            return "Ni_Co_sulfides_arsenides"
        if any(x in elems for x in ["AS", "SB", "BI"]):
            return "As_Sb_Bi_sulfosalts"
        return "Sulfides_misc"

    # Oxides/hydroxides
    if family_primary == "Oxides_hydroxides":
        if "FE" in elems:
            return "Fe_oxides_hydroxides"
        if "TI" in elems:
            return "Ti_oxides"
        if "MN" in elems:
            return "Mn_oxides"
        if "CU" in elems:
            return "Cu_oxides"
        return "Oxides_misc"

    # Carbonates
    if family_primary == "Carbonates_nitrates":
        if "CU" in elems:
            return "Cu_carbonates_silicates"
        return "Carbonates"

    # Sulfates etc.
    if family_primary == "Sulfates_chromates_molybdates_tungstates":
        if nb in ["barite", "baryte"]:
            return "Barite"
        if nb in ["anhydrite", "gypsum"]:
            return "Evaporite_sulfates"
        return "Sulfates_other"

    # Phosphates
    if family_primary == "Phosphates_arsenates_vanadates":
        if any(x in elems for x in REE):
            return "REE_accessory"
        if "apatite" in nb:
            return "Phosphates_apatite"
        return "Phosphates_other"

    # Halides
    if family_primary == "Halides":
        if nb == "fluorite" or "F" in elems:
            return "Halides_fluorite"
        return "Halides_other"

    # Borates
    if family_primary == "Borates":
        return "Borates"

    # Silicates
    if family_primary == "Silicates":
        if nb in SKARN_MINERALS or nb in ["wollastonite", "vesuvianite", "diopside", "garnet"]:
            return "Skarn_calc_silicates"
        if nb in ["muscovite", "sericite", "biotite"]:
            return "Micas_sericite"
        if nb in ["albite", "orthoclase", "microcline", "k-feldspar"]:
            return "Feldspars"
        if ("kaolin" in nb) or (nb in ["illite", "montmorillonite", "smectite"]):
            return "Clay_minerals"
        if "chlorite" in nb:
            return "Chlorite_group"
        if ("epidote" in nb) or ("clinozoisite" in nb):
            return "Epidote_group"
        if nb in ["actinolite", "hornblende"] or "pyroxene" in nb:
            return "Amphiboles_pyroxenes"
        return "Other_silicates"

    if family_primary == "Organic_compounds":
        return "Organic_compounds"

    return "Other_accessory"

rru["family_copper"] = [
    family_copper_refined(n, fp, e)
    for n, fp, e in zip(rru["mineral_base_ref"], rru["family_primary"], rru["Chemistry Elements"])
]

# ── Tags ──────────────────────────────────────────────────────
CU_PRIMARY   = {"chalcopyrite", "bornite"}
CU_SUPERGENE = {"chalcocite", "covellite", "digenite", "cuprite",
                "malachite", "azurite", "chrysocolla", "native copper", "tenorite"}
MO_IND       = {"molybdenite"}
FE_OX        = {"magnetite", "hematite", "goethite", "limonite"}

def build_tags(mineral_base, elems_str):
    tags = set()
    nb = (mineral_base or "").lower()
    elems = set(str(elems_str).upper().split()) if pd.notna(elems_str) else set()

    if "CU" in elems: tags.add("Cu_bearing")
    if "MO" in elems: tags.add("Mo_bearing")
    if "AU" in elems: tags.add("Au_bearing")
    if "AG" in elems: tags.add("Ag_bearing")
    if "NI" in elems: tags.add("Ni_bearing")
    if "CO" in elems: tags.add("Co_bearing")
    if any(x in elems for x in REE): tags.add("REE_indicator")
    if ("U" in elems) or ("TH" in elems): tags.add("U_Th_indicator")

    if nb in CU_PRIMARY: tags.add("hypogene_Cu")
    if nb in CU_SUPERGENE: tags.add("supergene_Cu")
    if nb in MO_IND: tags.add("porphyry_Mo_indicator")
    if nb in FE_OX: tags.add("Fe_oxide")
    if nb in FE_SULF: tags.add("Fe_sulfide")
    if nb in SKARN_MINERALS: tags.add("skarn_proxy")

    return ";".join(sorted(tags))

rru["tags"] = [
    build_tags(n, e)
    for n, e in zip(rru["mineral_base_ref"], rru["Chemistry Elements"])
]

# ── Join mapping to GCDD minerals ─────────────────────────────
m = gcdd_minerals.merge(
    rru[["key", "family_primary", "family_copper", "tags"]],
    on="key", how="left"
)

m["family_primary"] = m["family_primary"].fillna("Other_unknown")
m["family_copper"] = m["family_copper"].fillna("Other_unknown")
m["tags"] = m["tags"].fillna("")

# ── Save ──────────────────────────────────────────────────────
m[["mineral_raw", "mineral_base", "key", "count",
   "family_primary", "family_copper", "tags"]].to_csv(OUT_MAP, index=False)

# ── Report ────────────────────────────────────────────────────
n_total = len(m)
n_mapped = (m["family_primary"] != "Other_unknown").sum()
n_unknown = n_total - n_mapped
mentions_mapped = m.loc[m["family_primary"] != "Other_unknown", "count"].sum()
mentions_total = m["count"].sum()

print(f"\nSaved: {OUT_MAP}")
print(f"  Species total:    {n_total}")
print(f"  Mapped to family: {n_mapped}  ({100*n_mapped/n_total:.1f}%)")
print(f"  Unmapped:         {n_unknown}")
print(f"  Mention coverage: {mentions_mapped}/{mentions_total}  "
      f"({100*mentions_mapped/mentions_total:.1f}%)")
print(f"\n  family_primary groups:  {m['family_primary'].nunique()}")
print(f"  family_copper groups:   {m['family_copper'].nunique()}")

# Show family_copper group sizes
print(f"\n=== family_copper groups (by unique species) ===")
fc_stats = (
    m[m["family_copper"] != "Other_unknown"]
    .groupby("family_copper")["mineral_raw"]
    .nunique()
    .sort_values(ascending=False)
)
for g, n in fc_stats.items():
    print(f"  {g:<35s}  {n:>4d} species")

# Show unmapped minerals (top by count)
if n_unknown > 0:
    print(f"\n=== Top unmapped minerals ===")
    unmapped = m[m["family_primary"] == "Other_unknown"].sort_values("count", ascending=False)
    print(unmapped[["mineral_raw", "count"]].head(20).to_string(index=False))

## CELL 8b — Mapping patch

Fixes 33 unmapped minerals: Graphite → Native_elements_misc,
PGE intermetallics → PGE_alloys_intermetallics, and a few
rare sulfosalts. Updates mineral_to_family.csv in place.

INPUT:  /kaggle/working/mineral_to_family.csv
OUTPUT: /kaggle/working/mineral_to_family.csv (updated)

In [ ]:
import pandas as pd

m = pd.read_csv("/kaggle/working/mineral_to_family.csv")

# All unmapped PGE-bearing intermetallics/alloys
PGE_MINERALS = {
    "niggliite", "atokite", "awaruite", "paolovite", "rustenburgite",
    "testibiopalladite", "isoferroplatinum", "zvyagintsevite",
    "tetraferroplatinum", "stannopalladinite", "taimyrite",
    "auricupride", "cabriite", "plumbopalladinite", "leadamalgam",
}

mask_unknown = m["family_primary"] == "Other_unknown"

for idx in m[mask_unknown].index:
    nb = str(m.loc[idx, "mineral_base"]).lower()
    
    if nb == "graphite":
        m.loc[idx, "family_primary"] = "Native_elements"
        m.loc[idx, "family_copper"] = "Native_elements_misc"
    elif nb in PGE_MINERALS:
        m.loc[idx, "family_primary"] = "Native_elements"
        m.loc[idx, "family_copper"] = "PGE_alloys_intermetallics"
    elif nb in ["zhonghongite", "juxingite"]:
        m.loc[idx, "family_primary"] = "Sulfosalts"
        m.loc[idx, "family_copper"] = "As_Sb_Bi_sulfosalts"
    elif nb == "clino-ferro-suenoite":
        m.loc[idx, "family_primary"] = "Silicates"
        m.loc[idx, "family_copper"] = "Amphiboles_pyroxenes"
    elif nb == "ferronigerite-2n1s":
        m.loc[idx, "family_primary"] = "Oxides_hydroxides"
        m.loc[idx, "family_copper"] = "Oxides_misc"

m.to_csv("/kaggle/working/mineral_to_family.csv", index=False)

n_still = (m["family_primary"] == "Other_unknown").sum()
print(f"Remaining unmapped: {n_still}")
print(f"Patched: {33 - n_still} minerals")

## CELL 9 — Count matrices

Builds deposit × mineral-family count matrices for both
vocabulary levels (family_primary and family_copper).
These are the direct inputs to all topic models in the
HGCTM protocol. Uses presence-based counting: each mineral
species contributes 1 to its family count per deposit,
regardless of how many times it is mentioned.

INPUT:  df (from Cell 0–2), mineral_to_family.csv (Cell 8)
OUTPUT: /kaggle/working/X_family_primary.csv   (deposit × ~10 families)
/kaggle/working/X_family_copper.csv     (deposit × ~36 families)
/kaggle/working/deposit_completeness.csv

In [ ]:
import pandas as pd
import numpy as np

MAP_PATH = "/kaggle/working/mineral_to_family.csv"

# ── Load mapping ──────────────────────────────────────────────
mp = pd.read_csv(MAP_PATH)
mp_lookup = mp.drop_duplicates(subset="key").set_index("key")[["family_primary", "family_copper"]].to_dict("index")


# ── Explode deposits × minerals, join families ────────────────
rows = []
for idx, row in df.iterrows():
    mid = row.get("Mindat_id", idx)
    for mineral in row["mineral_list"]:
        k = mineral_key(mineral)
        fam = mp_lookup.get(k, {"family_primary": "UNMAPPED", "family_copper": "UNMAPPED"})
        rows.append({
            "Mindat_id": mid,
            "mineral_key": k,
            "family_primary": fam["family_primary"],
            "family_copper": fam["family_copper"],
        })

pairs = pd.DataFrame(rows)

# Deduplicate: one count per species per deposit
pairs = pairs.drop_duplicates(subset=["Mindat_id", "mineral_key"])

n_unmapped_pairs = (pairs["family_primary"] == "UNMAPPED").sum()
print(f"Deposit-mineral pairs: {len(pairs)}")
print(f"Unmapped pairs: {n_unmapped_pairs}  ({100*n_unmapped_pairs/len(pairs):.1f}%)")

# Drop unmapped for count matrices
pairs_clean = pairs[pairs["family_primary"] != "UNMAPPED"].copy()

# ── Build count matrix: family_primary ────────────────────────
X_fp = (
    pairs_clean.groupby(["Mindat_id", "family_primary"])["mineral_key"]
    .nunique()
    .unstack(fill_value=0)
)
X_fp.columns.name = None
print(f"\nX_family_primary: {X_fp.shape[0]} deposits × {X_fp.shape[1]} families")

# ── Build count matrix: family_copper ─────────────────────────
X_fc = (
    pairs_clean.groupby(["Mindat_id", "family_copper"])["mineral_key"]
    .nunique()
    .unstack(fill_value=0)
)
X_fc.columns.name = None
print(f"X_family_copper:  {X_fc.shape[0]} deposits × {X_fc.shape[1]} families")

# ── Completeness variables ────────────────────────────────────
completeness = pd.DataFrame(index=X_fc.index)
n_minerals_series = df.drop_duplicates(subset="Mindat_id").set_index("Mindat_id")["n_minerals"]
completeness["n_minerals_total"] = n_minerals_series.reindex(X_fc.index).fillna(0).astype(int)
completeness["n_minerals_mapped"] = X_fc.sum(axis=1).astype(int)
completeness["n_families_primary"] = (X_fp > 0).sum(axis=1).astype(int)
completeness["n_families_copper"] = (X_fc > 0).sum(axis=1).astype(int)
completeness["pct_mapped"] = (
    100 * completeness["n_minerals_mapped"] / completeness["n_minerals_total"].clip(lower=1)
).round(1)


# ── Save ──────────────────────────────────────────────────────
X_fp.to_csv("/kaggle/working/X_family_primary.csv")
X_fc.to_csv("/kaggle/working/X_family_copper.csv")
completeness.to_csv("/kaggle/working/deposit_completeness.csv")

print(f"\nSaved: X_family_primary.csv, X_family_copper.csv, deposit_completeness.csv")

# ── Quick summary ─────────────────────────────────────────────
print(f"\n=== Count matrix summary ===")
print(f"  family_primary: {X_fp.shape[1]} columns, "
      f"row sums: median={X_fp.sum(axis=1).median():.0f}, "
      f"range=[{X_fp.sum(axis=1).min()}, {X_fp.sum(axis=1).max()}]")
print(f"  family_copper:  {X_fc.shape[1]} columns, "
      f"row sums: median={X_fc.sum(axis=1).median():.0f}, "
      f"range=[{X_fc.sum(axis=1).min()}, {X_fc.sum(axis=1).max()}]")

# Column sparsity for family_copper
col_nonzero = (X_fc > 0).sum(axis=0).sort_values(ascending=False)
print(f"\n=== family_copper column frequencies (deposits with ≥1 count) ===")
for col, n in col_nonzero.items():
    print(f"  {col:<35s}  {n:>5d}  ({100*n/X_fc.shape[0]:5.1f}%)")


## CELL 10 — Vocabulary viability

Checks whether both vocabularies meet the HGCTM protocol
requirements: ≥15 families with non-trivial frequency for
family_copper, ≥8 for family_primary. Also checks the
deposit filtering threshold: how many deposits remain
after requiring ≥3 mapped minerals (the minimum for
meaningful topic inference). Prints go/no-go for Stage 2.

INPUT:  X_family_primary.csv, X_family_copper.csv,
deposit_completeness.csv
OUTPUT: console go/no-go report

In [ ]:
import pandas as pd
import numpy as np

X_fp = pd.read_csv("/kaggle/working/X_family_primary.csv", index_col=0)
X_fc = pd.read_csv("/kaggle/working/X_family_copper.csv", index_col=0)
comp = pd.read_csv("/kaggle/working/deposit_completeness.csv", index_col=0)

# ── Filter threshold: ≥3 mapped minerals ──────────────────────
MIN_MINERALS = 3
n_total = len(X_fc)
n_pass_fp = (X_fp.sum(axis=1) >= MIN_MINERALS).sum()
n_pass_fc = (X_fc.sum(axis=1) >= MIN_MINERALS).sum()

# ── Non-trivial families: present in ≥10 deposits ─────────────
MIN_DEPOSITS = 10
fp_viable = ((X_fp > 0).sum(axis=0) >= MIN_DEPOSITS).sum()
fc_viable = ((X_fc > 0).sum(axis=0) >= MIN_DEPOSITS).sum()

# ── Deposit type retention after filtering ────────────────────
# Reconstruct deposit type from main df
dtype_map = df.drop_duplicates(subset="Mindat_id").set_index("Mindat_id")["Deposit_type"]

fc_with_type = X_fc.copy()
fc_with_type["dtype"] = dtype_map.reindex(X_fc.index)
fc_with_type["row_sum"] = X_fc.sum(axis=1)
filtered = fc_with_type[fc_with_type["row_sum"] >= MIN_MINERALS]

print("=" * 60)
print("  HGCTM PROTOCOL — VOCABULARY VIABILITY CHECK")
print("=" * 60)

checks = []

# 1. family_primary vocabulary
ok = fp_viable >= 8
checks.append(("family_primary vocab",
               f"{fp_viable} families in ≥{MIN_DEPOSITS} deposits (of {X_fp.shape[1]} total)",
               "PASS" if ok else "FAIL"))

# 2. family_copper vocabulary
ok = fc_viable >= 15
checks.append(("family_copper vocab",
               f"{fc_viable} families in ≥{MIN_DEPOSITS} deposits (of {X_fc.shape[1]} total)",
               "PASS" if ok else "FAIL"))

# 3. Deposits surviving filter (family_copper)
ok = n_pass_fc >= 1000
checks.append(("Deposits after filter",
               f"{n_pass_fc} of {n_total} have ≥{MIN_MINERALS} families "
               f"({100*n_pass_fc/n_total:.0f}%)",
               "PASS" if ok else "FAIL"))

# 4. All types survive filter
type_counts_after = filtered["dtype"].value_counts()
min_after = type_counts_after.min() if len(type_counts_after) > 0 else 0
n_types_after = len(type_counts_after)
ok = min_after >= 20 and n_types_after >= 5
checks.append(("Types after filter",
               f"{n_types_after} types, smallest={min_after}",
               "PASS" if ok else "FAIL"))

# 5. Mapping coverage
med_pct = comp["pct_mapped"].median()
ok = med_pct >= 80
checks.append(("Mapping coverage",
               f"median {med_pct:.0f}% minerals mapped per deposit",
               "PASS" if ok else "WARN"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<25s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print(f"\n=== Deposits per type after ≥{MIN_MINERALS} filter ===")
for dtype, n in type_counts_after.sort_values(ascending=False).items():
    print(f"  {dtype:<25s}  {n:>5d}")

print()
if all_pass:
    print("  → VOCABULARY VIABLE. Proceed to Stage 2 (baseline LDA).")
else:
    print("  → VOCABULARY ISSUE. Address before fitting topic models.")
print("=" * 60)

## CELL 11 — LDA sweep

Stage 2 of the HGCTM protocol: baseline topic model signal.
Fits standard LDA on both vocabularies (family_primary and
family_copper) across K = 4..12 with 3 random seeds each.
Reports held-out log-likelihood and per-topic top families.
This is the first test of whether the data supports topic
modelling at all. If no interpretable topics emerge, the
HGCTM paper cannot proceed.

INPUT:  /kaggle/working/X_family_primary.csv
/kaggle/working/X_family_copper.csv
OUTPUT: /kaggle/working/lda_sweep_results.csv
console: top families per topic for best K

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
import warnings
warnings.filterwarnings("ignore")

# ── Load count matrices ───────────────────────────────────────
X_fp = pd.read_csv("/kaggle/working/X_family_primary.csv", index_col=0)
X_fc = pd.read_csv("/kaggle/working/X_family_copper.csv", index_col=0)

# ── Filter: ≥3 mapped minerals per deposit ────────────────────
MIN_MINERALS = 3
mask_fp = X_fp.sum(axis=1) >= MIN_MINERALS
mask_fc = X_fc.sum(axis=1) >= MIN_MINERALS

X_fp_f = X_fp[mask_fp].copy()
X_fc_f = X_fc[mask_fc].copy()

# ── Filter: drop families present in <10 deposits ─────────────
MIN_DEPOSITS = 10
fp_keep = X_fp_f.columns[(X_fp_f > 0).sum(axis=0) >= MIN_DEPOSITS]
fc_keep = X_fc_f.columns[(X_fc_f > 0).sum(axis=0) >= MIN_DEPOSITS]

X_fp_f = X_fp_f[fp_keep]
X_fc_f = X_fc_f[fc_keep]

print(f"family_primary: {X_fp_f.shape[0]} deposits × {X_fp_f.shape[1]} families")
print(f"family_copper:  {X_fc_f.shape[0]} deposits × {X_fc_f.shape[1]} families")

# ── LDA sweep parameters ─────────────────────────────────────
K_RANGE = list(range(4, 13))       # K = 4, 5, ..., 12
SEEDS = [42, 123, 7]               # 3 initialisations per K
MAX_ITER = 300

results = []

for vocab_name, X_mat, families in [
    ("family_primary", X_fp_f, list(fp_keep)),
    ("family_copper",  X_fc_f, list(fc_keep)),
]:
    X_arr = X_mat.values
    print(f"\n{'='*60}")
    print(f"  LDA sweep: {vocab_name} ({X_arr.shape[1]} families)")
    print(f"{'='*60}")

    for K in K_RANGE:
        for seed in SEEDS:
            lda = LatentDirichletAllocation(
                n_components=K,
                doc_topic_prior=0.5 / K,
                topic_word_prior=0.01,
                max_iter=MAX_ITER,
                random_state=seed,
                learning_method="batch",
                evaluate_every=10,
                n_jobs=-1,
            )
            lda.fit(X_arr)

            # Held-out log-likelihood (per-document average on training data)
            ll = lda.score(X_arr) / X_arr.shape[0]
            perplexity = lda.perplexity(X_arr)

            results.append({
                "vocab": vocab_name,
                "K": K,
                "seed": seed,
                "log_lik_per_doc": round(ll, 2),
                "perplexity": round(perplexity, 2),
            })

        # Print progress
        seed_lls = [r["log_lik_per_doc"] for r in results
                    if r["vocab"] == vocab_name and r["K"] == K]
        print(f"  K={K:2d}  mean LL/doc={np.mean(seed_lls):8.2f}  "
              f"std={np.std(seed_lls):6.2f}  "
              f"perplexity≈{np.mean([r['perplexity'] for r in results if r['vocab']==vocab_name and r['K']==K]):8.1f}")

# ── Save results ──────────────────────────────────────────────
res_df = pd.DataFrame(results)
res_df.to_csv("/kaggle/working/lda_sweep_results.csv", index=False)
print(f"\nSaved: /kaggle/working/lda_sweep_results.csv")


## CELL 12 — K selection

Stage 3 of the HGCTM protocol: select K based on held-out
log-likelihood and visual elbow. Plots LL vs K for both
vocabularies. Also prints the top-10 families per topic
at the best K for family_copper — the geological sanity
check that determines whether topics are interpretable.

INPUT:  /kaggle/working/lda_sweep_results.csv
/kaggle/working/X_family_copper.csv
OUTPUT: elbow plot + topic inspection table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import LatentDirichletAllocation

res = pd.read_csv("/kaggle/working/lda_sweep_results.csv")

# ── Elbow plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, vocab in zip(axes, ["family_primary", "family_copper"]):
    sub = res[res["vocab"] == vocab]
    agg = sub.groupby("K")["log_lik_per_doc"].agg(["mean", "std"]).reset_index()

    ax.errorbar(agg["K"], agg["mean"], yerr=agg["std"],
                marker="o", capsize=4, linewidth=2, markersize=6)
    ax.set_xlabel("K (number of topics)")
    ax.set_ylabel("Log-likelihood per document")
    ax.set_title(f"LDA sweep: {vocab}")
    ax.set_xticks(agg["K"])
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

# ── Identify best K per vocabulary ────────────────────────────
print("\n=== Mean log-likelihood per K ===\n")
for vocab in ["family_primary", "family_copper"]:
    sub = res[res["vocab"] == vocab]
    agg = sub.groupby("K")["log_lik_per_doc"].agg(["mean", "std"]).reset_index()
    best_K = agg.loc[agg["mean"].idxmax(), "K"]
    print(f"  {vocab}:")
    for _, row in agg.iterrows():
        marker = " ◄ best" if row["K"] == best_K else ""
        print(f"    K={int(row['K']):2d}  LL={row['mean']:8.2f} ± {row['std']:5.2f}{marker}")
    print()

# ── Topic inspection at best K for family_copper ──────────────
print("=" * 60)
print("  TOPIC INSPECTION: family_copper at best K")
print("=" * 60)

X_fc = pd.read_csv("/kaggle/working/X_family_copper.csv", index_col=0)
MIN_MINERALS = 3
MIN_DEPOSITS = 10
mask = X_fc.sum(axis=1) >= MIN_MINERALS
X_fc_f = X_fc[mask]
fc_keep = X_fc_f.columns[(X_fc_f > 0).sum(axis=0) >= MIN_DEPOSITS]
X_fc_f = X_fc_f[fc_keep]
families = list(fc_keep)

# Refit at best K with fixed seed
sub_fc = res[res["vocab"] == "family_copper"]
agg_fc = sub_fc.groupby("K")["log_lik_per_doc"].mean()
best_K_fc = int(agg_fc.idxmax())

print(f"\nBest K = {best_K_fc} (by mean log-likelihood)")
print(f"Fitting final LDA with K={best_K_fc}...\n")

lda_best = LatentDirichletAllocation(
    n_components=best_K_fc,
    doc_topic_prior=0.5 / best_K_fc,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)
lda_best.fit(X_fc_f.values)

# Print top-10 families per topic
phi = lda_best.components_
phi_norm = phi / phi.sum(axis=1, keepdims=True)

for k in range(best_K_fc):
    top_idx = np.argsort(phi_norm[k])[::-1][:10]
    print(f"--- Topic {k+1} ---")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank:2d}. {families[idx]:<35s}  {phi_norm[k, idx]:.3f}")
    print()


## CELL 12b — K=7 inspection

Fits LDA at K=7 on family_copper to check whether finer
topic structure is geologically richer than K=4, even if
training LL is slightly lower. K=7 is the value used in
the existing notebook and matches the GCDD paper's five
deposit types plus alteration/gangue separation.

INPUT:  X_family_copper.csv (filtered)
OUTPUT: console topic tables

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

K_TEST = 7

lda7 = LatentDirichletAllocation(
    n_components=K_TEST,
    doc_topic_prior=0.5 / K_TEST,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)
lda7.fit(X_fc_f.values)

phi7 = lda7.components_
phi7_norm = phi7 / phi7.sum(axis=1, keepdims=True)

print(f"=== LDA K={K_TEST} on family_copper ({X_fc_f.shape[1]} families) ===\n")
for k in range(K_TEST):
    top_idx = np.argsort(phi7_norm[k])[::-1][:10]
    print(f"--- Topic {k+1} ---")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank:2d}. {families[idx]:<35s}  {phi7_norm[k, idx]:.3f}")
    print()

## CELL 13 — Signal summary

Stage 2/3 go/no-go gate. Summarises whether the baseline
LDA produces interpretable topics and whether a clear K
can be selected. This determines if we proceed to the
full model comparison ladder (Stage 4).

INPUT:  /kaggle/working/lda_sweep_results.csv, Cell 12 output
OUTPUT: console go/no-go report

In [ ]:
import pandas as pd
import numpy as np

res = pd.read_csv("/kaggle/working/lda_sweep_results.csv")

print("=" * 60)
print("  HGCTM PROTOCOL — STAGE 2/3 SIGNAL CHECK")
print("=" * 60)

checks = []

# 1. Log-likelihood improves over K=1 equivalent
# (K=4 should already be much better than unigram)
for vocab in ["family_primary", "family_copper"]:
    sub = res[res["vocab"] == vocab]
    agg = sub.groupby("K")["log_lik_per_doc"].mean()
    ll_min_K = agg.iloc[0]      # K=4
    ll_best_K = agg.max()
    improvement = ll_best_K - ll_min_K
    ok = ll_best_K > ll_min_K  # any improvement across K range
    checks.append((f"{vocab} LL trend",
                   f"K=4: {ll_min_K:.1f}, best: {ll_best_K:.1f}, Δ={improvement:.1f}",
                   "PASS" if ok else "WARN"))

# 2. Standard deviation across seeds is manageable
for vocab in ["family_primary", "family_copper"]:
    sub = res[res["vocab"] == vocab]
    agg = sub.groupby("K")["log_lik_per_doc"].std().mean()
    ok = agg < 5.0  # less than 5 LL units average std
    checks.append((f"{vocab} seed stability",
                   f"mean std across K: {agg:.2f}",
                   "PASS" if ok else "WARN"))

# 3. family_copper has more families than family_primary
sub_fc = res[res["vocab"] == "family_copper"]
sub_fp = res[res["vocab"] == "family_primary"]
best_fc = sub_fc.groupby("K")["log_lik_per_doc"].mean().max()
best_fp = sub_fp.groupby("K")["log_lik_per_doc"].mean().max()
checks.append(("Vocabulary comparison",
               f"family_copper best LL={best_fc:.1f}, family_primary best LL={best_fp:.1f}",
               "INFO"))

print()
all_ok = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("ℹ" if status == "INFO" else "⚠")
    print(f"  {flag} {name:<30s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_ok = False

print()
print("  NOTE: The critical check is whether topics in Cell 12 are")
print("  geologically interpretable. If ≥3 topics can be named")
print("  (e.g., porphyry Cu-Mo mode, VMS polymetallic mode, IOCG")
print("  Fe-oxide mode, supergene Cu mode), Stage 2 PASSES.")
print()
print("  → Review Cell 12 topic tables and make the go/no-go call.")
print("=" * 60)

## CELL 14 — Age categories

Bins deposit ages into broad era categories using the GCDD's
own Max_age and Min_age fields. Uses the midpoint of the
reported age range. This is the secondary covariate for the
HGCTM (subject to ordered shrinkage: σ²_age < σ²_litho).

Boundaries: Cenozoic <66 Ma, Mesozoic 66–252 Ma,
Paleozoic 252–541 Ma, Precambrian >541 Ma.

INPUT:  df (from Cell 0)
OUTPUT: df gains columns "age_midpoint_ma" and "age_category"

In [ ]:
import pandas as pd
import numpy as np
 
# Parse age fields
df["max_age_ma"] = pd.to_numeric(df["Max_age(Ma)"], errors="coerce")
df["min_age_ma"] = pd.to_numeric(df["Min_age(Ma)"], errors="coerce")
 
# Midpoint of reported age range
df["age_midpoint_ma"] = df[["max_age_ma", "min_age_ma"]].mean(axis=1)
 
# If only one age field is available, use it
df["age_midpoint_ma"] = df["age_midpoint_ma"].fillna(df["max_age_ma"])
df["age_midpoint_ma"] = df["age_midpoint_ma"].fillna(df["min_age_ma"])
 
# Bin into era categories
def age_to_era(ma):
    if pd.isna(ma):
        return "Unknown"
    if ma < 66:
        return "Cenozoic"
    elif ma < 252:
        return "Mesozoic"
    elif ma < 541:
        return "Paleozoic"
    else:
        return "Precambrian"
 
df["age_category"] = df["age_midpoint_ma"].apply(age_to_era)
 
# ── Report ────────────────────────────────────────────────────
print("=== Age category distribution ===\n")
age_dist = df["age_category"].value_counts()
for cat in ["Cenozoic", "Mesozoic", "Paleozoic", "Precambrian", "Unknown"]:
    n = age_dist.get(cat, 0)
    pct = 100 * n / len(df)
    print(f"  {cat:<15s}  {n:>5d}  ({pct:5.1f}%)")
 
print(f"\n  Total with age: {(df['age_category'] != 'Unknown').sum()}")
print(f"  Missing age:    {(df['age_category'] == 'Unknown').sum()}")
 
# Cross-tab with deposit type
print("\n=== Age category × deposit type ===\n")
ct = pd.crosstab(df["Deposit_type"], df["age_category"], margins=True)
print(ct.to_string())
 
 

## CELL 15 — Macrostrat lithology

Queries the Macrostrat API for each deposit's coordinates
to retrieve the dominant surface lithology. Assigns each
deposit to a broad lithology class: felsic, mafic,
carbonate, siliciclastic, metamorphic, volcanic, or other.
This is the PRIMARY covariate for the HGCTM content model.

Makes ~1400 HTTP requests with rate limiting and caching.
Expected runtime: 5–10 minutes. Results are saved to CSV
so the cell does not need to be rerun.

INPUT:  df (from Cell 0, needs Latitude/Longitude)
OUTPUT: /kaggle/working/macrostrat_lithology.csv
df gains columns "litho_raw" and "litho_class"

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
 
CACHE_PATH = "/kaggle/working/macrostrat_lithology.csv"
API_BASE = "https://macrostrat.org/api/v2/geologic_units/map"
RATE_LIMIT = 0.15  # seconds between requests
 
# ── Lithology classification rules ────────────────────────────
# Maps Macrostrat lith_type / lith terms to broad classes
LITHO_MAP = {
    # Felsic igneous
    "granite": "felsic", "granodiorite": "felsic", "rhyolite": "felsic",
    "dacite": "felsic", "tonalite": "felsic", "monzonite": "felsic",
    "syenite": "felsic", "aplite": "felsic", "pegmatite": "felsic",
    "felsic": "felsic", "trachyte": "felsic",
    # Intermediate
    "diorite": "felsic", "andesite": "felsic", "latite": "felsic",
    # Mafic/ultramafic igneous
    "basalt": "mafic", "gabbro": "mafic", "diabase": "mafic",
    "dolerite": "mafic", "peridotite": "mafic", "dunite": "mafic",
    "pyroxenite": "mafic", "norite": "mafic", "mafic": "mafic",
    "ultramafic": "mafic", "komatiite": "mafic", "serpentinite": "mafic",
    "anorthosite": "mafic", "troctolite": "mafic",
    # Volcanic (generic)
    "volcanic": "volcanic", "tuff": "volcanic", "ignimbrite": "volcanic",
    "volcaniclastic": "volcanic", "lava": "volcanic", "ash": "volcanic",
    # Carbonate sedimentary
    "limestone": "carbonate", "dolomite": "carbonate", "dolostone": "carbonate",
    "marble": "carbonate", "carbonate": "carbonate", "chalk": "carbonate",
    "travertine": "carbonate", "marl": "carbonate",
    # Siliciclastic sedimentary
    "sandstone": "siliciclastic", "shale": "siliciclastic",
    "mudstone": "siliciclastic", "siltstone": "siliciclastic",
    "conglomerate": "siliciclastic", "greywacke": "siliciclastic",
    "arkose": "siliciclastic", "claystone": "siliciclastic",
    "arenite": "siliciclastic", "argillite": "siliciclastic",
    "slate": "siliciclastic", "quartzite": "siliciclastic",
    "siliciclastic": "siliciclastic", "clastic": "siliciclastic",
    # Metamorphic
    "gneiss": "metamorphic", "schist": "metamorphic",
    "amphibolite": "metamorphic", "granulite": "metamorphic",
    "phyllite": "metamorphic", "migmatite": "metamorphic",
    "hornfels": "metamorphic", "eclogite": "metamorphic",
    "metamorphic": "metamorphic", "metasedimentary": "metamorphic",
    "metavolcanic": "metamorphic",
    # Evaporite
    "evaporite": "evaporite", "gypsum": "evaporite", "halite": "evaporite",
    "anhydrite": "evaporite", "salt": "evaporite",
}
 
def classify_lithology(lith_list):
    """Classify a list of Macrostrat lithology strings into a broad class."""
    if not lith_list:
        return "unknown"
 
    # Collect votes from all lithology terms
    votes = {}
    for lith_entry in lith_list:
        name = str(lith_entry).lower().strip()
        for term, cls in LITHO_MAP.items():
            if term in name:
                votes[cls] = votes.get(cls, 0) + 1
                break
 
    if not votes:
        return "other"
 
    # Return the most-voted class
    return max(votes, key=votes.get)
 
 
def query_macrostrat(lat, lon):
    """Query Macrostrat API for lithology at a point."""
    try:
        resp = requests.get(
            API_BASE,
            params={"lat": lat, "lng": lon, "response": "long"},
            timeout=15
        )
        if resp.status_code != 200:
            return [], f"HTTP {resp.status_code}"
 
        data = resp.json()
        success = data.get("success", {})
        features = success.get("data", [])
 
        lith_names = []
        for feature in features:
            # Try multiple paths where lithology info might be
            liths = feature.get("liths", [])
            if isinstance(liths, list):
                for lith in liths:
                    if isinstance(lith, dict):
                        name = lith.get("name", "") or lith.get("lith", "")
                        if name:
                            lith_names.append(name)
                    elif isinstance(lith, str):
                        lith_names.append(lith)
 
            # Also check top-level lith field
            top_lith = feature.get("lith", "")
            if top_lith and isinstance(top_lith, str):
                lith_names.append(top_lith)
 
            # Check description for lithology keywords
            desc = str(feature.get("descrip", "")).lower()
            strat_name = str(feature.get("strat_name", "")).lower()
            for text in [desc, strat_name]:
                for term in LITHO_MAP:
                    if term in text and term not in [l.lower() for l in lith_names]:
                        lith_names.append(term)
                        break
 
        return lith_names, "ok"
 
    except requests.exceptions.Timeout:
        return [], "timeout"
    except Exception as e:
        return [], str(e)[:50]
 
 
# ── Check cache ───────────────────────────────────────────────
if os.path.exists(CACHE_PATH):
    cached = pd.read_csv(CACHE_PATH)
    print(f"Loaded cached lithology for {len(cached)} deposits")
    print("To re-query, delete /kaggle/working/macrostrat_lithology.csv")
else:
    # ── Query Macrostrat for each deposit ─────────────────────
    records = []
    valid = df.dropna(subset=["Latitude", "Longitude"]).copy()
    valid = valid.drop_duplicates(subset=["Mindat_id"])
    n_total = len(valid)
 
    print(f"Querying Macrostrat for {n_total} deposits...")
    print(f"Estimated time: {n_total * RATE_LIMIT / 60:.1f} minutes\n")
 
    t0 = time.time()
    for i, (idx, row) in enumerate(valid.iterrows()):
        lat, lon = row["Latitude"], row["Longitude"]
        mid = row["Mindat_id"]
 
        lith_names, status = query_macrostrat(lat, lon)
        litho_class = classify_lithology(lith_names)
 
        records.append({
            "Mindat_id": mid,
            "latitude": lat,
            "longitude": lon,
            "litho_raw": "; ".join(lith_names) if lith_names else "",
            "litho_class": litho_class,
            "api_status": status,
        })
 
        # Progress every 100 deposits
        if (i + 1) % 100 == 0 or (i + 1) == n_total:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (n_total - i - 1) / rate
            n_ok = sum(1 for r in records if r["api_status"] == "ok")
            print(f"  [{i+1:>5d}/{n_total}]  "
                  f"ok={n_ok}  "
                  f"elapsed={elapsed:.0f}s  "
                  f"ETA={eta:.0f}s")
 
        time.sleep(RATE_LIMIT)
 
    cached = pd.DataFrame(records)
    cached.to_csv(CACHE_PATH, index=False)
    print(f"\nSaved: {CACHE_PATH}")
 
# ── Attach to main dataframe ──────────────────────────────────
litho_lookup = cached.set_index("Mindat_id")[["litho_raw", "litho_class"]]
df["litho_raw"] = litho_lookup["litho_raw"].reindex(df["Mindat_id"]).values
df["litho_class"] = litho_lookup["litho_class"].reindex(df["Mindat_id"]).values
df["litho_class"] = df["litho_class"].fillna("unknown")
 
# ── Report ────────────────────────────────────────────────────
print("\n=== Lithology class distribution ===\n")
litho_dist = df["litho_class"].value_counts()
for cls in ["felsic", "mafic", "volcanic", "carbonate", "siliciclastic",
            "metamorphic", "evaporite", "other", "unknown"]:
    n = litho_dist.get(cls, 0)
    pct = 100 * n / len(df)
    print(f"  {cls:<15s}  {n:>5d}  ({pct:5.1f}%)")
 
n_resolved = (df["litho_class"].isin(
    ["felsic","mafic","volcanic","carbonate","siliciclastic","metamorphic","evaporite"]
)).sum()
print(f"\n  Resolved to geology: {n_resolved}  ({100*n_resolved/len(df):.1f}%)")
print(f"  Other/unknown:       {len(df) - n_resolved}")
 
# Cross-tab with deposit type
print("\n=== Lithology class × deposit type ===\n")
ct = pd.crosstab(df["Deposit_type"], df["litho_class"], margins=True)
print(ct.to_string())
 

## CELL 16 — Covariate check

Checks whether both covariates (age_category and litho_class)
have sufficient coverage and variation for the HGCTM.
This completes Stage 0 of the protocol: all data layers
are now attached and ready for the model comparison ladder.

INPUT:  df (with age_category and litho_class from Cells 14–15)
OUTPUT: console go/no-go report
/kaggle/working/covariates.csv

In [ ]:
import pandas as pd
import numpy as np
 
# ── Save covariates aligned to count matrix index ─────────────
X_fc = pd.read_csv("/kaggle/working/X_family_copper.csv", index_col=0)
MIN_MINERALS = 3
mask = X_fc.sum(axis=1) >= MIN_MINERALS
 
cov = df.drop_duplicates(subset="Mindat_id").set_index("Mindat_id")[
    ["Deposit_type", "age_category", "litho_class",
     "Latitude", "Longitude", "age_midpoint_ma"]
].reindex(X_fc.index)
 
cov_filtered = cov.loc[mask]
cov_filtered.to_csv("/kaggle/working/covariates.csv")
 
print("=" * 60)
print("  HGCTM PROTOCOL — COVARIATE VIABILITY CHECK")
print("=" * 60)
 
checks = []
 
# 1. Age coverage in modelling subset
n_age_ok = (cov_filtered["age_category"] != "Unknown").sum()
pct_age = 100 * n_age_ok / len(cov_filtered)
ok = pct_age >= 60
checks.append(("Age coverage (modelling set)",
               f"{n_age_ok}/{len(cov_filtered)} ({pct_age:.0f}%)",
               "PASS" if ok else "WARN"))
 
# 2. Age category variety
age_cats = cov_filtered["age_category"].value_counts()
n_age_cats = (age_cats.index != "Unknown").sum()
min_age_cat = age_cats[age_cats.index != "Unknown"].min() if n_age_cats > 0 else 0
ok = n_age_cats >= 3 and min_age_cat >= 20
checks.append(("Age category variety",
               f"{n_age_cats} eras, smallest={min_age_cat}",
               "PASS" if ok else "WARN"))
 
# 3. Lithology coverage
n_litho_ok = cov_filtered["litho_class"].isin(
    ["felsic","mafic","volcanic","carbonate","siliciclastic","metamorphic","evaporite"]
).sum()
pct_litho = 100 * n_litho_ok / len(cov_filtered)
ok = pct_litho >= 50
checks.append(("Lithology coverage (modelling set)",
               f"{n_litho_ok}/{len(cov_filtered)} ({pct_litho:.0f}%)",
               "PASS" if ok else "WARN"))
 
# 4. Lithology variety
litho_cats = cov_filtered["litho_class"].value_counts()
geo_classes = ["felsic","mafic","volcanic","carbonate","siliciclastic","metamorphic"]
n_litho_cats = sum(1 for c in geo_classes if litho_cats.get(c, 0) >= 10)
ok = n_litho_cats >= 3
checks.append(("Lithology class variety",
               f"{n_litho_cats} classes with ≥10 deposits",
               "PASS" if ok else "WARN"))
 
# 5. No single covariate dominates completely
top_litho_pct = 100 * litho_cats.iloc[0] / len(cov_filtered) if len(litho_cats) > 0 else 100
ok = top_litho_pct < 70
checks.append(("Covariate balance",
               f"largest litho class: {litho_cats.index[0]} ({top_litho_pct:.0f}%)",
               "PASS" if ok else "WARN"))
 
print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<35s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False
 
print(f"\n=== Covariates in modelling set ({len(cov_filtered)} deposits) ===")
print(f"\nAge categories:")
for cat, n in cov_filtered["age_category"].value_counts().items():
    print(f"  {cat:<15s}  {n:>5d}")
print(f"\nLithology classes:")
for cls, n in cov_filtered["litho_class"].value_counts().items():
    print(f"  {cls:<15s}  {n:>5d}")
 
print()
if all_pass:
    print("  → COVARIATES VIABLE. All data layers ready for Stage 4 (model ladder).")
else:
    print("  → SOME WARNINGS. Review covariate coverage before proceeding.")
print(f"\n  Saved: /kaggle/working/covariates.csv")
print("=" * 60)

## CELL 17 — Export bundle

Zips all Stage 0 outputs into a single archive for download.
Upload this as a Kaggle dataset so future notebooks can
skip Cells 0–16 and start directly from frozen data.

INPUT:  all CSVs from /kaggle/working/
OUTPUT: /kaggle/working/hgctm_stage0_bundle.zip

In [ ]:
import zipfile
import os

OUT_ZIP = "/kaggle/working/hgctm_stage0_bundle.zip"

files_to_bundle = [
    "/kaggle/working/X_family_primary.csv",
    "/kaggle/working/X_family_copper.csv",
    "/kaggle/working/mineral_to_family.csv",
    "/kaggle/working/deposit_completeness.csv",
    "/kaggle/working/covariates.csv",
    "/kaggle/working/macrostrat_lithology.csv",
    "/kaggle/working/lda_sweep_results.csv",
]

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in files_to_bundle:
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fpath))
            sz = os.path.getsize(fpath) / 1024
            print(f"  ✓ {os.path.basename(fpath):<40s}  {sz:7.1f} KB")
        else:
            print(f"  ✗ {os.path.basename(fpath):<40s}  NOT FOUND")

total_sz = os.path.getsize(OUT_ZIP) / 1024
print(f"\n  Archive: {OUT_ZIP}")
print(f"  Total size: {total_sz:.1f} KB")
print(f"\n  Download this file, then upload as a Kaggle dataset.")
print(f"  In the next notebook, add it as input and load directly.")

## CELL 0 — Stage 0 loader

Loads all frozen Stage 0 outputs from the uploaded dataset.
This is the starting point for the Stage 4 model comparison
notebook. No preprocessing or API calls needed — everything
was frozen in the Stage 0 notebook.

INPUT:  /kaggle/input/datasets/katerynaglin/hgctm-stage0/
OUTPUT: X_fp, X_fc, cov, comp (DataFrames in memory)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

STAGE0 = "/kaggle/input/datasets/katerynaglin/hgctm-stage0"

# Count matrices (deposit × mineral family)
X_fp = pd.read_csv(f"{STAGE0}/X_family_primary.csv", index_col=0)
X_fc = pd.read_csv(f"{STAGE0}/X_family_copper.csv", index_col=0)

# Covariates (deposit type, age, lithology, coordinates)
cov = pd.read_csv(f"{STAGE0}/covariates.csv", index_col=0)

# Completeness variables
comp = pd.read_csv(f"{STAGE0}/deposit_completeness.csv", index_col=0)

# Mineral mapping (for reference / interpretation)
mineral_map = pd.read_csv(f"{STAGE0}/mineral_to_family.csv")

# ── Filter: ≥3 mapped minerals per deposit ────────────────────
MIN_MINERALS = 3
MIN_DEPOSITS = 10

mask = X_fc.sum(axis=1) >= MIN_MINERALS
X_fc_f = X_fc[mask].copy()
X_fp_f = X_fp.reindex(X_fc_f.index).copy()

# Drop sparse families
fc_keep = X_fc_f.columns[(X_fc_f > 0).sum(axis=0) >= MIN_DEPOSITS]
fp_keep = X_fp_f.columns[(X_fp_f > 0).sum(axis=0) >= MIN_DEPOSITS]
X_fc_f = X_fc_f[fc_keep]
X_fp_f = X_fp_f[fp_keep]

# Align covariates
cov_f = cov.reindex(X_fc_f.index).copy()
comp_f = comp.reindex(X_fc_f.index).copy()

# Family names
families_fc = list(X_fc_f.columns)
families_fp = list(X_fp_f.columns)

# ── Summary ───────────────────────────────────────────────────
print("=== HGCTM Stage 4 — Data loaded ===\n")
print(f"  Deposits:          {X_fc_f.shape[0]}")
print(f"  family_copper:     {X_fc_f.shape[1]} families")
print(f"  family_primary:    {X_fp_f.shape[1]} families")
print(f"  Covariates:        age_category, litho_class")
print(f"\n  Deposit types:")
for dt, n in cov_f["Deposit_type"].value_counts().items():
    print(f"    {dt:<25s}  {n:>5d}")
print(f"\n  Age categories:")
for ac, n in cov_f["age_category"].value_counts().items():
    print(f"    {ac:<15s}  {n:>5d}")
print(f"\n  Lithology classes:")
for lc, n in cov_f["litho_class"].value_counts().items():
    print(f"    {lc:<15s}  {n:>5d}")

print(f"\n  ✓ Ready for model comparison ladder.")

## CELL 18 — Pyro setup

Installs Pyro and encodes covariates as integer indices
for the model ladder. Lithology and age categories become
integer-coded vectors that index into deviation parameter
matrices inside the HGCTM.

INPUT:  cov_f (from Cell 17 loader)
OUTPUT: covariate tensors in memory

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pyro-ppl", "-q"])

import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.optim import ClippedAdam
print(f"PyTorch {torch.__version__}, Pyro {pyro.__version__}")

# ── Encode covariates as integer indices ──────────────────────
from sklearn.preprocessing import LabelEncoder

# Lithology
le_litho = LabelEncoder()
cov_f["litho_idx"] = le_litho.fit_transform(cov_f["litho_class"].fillna("unknown"))
litho_classes = list(le_litho.classes_)
L = len(litho_classes)

# Age
le_age = LabelEncoder()
cov_f["age_idx"] = le_age.fit_transform(cov_f["age_category"].fillna("Unknown"))
age_classes = list(le_age.classes_)
A = len(age_classes)

# ── Convert data to tensors ───────────────────────────────────
X_tensor = torch.tensor(X_fc_f.values, dtype=torch.float32)
litho_tensor = torch.tensor(cov_f["litho_idx"].values, dtype=torch.long)
age_tensor = torch.tensor(cov_f["age_idx"].values, dtype=torch.long)
n_deposits, F = X_tensor.shape

K = 7  # locked from Stage 2/3

print(f"\n=== Model ladder inputs ===")
print(f"  Deposits (N):       {n_deposits}")
print(f"  Families (F):       {F}")
print(f"  Topics (K):         {K}")
print(f"  Lithology classes:  {L}  {litho_classes}")
print(f"  Age categories:     {A}  {age_classes}")
print(f"  X tensor shape:     {X_tensor.shape}")


## CELL 19 — M1 baseline

Fits standard LDA (M1) at K=7 on family_copper. This is
the no-covariate baseline that all other models must beat.
Computes held-out log-likelihood via 5-fold CV and stores
the full-data phi matrix for topic comparison.

INPUT:  X_fc_f (from loader), K=7
OUTPUT: m1_results dict, phi_m1 matrix

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import KFold
import numpy as np

K = 7
N_FOLDS = 5
SEEDS = [42, 123, 7]

X_arr = X_fc_f.values.astype(np.float64)

# ── 5-fold held-out log-likelihood ────────────────────────────
print("=== M1: Standard LDA (K=7, no covariates) ===\n")

m1_ll_scores = []
for seed in SEEDS:
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_scores = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_arr)):
        lda = LatentDirichletAllocation(
            n_components=K,
            doc_topic_prior=0.5 / K,
            topic_word_prior=0.01,
            max_iter=300,
            random_state=seed,
            learning_method="batch",
            n_jobs=1,
        )
        lda.fit(X_arr[train_idx])
        ll = lda.score(X_arr[test_idx]) / len(test_idx)
        fold_scores.append(ll)
    mean_ll = np.mean(fold_scores)
    m1_ll_scores.append(mean_ll)
    print(f"  seed={seed}  mean held-out LL/doc = {mean_ll:.2f}")

m1_mean = np.mean(m1_ll_scores)
m1_std = np.std(m1_ll_scores)
print(f"\n  M1 summary: LL/doc = {m1_mean:.2f} ± {m1_std:.2f}")

# ── Fit on full data for topic inspection ─────────────────────
lda_m1 = LatentDirichletAllocation(
    n_components=K,
    doc_topic_prior=0.5 / K,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)
lda_m1.fit(X_arr)
phi_m1 = lda_m1.components_ / lda_m1.components_.sum(axis=1, keepdims=True)
theta_m1 = lda_m1.transform(X_arr)

# Store results
m1_results = {
    "model": "M1_LDA",
    "ll_mean": m1_mean,
    "ll_std": m1_std,
    "phi": phi_m1,
    "theta": theta_m1,
}

print(f"\n  phi shape: {phi_m1.shape}  (K × F)")
print(f"  theta shape: {theta_m1.shape}  (N × K)")

## CELL 20 — HGCTM definition

Defines the Hierarchical Geological Context Topic Model
in Pyro. This single class implements M2 through M6 via
configuration flags:
M2: prevalence_covariates=True,  content_covariates=False
M3: prevalence_covariates=True,  content_covariates=True,  ordered_shrinkage=False
M4: prevalence_covariates=True,  content_covariates=True,  ordered_shrinkage=False (HGCTM arch)
M5: prevalence_covariates=True,  content_covariates=True,  ordered_shrinkage=True
M6: same as M5 but likelihood="multinomial"

INPUT:  nothing (class definition only)
OUTPUT: HGCTM class in memory

In [ ]:
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax


class HGCTM(nn.Module):
    """Hierarchical Geological Context Topic Model."""

    def __init__(self, K, F, L, A,
                 prevalence_covariates=True,
                 content_covariates=True,
                 ordered_shrinkage=True,
                 likelihood="dirichlet_multinomial",
                 sigma_mu=2.0,
                 sigma_litho=1.0,
                 sigma_age=0.5,
                 omega_a_init=0.5,
                 kappa_init=50.0):
        super().__init__()
        self.K = K
        self.F = F
        self.L = L
        self.A = A
        self.prevalence_covariates = prevalence_covariates
        self.content_covariates = content_covariates
        self.ordered_shrinkage = ordered_shrinkage
        self.likelihood = likelihood
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age
        self.n_prev_covariates = L + A if prevalence_covariates else 0

    def model(self, X, litho_idx, age_idx):
        N, F = X.shape
        K = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(K, F),
                self.sigma_mu * torch.ones(K, F)
            ).to_event(2)
        )

        if self.content_covariates:
            delta_litho = pyro.sample(
                "delta_litho",
                dist.Normal(
                    torch.zeros(K, self.L, F),
                    self.sigma_litho * torch.ones(K, self.L, F)
                ).to_event(3)
            )
            delta_age = pyro.sample(
                "delta_age",
                dist.Normal(
                    torch.zeros(K, self.A, F),
                    self.sigma_age * torch.ones(K, self.A, F)
                ).to_event(3)
            )
            omega_a = pyro.sample(
                "omega_a",
                dist.Beta(2.0, 2.0)
            )
        else:
            delta_litho = torch.zeros(K, self.L, F)
            delta_age = torch.zeros(K, self.A, F)
            omega_a = torch.tensor(0.0)

        if self.prevalence_covariates:
            B = pyro.sample(
                "B",
                dist.Normal(
                    torch.zeros(self.L + self.A, K - 1),
                    torch.ones(self.L + self.A, K - 1)
                ).to_event(2)
            )
            intercept = pyro.sample(
                "intercept",
                dist.Normal(
                    torch.zeros(K - 1),
                    torch.ones(K - 1)
                ).to_event(1)
            )

        if self.likelihood == "dirichlet_multinomial":
            log_kappa = pyro.sample(
                "log_kappa",
                dist.Normal(3.5, 1.0)
            )
            kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N):
            if self.prevalence_covariates:
                z_litho = torch.nn.functional.one_hot(litho_idx, self.L).float()
                z_age = torch.nn.functional.one_hot(age_idx, self.A).float()
                z = torch.cat([z_litho, z_age], dim=1)
                eta = z @ B + intercept
                eta_full = torch.cat([eta, torch.zeros(N, 1)], dim=1)
                theta = softmax(eta_full, dim=1)
            else:
                theta = pyro.sample(
                    "theta",
                    dist.Dirichlet((0.5 / K) * torch.ones(K)).expand([N])
                )

            if self.content_covariates:
                litho_dev = delta_litho[:, litho_idx, :].permute(1, 0, 2)
                age_dev = delta_age[:, age_idx, :].permute(1, 0, 2)
                psi = mu.unsqueeze(0) + litho_dev + omega_a * age_dev
                phi = softmax(psi, dim=2)
            else:
                phi = softmax(mu, dim=1).unsqueeze(0).expand(N, K, F)

            p = torch.einsum("nk,nkf->nf", theta, phi)
            p = p.clamp(min=1e-8)
            p = p / p.sum(dim=1, keepdim=True)

            n_i = X.sum(dim=1)

            if self.likelihood == "dirichlet_multinomial":
                alpha = kappa * p
                pyro.sample(
                    "obs",
                    dist.DirichletMultinomial(
                        total_count=n_i.int(),
                        concentration=alpha
                    ),
                    obs=X
                )
            else:
                log_p = torch.log(p)
                log_lik = (X * log_p).sum(dim=1)
                pyro.factor("obs", log_lik.sum())

    def guide(self, X, litho_idx, age_idx):
        N, F = X.shape
        K = self.K

        mu_loc = pyro.param("mu_loc", torch.randn(K, F) * 0.1)
        mu_scale = pyro.param(
            "mu_scale", 0.5 * torch.ones(K, F),
            constraint=torch.distributions.constraints.positive
        )
        pyro.sample("mu", dist.Normal(mu_loc, mu_scale).to_event(2))

        if self.content_covariates:
            dl_loc = pyro.param("delta_litho_loc", torch.zeros(K, self.L, F))
            dl_scale = pyro.param(
                "delta_litho_scale", 0.3 * torch.ones(K, self.L, F),
                constraint=torch.distributions.constraints.positive
            )
            pyro.sample("delta_litho", dist.Normal(dl_loc, dl_scale).to_event(3))

            da_loc = pyro.param("delta_age_loc", torch.zeros(K, self.A, F))
            da_scale = pyro.param(
                "delta_age_scale", 0.2 * torch.ones(K, self.A, F),
                constraint=torch.distributions.constraints.positive
            )
            pyro.sample("delta_age", dist.Normal(da_loc, da_scale).to_event(3))

            omega_a_a = pyro.param(
                "omega_a_a", torch.tensor(2.0),
                constraint=torch.distributions.constraints.positive
            )
            omega_a_b = pyro.param(
                "omega_a_b", torch.tensor(2.0),
                constraint=torch.distributions.constraints.positive
            )
            pyro.sample("omega_a", dist.Beta(omega_a_a, omega_a_b))

        if self.prevalence_covariates:
            B_loc = pyro.param("B_loc", torch.zeros(self.L + self.A, K - 1))
            B_scale = pyro.param(
                "B_scale", 0.5 * torch.ones(self.L + self.A, K - 1),
                constraint=torch.distributions.constraints.positive
            )
            pyro.sample("B", dist.Normal(B_loc, B_scale).to_event(2))

            int_loc = pyro.param("intercept_loc", torch.zeros(K - 1))
            int_scale = pyro.param(
                "intercept_scale", 0.5 * torch.ones(K - 1),
                constraint=torch.distributions.constraints.positive
            )
            pyro.sample("intercept", dist.Normal(int_loc, int_scale).to_event(1))
        else:
            theta_conc = pyro.param(
                "theta_conc",
                torch.ones(N, K),
                constraint=torch.distributions.constraints.positive
            )
            with pyro.plate("deposits", N):
                pyro.sample("theta", dist.Dirichlet(theta_conc))

        if self.likelihood == "dirichlet_multinomial":
            lk_loc = pyro.param("log_kappa_loc", torch.tensor(3.5))
            lk_scale = pyro.param(
                "log_kappa_scale", torch.tensor(0.5),
                constraint=torch.distributions.constraints.positive
            )
            pyro.sample("log_kappa", dist.Normal(lk_loc, lk_scale))


def fit_hgctm(model, X, litho_idx, age_idx,
               n_steps=2000, lr=0.005, print_every=200):
    """Fit an HGCTM model via SVI and return ELBO trace."""
    pyro.clear_param_store()
    optimizer = ClippedAdam({"lr": lr, "betas": (0.9, 0.999)})
    svi = SVI(model.model, model.guide, optimizer, loss=Trace_ELBO())
    losses = []
    for step in range(n_steps):
        loss = svi.step(X, litho_idx, age_idx)
        losses.append(loss)
        if (step + 1) % print_every == 0:
            print(f"    step {step+1:>5d}  ELBO loss = {loss:>12.1f}")
    return losses


def extract_topics(model, X, litho_idx, age_idx):
    """Extract fitted topic parameters from Pyro param store."""
    mu = pyro.param("mu_loc").detach().numpy()
    phi_global = np.exp(mu) / np.exp(mu).sum(axis=1, keepdims=True)
    result = {"phi_global": phi_global, "mu": mu}
    if model.content_covariates:
        result["delta_litho"] = pyro.param("delta_litho_loc").detach().numpy()
        result["delta_age"] = pyro.param("delta_age_loc").detach().numpy()
        result["omega_a"] = (
            pyro.param("omega_a_a").item() /
            (pyro.param("omega_a_a").item() + pyro.param("omega_a_b").item())
        )
    if model.prevalence_covariates:
        result["B"] = pyro.param("B_loc").detach().numpy()
    if model.likelihood == "dirichlet_multinomial":
        result["kappa"] = np.exp(pyro.param("log_kappa_loc").item())
    return result


print("✓ HGCTM class defined. Ready to fit M2–M6.")

## CELL 21 — Model ladder

Fits M2 through M6 and compares all models on training ELBO.
Each model is a configuration of the HGCTM class:
M2: prevalence only (no content deviations)
M3: prevalence + content, equal shrinkage, uniform priors
M4: HGCTM architecture, equal shrinkage (σ²_a = σ²_l)
M5: HGCTM full model, ordered shrinkage (σ²_a < σ²_l)
M6: Same as M5 but Multinomial likelihood (DM ablation)

For fair comparison, all Pyro models use the same number of
SVI steps, learning rate, and random seed.

INPUT:  X_tensor, litho_tensor, age_tensor, K, L, A, F
m1_results (from Cell 19)
OUTPUT: model_results dict, ELBO comparison table

In [ ]:
import time
import numpy as np

N_STEPS = 3000
LR = 0.005
PRINT_EVERY = 500

model_configs = {
    "M2": dict(
        prevalence_covariates=True,
        content_covariates=False,
        ordered_shrinkage=False,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=1.0, sigma_age=1.0,
    ),
    "M3": dict(
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=False,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=1.0, sigma_age=1.0,  # equal shrinkage
    ),
    "M4": dict(
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=False,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=1.0, sigma_age=1.0,  # HGCTM arch, equal
    ),
    "M5": dict(
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=1.0, sigma_age=0.5,  # ORDERED: σ_age < σ_litho
    ),
    "M6": dict(
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="multinomial",  # DM ablation
        sigma_mu=2.0, sigma_litho=1.0, sigma_age=0.5,
    ),
}

# Note: M3 and M4 have the same config here because the
# architectural difference (additive decomposition vs generic
# interaction) is in interpretation, not in the Pyro code.
# Both use additive deviations. The distinction becomes
# meaningful in the topic inspection and geological
# interpretation stages.

model_results = {"M1": m1_results}
model_losses = {}

print("=" * 60)
print("  STAGE 4: MODEL COMPARISON LADDER")
print("=" * 60)
print(f"  N={n_deposits}, F={F}, K={K}, L={L}, A={A}")
print(f"  SVI steps={N_STEPS}, lr={LR}")
print()

for name, cfg in model_configs.items():
    print(f"{'─'*60}")
    print(f"  Fitting {name}: prev={cfg['prevalence_covariates']}, "
          f"content={cfg['content_covariates']}, "
          f"ordered={cfg['ordered_shrinkage']}, "
          f"lik={cfg['likelihood']}")
    print(f"  σ_mu={cfg['sigma_mu']}, σ_litho={cfg['sigma_litho']}, "
          f"σ_age={cfg['sigma_age']}")
    print(f"{'─'*60}")

    t0 = time.time()

    model = HGCTM(
        K=K, F=F, L=L, A=A,
        prevalence_covariates=cfg["prevalence_covariates"],
        content_covariates=cfg["content_covariates"],
        ordered_shrinkage=cfg["ordered_shrinkage"],
        likelihood=cfg["likelihood"],
        sigma_mu=cfg["sigma_mu"],
        sigma_litho=cfg["sigma_litho"],
        sigma_age=cfg["sigma_age"],
    )

    losses = fit_hgctm(
        model, X_tensor, litho_tensor, age_tensor,
        n_steps=N_STEPS, lr=LR, print_every=PRINT_EVERY,
    )

    elapsed = time.time() - t0
    final_elbo = -losses[-1]  # ELBO = -loss
    avg_last100 = -np.mean(losses[-100:])

    # Extract parameters
    params = extract_topics(model, X_tensor, litho_tensor, age_tensor)

    model_results[name] = {
        "model": name,
        "final_elbo": final_elbo,
        "avg_elbo_last100": avg_last100,
        "elapsed_sec": elapsed,
        "losses": losses,
        "params": params,
        "config": cfg,
    }
    model_losses[name] = losses

    print(f"  → {name} done in {elapsed:.0f}s. "
          f"Final ELBO={final_elbo:.0f}, "
          f"avg(last 100)={avg_last100:.0f}")

    if cfg.get("content_covariates") and "omega_a" in params:
        print(f"    ω_a (learned age weight) = {params['omega_a']:.3f}")
    if "kappa" in params:
        print(f"    κ (DM concentration) = {params['kappa']:.1f}")
    print()


# ── Summary table ─────────────────────────────────────────────
print("=" * 60)
print("  MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"\n  {'Model':<8s}  {'ELBO (final)':>14s}  {'ELBO (avg100)':>14s}  {'Time':>6s}")
print(f"  {'─'*48}")

# M1 doesn't have ELBO, report its LL differently
print(f"  {'M1':<8s}  {'(sklearn LL)':>14s}  "
      f"{'LL/doc='+str(round(m1_results['ll_mean'],1)):>14s}  {'─':>6s}")

for name in ["M2", "M3", "M4", "M5", "M6"]:
    r = model_results[name]
    print(f"  {name:<8s}  {r['final_elbo']:>14.0f}  "
          f"{r['avg_elbo_last100']:>14.0f}  "
          f"{r['elapsed_sec']:>5.0f}s")

print()

# ── Key comparisons from protocol ─────────────────────────────
e = lambda n: model_results[n]["avg_elbo_last100"]

print("  Pairwise comparisons (ELBO, higher is better):")
comparisons = [
    ("M2 vs M3", "M2", "M3", "content covariates help?"),
    ("M3 vs M5", "M3", "M5", "HGCTM ordered shrinkage helps?"),
    ("M4 vs M5", "M4", "M5", "ordered > equal shrinkage?"),
    ("M5 vs M6", "M5", "M6", "DM > Multinomial?"),
]
for label, a, b, question in comparisons:
    diff = e(b) - e(a)
    winner = b if diff > 0 else a
    print(f"    {label}: Δ = {diff:+.0f}  → {winner} wins  ({question})")

print()
print("  NOTE: ELBO comparisons are on training data.")
print("  Held-out comparison requires cross-validation (next cell).")
print("=" * 60)

## CELL 21c — Extended run

Continues M3, M4, M5 for 4000 additional SVI steps (total
~7000 effective). Content models have more parameters and
need longer optimisation. Reuses the current param store
state so this is a warm continuation, not a restart.

INPUT:  model configs, X_tensor, litho_tensor, age_tensor
OUTPUT: updated model_results with extended ELBO

In [ ]:
import time

EXTRA_STEPS = 4000

for name, cfg in [("M3", model_configs["M3"]),
                   ("M4", model_configs["M4"]),
                   ("M5", model_configs["M5"])]:
    print(f"\n{'─'*50}")
    print(f"  Extending {name} for {EXTRA_STEPS} more steps...")
    print(f"{'─'*50}")

    t0 = time.time()

    model = HGCTM(
        K=K, F=F, L=L, A=A,
        prevalence_covariates=cfg["prevalence_covariates"],
        content_covariates=cfg["content_covariates"],
        ordered_shrinkage=cfg["ordered_shrinkage"],
        likelihood=cfg["likelihood"],
        sigma_mu=cfg["sigma_mu"],
        sigma_litho=cfg["sigma_litho"],
        sigma_age=cfg["sigma_age"],
    )

    # Continue from existing param store (warm start)
    optimizer = ClippedAdam({"lr": 0.003})  # slightly lower lr for fine-tuning
    svi = SVI(model.model, model.guide, optimizer, loss=Trace_ELBO())

    ext_losses = []
    for step in range(EXTRA_STEPS):
        loss = svi.step(X_tensor, litho_tensor, age_tensor)
        ext_losses.append(loss)
        if (step + 1) % 500 == 0:
            print(f"    step {step+1:>5d}  ELBO loss = {loss:>12.1f}")

    elapsed = time.time() - t0

    # Update results
    all_losses = model_results[name]["losses"] + ext_losses
    model_results[name]["losses"] = all_losses
    model_results[name]["final_elbo"] = -all_losses[-1]
    model_results[name]["avg_elbo_last100"] = -np.mean(all_losses[-100:])
    model_results[name]["params"] = extract_topics(
        model, X_tensor, litho_tensor, age_tensor
    )

    print(f"  → {name} extended. avg(last 100) = {-np.mean(all_losses[-100:]):.0f}")
    if "omega_a" in model_results[name]["params"]:
        print(f"    ω_a = {model_results[name]['params']['omega_a']:.3f}")

# ── Updated comparison ────────────────────────────────────────
print("\n" + "=" * 60)
print("  EXTENDED MODEL COMPARISON")
print("=" * 60)
e = lambda n: model_results[n]["avg_elbo_last100"]
for name in ["M2", "M3", "M4", "M5"]:
    r = model_results[name]
    print(f"  {name}: avg ELBO(last 100) = {r['avg_elbo_last100']:.0f}")

print(f"\n  M2→M5 gap: {e('M5')-e('M2'):+.0f}")
print(f"  M4→M5 (ordered helps?): {e('M5')-e('M4'):+.0f}")
print(f"  M5 ω_a = {model_results['M5']['params']['omega_a']:.3f}")
print("=" * 60)

## CELL 22 — Topic comparison

Compares M2 (prevalence only) and M5 (full HGCTM) on topic
quality: top families per topic, and whether M5's lithology
deviations reveal geologically meaningful structure. This is
the critical test: does the content model discover something
that the simpler model misses?

INPUT:  model_results from Cells 19/21
OUTPUT: side-by-side topic tables + deviation analysis

In [ ]:
import numpy as np

# ── Extract M5 global topics ──────────────────────────────────
mu_m5 = pyro.param("mu_loc").detach().numpy()
phi_m5_global = np.exp(mu_m5) / np.exp(mu_m5).sum(axis=1, keepdims=True)

# M1 topics (from Cell 19)
phi_m1 = m1_results["phi"]

print("=" * 70)
print("  M1 (standard LDA) vs M5 (HGCTM) — GLOBAL TOPIC CORES")
print("=" * 70)

for k in range(K):
    print(f"\n{'─'*70}")
    print(f"  Topic {k+1}")
    print(f"  {'M1 (LDA)':<35s}  {'M5 (HGCTM global core)'}")
    print(f"{'─'*70}")

    top_m1 = np.argsort(phi_m1[k])[::-1][:8]
    top_m5 = np.argsort(phi_m5_global[k])[::-1][:8]

    for rank in range(8):
        m1_fam = families_fc[top_m1[rank]]
        m1_val = phi_m1[k, top_m1[rank]]
        m5_fam = families_fc[top_m5[rank]]
        m5_val = phi_m5_global[k, top_m5[rank]]
        print(f"  {rank+1}. {m1_fam:<22s} {m1_val:.3f}   "
              f"{m5_fam:<22s} {m5_val:.3f}")

# ── M5 lithology deviations ──────────────────────────────────
print("\n" + "=" * 70)
print("  M5 LITHOLOGY DEVIATIONS (δ_litho)")
print("  Positive = family enriched in that host rock")
print("  Negative = family depleted in that host rock")
print("=" * 70)

delta_litho = pyro.param("delta_litho_loc").detach().numpy()  # (K, L, F)

# For each topic, show the strongest litho-family deviations
for k in range(K):
    print(f"\n--- Topic {k+1} ---")
    # Flatten deviations for this topic: (L, F)
    dev_k = delta_litho[k]  # (L, F)

    # Find top positive and negative deviations
    deviations = []
    for l_idx in range(len(litho_classes)):
        for f_idx in range(len(families_fc)):
            val = dev_k[l_idx, f_idx]
            if abs(val) > 0.3:  # threshold for notable deviations
                deviations.append((litho_classes[l_idx],
                                   families_fc[f_idx],
                                   val))

    deviations.sort(key=lambda x: abs(x[2]), reverse=True)
    if deviations:
        for litho, fam, val in deviations[:6]:
            sign = "+" if val > 0 else ""
            print(f"  {litho:<15s}  {fam:<30s}  {sign}{val:.2f}")
    else:
        print("  (no deviations > 0.3)")

# ── M5 age deviations magnitude vs litho ──────────────────────
delta_age = pyro.param("delta_age_loc").detach().numpy()  # (K, A, F)
omega_a_val = model_results["M5"]["params"]["omega_a"]

litho_mag = np.abs(delta_litho).mean()
age_mag = np.abs(delta_age).mean()
age_effective = age_mag * omega_a_val

print(f"\n{'='*70}")
print(f"  DEVIATION MAGNITUDE SUMMARY")
print(f"{'='*70}")
print(f"  |δ_litho| mean:           {litho_mag:.4f}")
print(f"  |δ_age| mean:             {age_mag:.4f}")
print(f"  ω_a (learned):            {omega_a_val:.3f}")
print(f"  Effective |ω_a × δ_age|:  {age_effective:.4f}")
print(f"  Ratio litho/age_eff:      {litho_mag/max(age_effective, 1e-8):.1f}×")
print(f"\n  → Lithology deviations are {litho_mag/max(age_effective, 1e-8):.1f}× "
      f"stronger than age deviations")
print(f"  → This confirms the geological prior: host rock matters more than age")
print(f"{'='*70}")

## CELL 23 — Tighter refit

Refits M5 with tighter deviation priors (σ_litho=0.3,
σ_age=0.15) so that global topic cores carry the main
mineralogical signal and deviations only modulate it.
Also fixes M6: the pyro.factor inside pyro.plate must
receive per-deposit log-likelihoods (not .sum()), because
the plate already handles summation across deposits.

INPUT:  X_tensor, litho_tensor, age_tensor, K, L, A, F
OUTPUT: updated model_results for M5_tight and M6_fixed

In [ ]:
import time
import numpy as np

N_STEPS = 5000  # longer run for tight priors
LR = 0.005
PRINT_EVERY = 500


# ── Fix M6: subclass with corrected factor ────────────────────
class HGCTM_M6_fixed(HGCTM):
    """M6 with corrected multinomial log-likelihood.
    pyro.factor inside pyro.plate must be per-deposit, not summed.
    """
    def model(self, X, litho_idx, age_idx):
        N, F = X.shape
        K = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(torch.zeros(K, F),
                        self.sigma_mu * torch.ones(K, F)).to_event(2)
        )

        if self.content_covariates:
            delta_litho = pyro.sample(
                "delta_litho",
                dist.Normal(torch.zeros(K, self.L, F),
                            self.sigma_litho * torch.ones(K, self.L, F)).to_event(3)
            )
            delta_age = pyro.sample(
                "delta_age",
                dist.Normal(torch.zeros(K, self.A, F),
                            self.sigma_age * torch.ones(K, self.A, F)).to_event(3)
            )
            omega_a = pyro.sample("omega_a", dist.Beta(2.0, 2.0))
        else:
            delta_litho = torch.zeros(K, self.L, F)
            delta_age = torch.zeros(K, self.A, F)
            omega_a = torch.tensor(0.0)

        if self.prevalence_covariates:
            B = pyro.sample(
                "B",
                dist.Normal(torch.zeros(self.L + self.A, K - 1),
                            torch.ones(self.L + self.A, K - 1)).to_event(2)
            )
            intercept = pyro.sample(
                "intercept",
                dist.Normal(torch.zeros(K - 1),
                            torch.ones(K - 1)).to_event(1)
            )

        with pyro.plate("deposits", N):
            if self.prevalence_covariates:
                z_litho = torch.nn.functional.one_hot(litho_idx, self.L).float()
                z_age = torch.nn.functional.one_hot(age_idx, self.A).float()
                z = torch.cat([z_litho, z_age], dim=1)
                eta = z @ B + intercept
                eta_full = torch.cat([eta, torch.zeros(N, 1)], dim=1)
                theta = softmax(eta_full, dim=1)
            else:
                theta = pyro.sample(
                    "theta",
                    dist.Dirichlet((0.5 / K) * torch.ones(K)).expand([N])
                )

            if self.content_covariates:
                litho_dev = delta_litho[:, litho_idx, :].permute(1, 0, 2)
                age_dev = delta_age[:, age_idx, :].permute(1, 0, 2)
                psi = mu.unsqueeze(0) + litho_dev + omega_a * age_dev
                phi = softmax(psi, dim=2)
            else:
                phi = softmax(mu, dim=1).unsqueeze(0).expand(N, K, F)

            p = torch.einsum("nk,nkf->nf", theta, phi)
            p = p.clamp(min=1e-8)
            p = p / p.sum(dim=1, keepdim=True)

            # Per-deposit log-likelihood (NO .sum() — plate handles it)
            log_lik = (X * torch.log(p)).sum(dim=1)  # (N,)
            pyro.factor("obs", log_lik)


# ══════════════════════════════════════════════════════════════
# FIT M5_tight: ordered shrinkage with tighter priors
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("  Fitting M5_tight: σ_litho=0.3, σ_age=0.15")
print("=" * 60)

t0 = time.time()
model_m5t = HGCTM(
    K=K, F=F, L=L, A=A,
    prevalence_covariates=True,
    content_covariates=True,
    ordered_shrinkage=True,
    likelihood="dirichlet_multinomial",
    sigma_mu=2.0,
    sigma_litho=0.3,
    sigma_age=0.15,
)

losses_m5t = fit_hgctm(
    model_m5t, X_tensor, litho_tensor, age_tensor,
    n_steps=N_STEPS, lr=LR, print_every=PRINT_EVERY,
)
elapsed_m5t = time.time() - t0
params_m5t = extract_topics(model_m5t, X_tensor, litho_tensor, age_tensor)

print(f"\n  M5_tight done in {elapsed_m5t:.0f}s")
print(f"  avg ELBO(last 100) = {-np.mean(losses_m5t[-100:]):.0f}")
print(f"  ω_a = {params_m5t['omega_a']:.3f}")
print(f"  κ = {params_m5t['kappa']:.1f}")

model_results["M5_tight"] = {
    "model": "M5_tight",
    "final_elbo": -losses_m5t[-1],
    "avg_elbo_last100": -np.mean(losses_m5t[-100:]),
    "elapsed_sec": elapsed_m5t,
    "losses": losses_m5t,
    "params": params_m5t,
}


# ══════════════════════════════════════════════════════════════
# FIT M6_fixed: multinomial with corrected factor
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  Fitting M6_fixed: Multinomial (corrected), σ_litho=0.3, σ_age=0.15")
print("=" * 60)

t0 = time.time()
model_m6f = HGCTM_M6_fixed(
    K=K, F=F, L=L, A=A,
    prevalence_covariates=True,
    content_covariates=True,
    ordered_shrinkage=True,
    likelihood="multinomial",
    sigma_mu=2.0,
    sigma_litho=0.3,
    sigma_age=0.15,
)

losses_m6f = fit_hgctm(
    model_m6f, X_tensor, litho_tensor, age_tensor,
    n_steps=N_STEPS, lr=LR, print_every=PRINT_EVERY,
)
elapsed_m6f = time.time() - t0
params_m6f = extract_topics(model_m6f, X_tensor, litho_tensor, age_tensor)

print(f"\n  M6_fixed done in {elapsed_m6f:.0f}s")
print(f"  avg ELBO(last 100) = {-np.mean(losses_m6f[-100:]):.0f}")
print(f"  ω_a = {params_m6f['omega_a']:.3f}")

model_results["M6_fixed"] = {
    "model": "M6_fixed",
    "final_elbo": -losses_m6f[-1],
    "avg_elbo_last100": -np.mean(losses_m6f[-100:]),
    "elapsed_sec": elapsed_m6f,
    "losses": losses_m6f,
    "params": params_m6f,
}


# ══════════════════════════════════════════════════════════════
# INSPECT M5_tight TOPICS
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  M5_tight GLOBAL TOPIC CORES")
print("=" * 60)

phi_m5t = params_m5t["phi_global"]
for k in range(K):
    top_idx = np.argsort(phi_m5t[k])[::-1][:10]
    print(f"\n--- Topic {k+1} ---")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank:2d}. {families_fc[idx]:<35s}  {phi_m5t[k, idx]:.3f}")


# ══════════════════════════════════════════════════════════════
# M5_tight LITHOLOGY DEVIATIONS
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  M5_tight LITHOLOGY DEVIATIONS")
print("=" * 60)

delta_litho_t = params_m5t["delta_litho"]  # (K, L, F)
delta_age_t = params_m5t["delta_age"]      # (K, A, F)

for k in range(K):
    print(f"\n--- Topic {k+1} ---")
    dev_k = delta_litho_t[k]
    deviations = []
    for l_idx in range(len(litho_classes)):
        for f_idx in range(len(families_fc)):
            val = dev_k[l_idx, f_idx]
            if abs(val) > 0.15:  # tighter threshold for tighter priors
                deviations.append((litho_classes[l_idx],
                                   families_fc[f_idx], val))
    deviations.sort(key=lambda x: abs(x[2]), reverse=True)
    if deviations:
        for litho, fam, val in deviations[:6]:
            sign = "+" if val > 0 else ""
            print(f"  {litho:<15s}  {fam:<30s}  {sign}{val:.3f}")
    else:
        print("  (no deviations > 0.15)")


# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
litho_mag_t = np.abs(delta_litho_t).mean()
age_mag_t = np.abs(delta_age_t).mean()
omega_t = params_m5t["omega_a"]
age_eff_t = age_mag_t * omega_t

print(f"\n{'=' * 60}")
print("  FULL COMPARISON (all models)")
print("=" * 60)
print(f"\n  {'Model':<12s}  {'ELBO(avg100)':>14s}  {'Notes'}")
print(f"  {'─'*60}")
print(f"  {'M1':<12s}  {'LL/doc=-66.6':>14s}  sklearn LDA (different scale)")
print(f"  {'M2':<12s}  {model_results['M2']['avg_elbo_last100']:>14.0f}  prevalence only")
print(f"  {'M5':<12s}  {model_results['M5']['avg_elbo_last100']:>14.0f}  ordered σ_l=1.0, σ_a=0.5")
print(f"  {'M5_tight':<12s}  {model_results['M5_tight']['avg_elbo_last100']:>14.0f}  ordered σ_l=0.3, σ_a=0.15 ◄")
print(f"  {'M6_fixed':<12s}  {model_results['M6_fixed']['avg_elbo_last100']:>14.0f}  Multinomial (corrected)")

print(f"\n  M5_tight vs M2:         {model_results['M5_tight']['avg_elbo_last100'] - model_results['M2']['avg_elbo_last100']:+.0f}")
print(f"  M5_tight vs M6_fixed:   {model_results['M5_tight']['avg_elbo_last100'] - model_results['M6_fixed']['avg_elbo_last100']:+.0f}  (DM vs Multinomial)")

print(f"\n  M5_tight deviation magnitudes:")
print(f"    |δ_litho| mean:          {litho_mag_t:.4f}")
print(f"    |δ_age| mean:            {age_mag_t:.4f}")
print(f"    ω_a:                     {omega_t:.3f}")
print(f"    Effective |ω_a × δ_age|: {age_eff_t:.4f}")
print(f"    Ratio litho/age_eff:     {litho_mag_t/max(age_eff_t,1e-8):.1f}×")
print("=" * 60)

## CELL 24 — Novel models

Three novel model variants that address identified weaknesses:

M7: Sparse-deviation HGCTM — learns per-family relevance
weights τ_f that gate lithology deviations. Families that
don't respond to host rock get τ_f→0 automatically.
KEY NOVELTY: produces a ranked list of which mineral
families are geologically context-sensitive.

M8: Completeness-aware HGCTM — deposit-specific concentration
κ_i = κ_base × sigmoid(w · log(n_minerals_i) + b).
Data-rich deposits get tighter concentration (more
confidence), data-sparse deposits get looser (more
uncertainty). Addresses the GCDD's main data quality issue.

M9: Semi-supervised HGCTM — adds a deposit-type prediction
head. Loss = ELBO + λ × classification cross-entropy.
Topics are encouraged to be discriminative, bridging
generative topic modelling and supervised classification.

INPUT:  X_tensor, litho_tensor, age_tensor, cov_f, comp_f
OUTPUT: model_results updated with M7, M8, M9

In [ ]:
import time
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax

N_STEPS = 5000
LR = 0.005
PRINT_EVERY = 500


# ══════════════════════════════════════════════════════════════
# M7: SPARSE-DEVIATION HGCTM
# ══════════════════════════════════════════════════════════════

class HGCTM_Sparse(HGCTM):
    """HGCTM with per-family relevance gating on deviations.

    Each family f has a learned relevance weight tau_f in [0,1].
    The effective deviation becomes:
        psi_{ik} = mu_k + tau * delta^l_{k,l_i} + omega_a * tau * delta^a_{k,a_i}
    where tau is broadcast across (K, L) but specific to F.

    Families with tau_f ≈ 0 are lithology-invariant.
    Families with tau_f ≈ 1 are lithology-sensitive.
    """

    def model(self, X, litho_idx, age_idx):
        N, F = X.shape
        K = self.K

        # Global topic cores
        mu = pyro.sample(
            "mu",
            dist.Normal(torch.zeros(K, F),
                        self.sigma_mu * torch.ones(K, F)).to_event(2)
        )

        # Content deviations (always present in M7)
        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(torch.zeros(K, self.L, F),
                        self.sigma_litho * torch.ones(K, self.L, F)).to_event(3)
        )
        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(torch.zeros(K, self.A, F),
                        self.sigma_age * torch.ones(K, self.A, F)).to_event(3)
        )
        omega_a = pyro.sample("omega_a", dist.Beta(2.0, 2.0))

        # Per-family relevance gate: tau_f ~ Beta(1, 3)
        # Prior encourages sparsity (mean = 0.25, most families → low tau)
        tau = pyro.sample(
            "tau",
            dist.Beta(1.0 * torch.ones(F),
                      3.0 * torch.ones(F)).to_event(1)
        )

        # Prevalence
        B = pyro.sample(
            "B",
            dist.Normal(torch.zeros(self.L + self.A, K - 1),
                        torch.ones(self.L + self.A, K - 1)).to_event(2)
        )
        intercept = pyro.sample(
            "intercept",
            dist.Normal(torch.zeros(K - 1),
                        torch.ones(K - 1)).to_event(1)
        )

        # DM concentration
        log_kappa = pyro.sample("log_kappa", dist.Normal(3.5, 1.0))
        kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N):
            # Prevalence
            z_litho = torch.nn.functional.one_hot(litho_idx, self.L).float()
            z_age = torch.nn.functional.one_hot(age_idx, self.A).float()
            z = torch.cat([z_litho, z_age], dim=1)
            eta = z @ B + intercept
            eta_full = torch.cat([eta, torch.zeros(N, 1)], dim=1)
            theta = softmax(eta_full, dim=1)

            # Content with sparse gating
            litho_dev = delta_litho[:, litho_idx, :].permute(1, 0, 2)  # (N, K, F)
            age_dev = delta_age[:, age_idx, :].permute(1, 0, 2)        # (N, K, F)

            # tau gates the deviations per family
            # tau shape: (F,) → broadcast to (N, K, F)
            gated_litho = tau.unsqueeze(0).unsqueeze(0) * litho_dev
            gated_age = tau.unsqueeze(0).unsqueeze(0) * age_dev

            psi = mu.unsqueeze(0) + gated_litho + omega_a * gated_age
            phi = softmax(psi, dim=2)

            # Mixture
            p = torch.einsum("nk,nkf->nf", theta, phi)
            p = p.clamp(min=1e-8)
            p = p / p.sum(dim=1, keepdim=True)

            # DM likelihood
            n_i = X.sum(dim=1)
            alpha = kappa * p
            pyro.sample("obs",
                        dist.DirichletMultinomial(total_count=n_i.int(),
                                                  concentration=alpha),
                        obs=X)

    def guide(self, X, litho_idx, age_idx):
        N, F = X.shape
        K = self.K

        # mu
        mu_loc = pyro.param("mu_loc", torch.randn(K, F) * 0.1)
        mu_scale = pyro.param("mu_scale", 0.5 * torch.ones(K, F),
                              constraint=torch.distributions.constraints.positive)
        pyro.sample("mu", dist.Normal(mu_loc, mu_scale).to_event(2))

        # delta_litho
        dl_loc = pyro.param("delta_litho_loc", torch.zeros(K, self.L, F))
        dl_scale = pyro.param("delta_litho_scale", 0.2 * torch.ones(K, self.L, F),
                              constraint=torch.distributions.constraints.positive)
        pyro.sample("delta_litho", dist.Normal(dl_loc, dl_scale).to_event(3))

        # delta_age
        da_loc = pyro.param("delta_age_loc", torch.zeros(K, self.A, F))
        da_scale = pyro.param("delta_age_scale", 0.1 * torch.ones(K, self.A, F),
                              constraint=torch.distributions.constraints.positive)
        pyro.sample("delta_age", dist.Normal(da_loc, da_scale).to_event(3))

        # omega_a
        omega_a_a = pyro.param("omega_a_a", torch.tensor(2.0),
                               constraint=torch.distributions.constraints.positive)
        omega_a_b = pyro.param("omega_a_b", torch.tensor(2.0),
                               constraint=torch.distributions.constraints.positive)
        pyro.sample("omega_a", dist.Beta(omega_a_a, omega_a_b))

        # tau (per-family relevance)
        tau_a = pyro.param("tau_a", 1.0 * torch.ones(F),
                           constraint=torch.distributions.constraints.positive)
        tau_b = pyro.param("tau_b", 3.0 * torch.ones(F),
                           constraint=torch.distributions.constraints.positive)
        pyro.sample("tau", dist.Beta(tau_a, tau_b).to_event(1))

        # B, intercept
        B_loc = pyro.param("B_loc", torch.zeros(self.L + self.A, K - 1))
        B_scale = pyro.param("B_scale", 0.5 * torch.ones(self.L + self.A, K - 1),
                             constraint=torch.distributions.constraints.positive)
        pyro.sample("B", dist.Normal(B_loc, B_scale).to_event(2))

        int_loc = pyro.param("intercept_loc", torch.zeros(K - 1))
        int_scale = pyro.param("intercept_scale", 0.5 * torch.ones(K - 1),
                               constraint=torch.distributions.constraints.positive)
        pyro.sample("intercept", dist.Normal(int_loc, int_scale).to_event(1))

        # kappa
        lk_loc = pyro.param("log_kappa_loc", torch.tensor(3.5))
        lk_scale = pyro.param("log_kappa_scale", torch.tensor(0.5),
                              constraint=torch.distributions.constraints.positive)
        pyro.sample("log_kappa", dist.Normal(lk_loc, lk_scale))


# ══════════════════════════════════════════════════════════════
# M8: COMPLETENESS-AWARE HGCTM
# ══════════════════════════════════════════════════════════════

class HGCTM_Completeness(HGCTM):
    """HGCTM with deposit-specific DM concentration.

    kappa_i = kappa_base * sigmoid(w * log(n_minerals_i + 1) + b)
    so data-rich deposits have tighter concentration (more
    confidence) and data-sparse deposits have looser (more
    uncertainty in proportions).
    """

    def __init__(self, *args, n_minerals=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Pre-compute log-completeness as a fixed covariate
        if n_minerals is not None:
            self.log_completeness = torch.log(n_minerals.float() + 1.0)
        else:
            self.log_completeness = None

    def model(self, X, litho_idx, age_idx):
        N, F = X.shape
        K = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(torch.zeros(K, F),
                        self.sigma_mu * torch.ones(K, F)).to_event(2)
        )

        if self.content_covariates:
            delta_litho = pyro.sample(
                "delta_litho",
                dist.Normal(torch.zeros(K, self.L, F),
                            self.sigma_litho * torch.ones(K, self.L, F)).to_event(3)
            )
            delta_age = pyro.sample(
                "delta_age",
                dist.Normal(torch.zeros(K, self.A, F),
                            self.sigma_age * torch.ones(K, self.A, F)).to_event(3)
            )
            omega_a = pyro.sample("omega_a", dist.Beta(2.0, 2.0))
        else:
            delta_litho = torch.zeros(K, self.L, F)
            delta_age = torch.zeros(K, self.A, F)
            omega_a = torch.tensor(0.0)

        B = pyro.sample(
            "B",
            dist.Normal(torch.zeros(self.L + self.A, K - 1),
                        torch.ones(self.L + self.A, K - 1)).to_event(2)
        )
        intercept = pyro.sample(
            "intercept",
            dist.Normal(torch.zeros(K - 1),
                        torch.ones(K - 1)).to_event(1)
        )

        # Deposit-specific kappa
        log_kappa_base = pyro.sample("log_kappa", dist.Normal(3.5, 1.0))
        kappa_base = torch.exp(log_kappa_base)

        # Completeness scaling parameters
        w_comp = pyro.sample("w_comp", dist.Normal(1.0, 0.5))
        b_comp = pyro.sample("b_comp", dist.Normal(0.0, 1.0))

        with pyro.plate("deposits", N):
            # Prevalence
            z_litho = torch.nn.functional.one_hot(litho_idx, self.L).float()
            z_age = torch.nn.functional.one_hot(age_idx, self.A).float()
            z = torch.cat([z_litho, z_age], dim=1)
            eta = z @ B + intercept
            eta_full = torch.cat([eta, torch.zeros(N, 1)], dim=1)
            theta = softmax(eta_full, dim=1)

            # Content
            if self.content_covariates:
                litho_dev = delta_litho[:, litho_idx, :].permute(1, 0, 2)
                age_dev = delta_age[:, age_idx, :].permute(1, 0, 2)
                psi = mu.unsqueeze(0) + litho_dev + omega_a * age_dev
                phi = softmax(psi, dim=2)
            else:
                phi = softmax(mu, dim=1).unsqueeze(0).expand(N, K, F)

            p = torch.einsum("nk,nkf->nf", theta, phi)
            p = p.clamp(min=1e-8)
            p = p / p.sum(dim=1, keepdim=True)

            # Deposit-specific kappa
            if self.log_completeness is not None:
                scale = torch.sigmoid(w_comp * self.log_completeness + b_comp)
            else:
                scale = torch.ones(N)
            kappa_i = kappa_base * (0.5 + scale)  # range [0.5*base, 1.5*base]

            n_i = X.sum(dim=1)
            alpha = kappa_i.unsqueeze(1) * p
            pyro.sample("obs",
                        dist.DirichletMultinomial(total_count=n_i.int(),
                                                  concentration=alpha),
                        obs=X)

    def guide(self, X, litho_idx, age_idx):
        # Reuse parent guide for most parameters
        super().guide(X, litho_idx, age_idx)

        # Additional completeness parameters
        w_loc = pyro.param("w_comp_loc", torch.tensor(1.0))
        w_scale = pyro.param("w_comp_scale", torch.tensor(0.3),
                             constraint=torch.distributions.constraints.positive)
        pyro.sample("w_comp", dist.Normal(w_loc, w_scale))

        b_loc = pyro.param("b_comp_loc", torch.tensor(0.0))
        b_scale = pyro.param("b_comp_scale", torch.tensor(0.5),
                             constraint=torch.distributions.constraints.positive)
        pyro.sample("b_comp", dist.Normal(b_loc, b_scale))


# ══════════════════════════════════════════════════════════════
# FIT ALL THREE
# ══════════════════════════════════════════════════════════════

# Prepare completeness tensor for M8
n_minerals_tensor = torch.tensor(
    comp_f["n_minerals_mapped"].values, dtype=torch.float32
)

# Prepare deposit type labels for M9 (used later if needed)
from sklearn.preprocessing import LabelEncoder
le_dtype = LabelEncoder()
dtype_labels = le_dtype.fit_transform(cov_f["Deposit_type"].fillna("Unknown"))
dtype_tensor = torch.tensor(dtype_labels, dtype=torch.long)
n_types = len(le_dtype.classes_)

novel_models = {}

# ── M7: Sparse deviations ─────────────────────────────────────
print("=" * 60)
print("  Fitting M7: Sparse-deviation HGCTM")
print("  (per-family relevance gate τ_f)")
print("=" * 60)

t0 = time.time()
model_m7 = HGCTM_Sparse(
    K=K, F=F, L=L, A=A,
    prevalence_covariates=True,
    content_covariates=True,
    ordered_shrinkage=True,
    likelihood="dirichlet_multinomial",
    sigma_mu=2.0,
    sigma_litho=0.3,
    sigma_age=0.15,
)

losses_m7 = fit_hgctm(
    model_m7, X_tensor, litho_tensor, age_tensor,
    n_steps=N_STEPS, lr=LR, print_every=PRINT_EVERY,
)
elapsed_m7 = time.time() - t0

# Extract tau (family relevance)
tau_a = pyro.param("tau_a").detach()
tau_b = pyro.param("tau_b").detach()
tau_mean = (tau_a / (tau_a + tau_b)).numpy()

params_m7 = extract_topics(model_m7, X_tensor, litho_tensor, age_tensor)
params_m7["tau"] = tau_mean

novel_models["M7"] = {
    "model": "M7_sparse",
    "final_elbo": -losses_m7[-1],
    "avg_elbo_last100": -np.mean(losses_m7[-100:]),
    "elapsed_sec": elapsed_m7,
    "losses": losses_m7,
    "params": params_m7,
}

print(f"\n  M7 done in {elapsed_m7:.0f}s")
print(f"  avg ELBO(last 100) = {-np.mean(losses_m7[-100:]):.0f}")
print(f"  ω_a = {params_m7['omega_a']:.3f}")
print(f"  κ = {params_m7['kappa']:.1f}")

# Print family relevance ranking
print(f"\n  Family relevance τ_f (higher = more lithology-sensitive):")
ranking = sorted(zip(families_fc, tau_mean), key=lambda x: -x[1])
for fam, t in ranking:
    bar = "█" * int(t * 40)
    label = "◄ sensitive" if t > 0.4 else ("  invariant" if t < 0.15 else "")
    print(f"    {fam:<35s}  τ={t:.3f}  {bar}  {label}")


# ── M8: Completeness-aware ────────────────────────────────────
print(f"\n{'=' * 60}")
print("  Fitting M8: Completeness-aware HGCTM")
print("  (deposit-specific κ_i)")
print("=" * 60)

t0 = time.time()
model_m8 = HGCTM_Completeness(
    K=K, F=F, L=L, A=A,
    prevalence_covariates=True,
    content_covariates=True,
    ordered_shrinkage=True,
    likelihood="dirichlet_multinomial",
    sigma_mu=2.0,
    sigma_litho=0.3,
    sigma_age=0.15,
    n_minerals=n_minerals_tensor,
)

losses_m8 = fit_hgctm(
    model_m8, X_tensor, litho_tensor, age_tensor,
    n_steps=N_STEPS, lr=LR, print_every=PRINT_EVERY,
)
elapsed_m8 = time.time() - t0

params_m8 = extract_topics(model_m8, X_tensor, litho_tensor, age_tensor)
w_comp = pyro.param("w_comp_loc").item()
b_comp = pyro.param("b_comp_loc").item()
params_m8["w_comp"] = w_comp
params_m8["b_comp"] = b_comp

novel_models["M8"] = {
    "model": "M8_completeness",
    "final_elbo": -losses_m8[-1],
    "avg_elbo_last100": -np.mean(losses_m8[-100:]),
    "elapsed_sec": elapsed_m8,
    "losses": losses_m8,
    "params": params_m8,
}

print(f"\n  M8 done in {elapsed_m8:.0f}s")
print(f"  avg ELBO(last 100) = {-np.mean(losses_m8[-100:]):.0f}")
print(f"  ω_a = {params_m8['omega_a']:.3f}")
print(f"  κ_base = {params_m8['kappa']:.1f}")
print(f"  w_comp = {w_comp:.3f}  (positive = more minerals → tighter κ)")
print(f"  b_comp = {b_comp:.3f}")


# ══════════════════════════════════════════════════════════════
# FULL COMPARISON
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  FULL MODEL COMPARISON (all models)")
print("=" * 60)

all_models = {
    "M2": model_results["M2"],
    "M5_tight": model_results["M5_tight"],
    "M7_sparse": novel_models["M7"],
    "M8_complete": novel_models["M8"],
}

print(f"\n  {'Model':<14s}  {'ELBO(avg100)':>14s}  {'Notes'}")
print(f"  {'─'*65}")
print(f"  {'M1':<14s}  {'(different)':>14s}  sklearn LDA baseline")
print(f"  {'M2':<14s}  {model_results['M2']['avg_elbo_last100']:>14.0f}  prevalence only")
print(f"  {'M5_tight':<14s}  {model_results['M5_tight']['avg_elbo_last100']:>14.0f}  HGCTM tight priors")
print(f"  {'M7_sparse':<14s}  {novel_models['M7']['avg_elbo_last100']:>14.0f}  sparse deviations ◄")
print(f"  {'M8_complete':<14s}  {novel_models['M8']['avg_elbo_last100']:>14.0f}  completeness-aware")

best_name = max(all_models, key=lambda k: all_models[k]["avg_elbo_last100"])
print(f"\n  Best model: {best_name}")

# Key comparisons
e = lambda n: all_models[n]["avg_elbo_last100"] if n in all_models else model_results[n]["avg_elbo_last100"]
print(f"\n  Pairwise vs M2 (prevalence only):")
for name in ["M5_tight", "M7_sparse", "M8_complete"]:
    diff = all_models[name]["avg_elbo_last100"] - model_results["M2"]["avg_elbo_last100"]
    print(f"    {name} - M2 = {diff:+.0f}")

print("=" * 60)

## CELL 25 — Topic stability

Stage 7 of the HGCTM protocol (HARD GATE).
Bootstrap resampling: draw deposits with replacement,
refit M1 (LDA) and M7 (sparse HGCTM), align topics to a
reference via Hungarian algorithm on cosine similarity.
Reports per-topic recovery rate and mean cosine similarity.

Pass criterion: ≥70% of topics recur in ≥80% of runs
with mean cosine similarity > 0.80.

Expected runtime: ~40 minutes (20 bootstraps × 2 models).
M1 uses sklearn (fast), M7 uses Pyro at 3000 steps.

INPUT:  X_fc_f, litho_tensor, age_tensor, families_fc
phi_m1 (from Cell 19), M7 params (from Cell 24)
OUTPUT: stability comparison table + go/no-go

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.decomposition import LatentDirichletAllocation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
import torch
import pyro
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────
N_BOOT = 20
K = 7
M7_STEPS = 5000  # shorter for bootstrap (converged by 3000)
M7_LR = 0.005
SEED_BASE = 100

X_arr = X_fc_f.values.astype(np.float64)
N, F = X_arr.shape

# ── Reference phi matrices ────────────────────────────────────
# M1 reference: from full-data fit in Cell 19
phi_ref_m1 = phi_m1.copy()  # (K, F), already normalised

# M7 reference: from full-data fit in Cell 24
phi_ref_m7 = novel_models["M7"]["params"]["phi_global"].copy()  # (K, F)


def cosine_sim(a, b):
    """Cosine similarity between two vectors."""
    d = cosine(a, b)
    return 1.0 - d


def match_topics(phi_boot, phi_ref):
    """Align bootstrap topics to reference via Hungarian algorithm.
    Returns: permutation indices, per-topic cosine similarities.
    """
    K_boot, K_ref = phi_boot.shape[0], phi_ref.shape[0]
    # Cost matrix: negative cosine similarity
    cost = np.zeros((K_boot, K_ref))
    for i in range(K_boot):
        for j in range(K_ref):
            cost[i, j] = -cosine_sim(phi_boot[i], phi_ref[j])
    row_ind, col_ind = linear_sum_assignment(cost)
    similarities = [-cost[r, c] for r, c in zip(row_ind, col_ind)]
    return col_ind, similarities


def top5_match(phi_boot_k, phi_ref_k, families):
    """Check if top-5 families overlap between boot and ref."""
    top5_boot = set(np.argsort(phi_boot_k)[::-1][:5])
    top5_ref = set(np.argsort(phi_ref_k)[::-1][:5])
    return len(top5_boot & top5_ref)


# ── Storage ───────────────────────────────────────────────────
# Per-topic, per-bootstrap: cosine similarity and top-5 overlap
m1_sims = np.zeros((N_BOOT, K))   # cosine sim per topic per boot
m7_sims = np.zeros((N_BOOT, K))
m1_top5 = np.zeros((N_BOOT, K))   # top-5 overlap count (0-5)
m7_top5 = np.zeros((N_BOOT, K))

print("=" * 60)
print("  STAGE 7: TOPIC STABILITY (BOOTSTRAP)")
print(f"  {N_BOOT} resamples, K={K}")
print("=" * 60)

t0_total = time.time()

for b in range(N_BOOT):
    seed = SEED_BASE + b
    rng = np.random.RandomState(seed)
    boot_idx = rng.choice(N, size=N, replace=True)

    X_boot = X_arr[boot_idx]

    # ── M1: sklearn LDA ──────────────────────────────────────
    lda_boot = LatentDirichletAllocation(
        n_components=K,
        doc_topic_prior=0.5 / K,
        topic_word_prior=0.01,
        max_iter=300,
        random_state=seed,
        learning_method="batch",
        n_jobs=1,
    )
    lda_boot.fit(X_boot)
    phi_boot_m1 = lda_boot.components_ / lda_boot.components_.sum(axis=1, keepdims=True)

    # Match to reference
    perm_m1, sims_m1 = match_topics(phi_boot_m1, phi_ref_m1)
    for k_ref, (k_boot, sim) in enumerate(zip(perm_m1, sims_m1)):
        # perm maps boot topic → ref topic
        pass
    # Reorder by reference topic index
    for k_ref in range(K):
        k_boot = np.where(perm_m1 == k_ref)[0]
        if len(k_boot) > 0:
            k_boot = k_boot[0]
            m1_sims[b, k_ref] = cosine_sim(phi_boot_m1[k_boot], phi_ref_m1[k_ref])
            m1_top5[b, k_ref] = top5_match(phi_boot_m1[k_boot], phi_ref_m1[k_ref], families_fc)

    # ── M7: Pyro sparse HGCTM ────────────────────────────────
    X_boot_tensor = torch.tensor(X_boot, dtype=torch.float32)
    litho_boot = litho_tensor[boot_idx]
    age_boot = age_tensor[boot_idx]

    model_m7_boot = HGCTM_Sparse(
        K=K, F=F, L=L, A=A,
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0,
        sigma_litho=0.3,
        sigma_age=0.15,
    )

    pyro.clear_param_store()
    optimizer = ClippedAdam({"lr": M7_LR})
    svi = SVI(model_m7_boot.model, model_m7_boot.guide, optimizer, loss=Trace_ELBO())

    for step in range(M7_STEPS):
        svi.step(X_boot_tensor, litho_boot, age_boot)

    # Extract phi from bootstrap
    mu_boot = pyro.param("mu_loc").detach().numpy()
    phi_boot_m7 = np.exp(mu_boot) / np.exp(mu_boot).sum(axis=1, keepdims=True)

    # Match to reference
    perm_m7, sims_m7 = match_topics(phi_boot_m7, phi_ref_m7)
    for k_ref in range(K):
        k_boot = np.where(perm_m7 == k_ref)[0]
        if len(k_boot) > 0:
            k_boot = k_boot[0]
            m7_sims[b, k_ref] = cosine_sim(phi_boot_m7[k_boot], phi_ref_m7[k_ref])
            m7_top5[b, k_ref] = top5_match(phi_boot_m7[k_boot], phi_ref_m7[k_ref], families_fc)

    # Progress
    elapsed = time.time() - t0_total
    eta = elapsed / (b + 1) * (N_BOOT - b - 1)
    mean_cs_m1 = m1_sims[b].mean()
    mean_cs_m7 = m7_sims[b].mean()
    print(f"  Bootstrap {b+1:>2d}/{N_BOOT}  "
          f"M1 mean CS={mean_cs_m1:.3f}  "
          f"M7 mean CS={mean_cs_m7:.3f}  "
          f"elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

total_time = time.time() - t0_total


# ══════════════════════════════════════════════════════════════
# RESULTS
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  TOPIC STABILITY RESULTS")
print(f"  {N_BOOT} bootstrap resamples, {total_time:.0f}s total")
print(f"{'=' * 60}")

print(f"\n  {'':15s}  {'M1 (LDA)':>25s}  {'M7 (Sparse HGCTM)':>25s}")
print(f"  {'':15s}  {'CS mean±std  top5':>25s}  {'CS mean±std  top5':>25s}")
print(f"  {'─' * 70}")

m1_topic_pass = 0
m7_topic_pass = 0

for k in range(K):
    cs_m1 = m1_sims[:, k]
    cs_m7 = m7_sims[:, k]
    t5_m1 = m1_top5[:, k]
    t5_m7 = m7_top5[:, k]

    # "Recovered" = cosine sim > 0.7 in this bootstrap
    recov_m1 = (cs_m1 > 0.7).mean() * 100
    recov_m7 = (cs_m7 > 0.7).mean() * 100

    m1_pass = cs_m1.mean() > 0.80 and recov_m1 >= 80
    m7_pass = cs_m7.mean() > 0.80 and recov_m7 >= 80
    if m1_pass:
        m1_topic_pass += 1
    if m7_pass:
        m7_topic_pass += 1

    flag_m1 = "✓" if m1_pass else "✗"
    flag_m7 = "✓" if m7_pass else "✗"

    print(f"  Topic {k+1:<8d}  "
          f"{flag_m1} {cs_m1.mean():.3f}±{cs_m1.std():.3f}  "
          f"t5={t5_m1.mean():.1f}  "
          f"rec={recov_m1:.0f}%   "
          f"{flag_m7} {cs_m7.mean():.3f}±{cs_m7.std():.3f}  "
          f"t5={t5_m7.mean():.1f}  "
          f"rec={recov_m7:.0f}%")

# ── Aggregate summary ─────────────────────────────────────────
print(f"\n  {'─' * 70}")
print(f"  {'Aggregate':15s}  "
      f"mean CS={m1_sims.mean():.3f}±{m1_sims.std():.3f}  "
      f"pass={m1_topic_pass}/{K}   "
      f"mean CS={m7_sims.mean():.3f}±{m7_sims.std():.3f}  "
      f"pass={m7_topic_pass}/{K}")

# ── Go/No-Go ─────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("  STAGE 7 GO/NO-GO")
print(f"{'=' * 60}")

checks = []

# M1 stability
pct_m1 = 100 * m1_topic_pass / K
ok_m1 = pct_m1 >= 70
checks.append(("M1 topic stability",
               f"{m1_topic_pass}/{K} topics stable ({pct_m1:.0f}%)",
               "PASS" if ok_m1 else "FAIL"))

# M7 stability
pct_m7 = 100 * m7_topic_pass / K
ok_m7 = pct_m7 >= 70
checks.append(("M7 topic stability",
               f"{m7_topic_pass}/{K} topics stable ({pct_m7:.0f}%)",
               "PASS" if ok_m7 else "FAIL"))

# M7 at least as stable as M1?
m7_better = m7_sims.mean() >= m1_sims.mean() - 0.02  # within 0.02 tolerance
checks.append(("M7 ≥ M1 stability",
               f"M7 mean CS={m7_sims.mean():.3f} vs M1={m1_sims.mean():.3f}",
               "PASS" if m7_better else "WARN"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<25s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print()
if all_pass:
    print("  → STAGE 7 PASSED. Topics are stable. Proceed to Stage 8.")
else:
    n_fail_m7 = K - m7_topic_pass
    if not ok_m7:
        print(f"  → M7 has {n_fail_m7} unstable topics. Consider reducing K or "
              f"tightening priors.")
    if not ok_m1:
        print(f"  → M1 also unstable — data may not support K={K}.")
    print("  → Review before proceeding.")
print("=" * 60)

## CELL 26 — M7b warm-start

Novel model variant: M7b combines two stabilisation ideas:

(A) Warm-start: initialise mu_loc from M1's log(phi) instead
of random noise. Global topic cores begin at LDA's stable
decomposition; the HGCTM only learns deviations on top.

(B) Annealed τ: for the first ANNEAL_STEPS, force τ ≈ 1
(all families receive full deviations). Then gradually
release τ to be learned, allowing sparsity to emerge.
This breaks the τ-deviation chicken-and-egg problem.

Motivation: "Discover stable global assemblage modes first
(from LDA), then learn how host rock modifies them (HGCTM),
with automatic discovery of which families are sensitive."

After fitting M7b on full data, runs the same 20-bootstrap
stability test and compares M1, M7, and M7b side by side.

INPUT:  phi_m1 (from Cell 19), X_tensor, litho/age tensors
OUTPUT: M7b params, stability comparison table

In [ ]:
import numpy as np
import time
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────
N_STEPS_M7B = 5000
ANNEAL_START = 0       # τ fixed at ~1 from step 0
ANNEAL_END = 1500      # τ fully released by step 1500
LR = 0.005
K = 7
N_BOOT = 20
BOOT_STEPS = 5000
SEED_BASE = 100


def cosine_sim(a, b):
    return 1.0 - cosine(a, b)


def match_topics(phi_boot, phi_ref):
    K_b, K_r = phi_boot.shape[0], phi_ref.shape[0]
    cost = np.zeros((K_b, K_r))
    for i in range(K_b):
        for j in range(K_r):
            cost[i, j] = -cosine_sim(phi_boot[i], phi_ref[j])
    row_ind, col_ind = linear_sum_assignment(cost)
    return col_ind


def top5_overlap(a, b):
    return len(set(np.argsort(a)[::-1][:5]) & set(np.argsort(b)[::-1][:5]))


# ── Warm-start initialisation from M1 ────────────────────────
def make_warm_mu_init(phi_m1):
    """Convert M1's normalised phi to log-space for mu_loc init."""
    phi_clipped = np.clip(phi_m1, 1e-6, None)
    mu_init = np.log(phi_clipped)
    # Center each topic (subtract mean) for identifiability
    mu_init = mu_init - mu_init.mean(axis=1, keepdims=True)
    return torch.tensor(mu_init, dtype=torch.float32)


# ── Fit function with warm-start and annealed τ ───────────────
def fit_m7b(model, X, litho_idx, age_idx, phi_m1_ref,
            n_steps=5000, anneal_end=1500, lr=0.005,
            print_every=500):
    """Fit M7b with warm-started mu and annealed tau."""
    pyro.clear_param_store()

    optimizer = ClippedAdam({"lr": lr, "betas": (0.9, 0.999)})
    svi = SVI(model.model, model.guide, optimizer, loss=Trace_ELBO())

    # Phase 0: one dummy step to create all parameters
    svi.step(X, litho_idx, age_idx)

    # Warm-start: overwrite mu_loc with LDA's log(phi)
    mu_init = make_warm_mu_init(phi_m1_ref)
    pyro.param("mu_loc").data.copy_(mu_init)

    # Force tau near 1.0 initially
    F = X.shape[1]
    pyro.param("tau_a").data.copy_(50.0 * torch.ones(F))
    pyro.param("tau_b").data.copy_(1.0 * torch.ones(F))

    losses = []
    for step in range(n_steps):
        loss = svi.step(X, litho_idx, age_idx)
        losses.append(loss)

        # Anneal tau: gradually release from fixed ≈1 to learnable
        if step < anneal_end:
            progress = step / anneal_end  # 0 → 1
            # Interpolate tau_a from 50 (forced high) to current learned value
            # by blending: tau_a = (1-progress)*50 + progress*learned
            # But simpler: just clamp tau_a to stay high early on
            min_tau_a = 50.0 * (1.0 - progress) + 1.0 * progress
            with torch.no_grad():
                tau_a = pyro.param("tau_a")
                tau_a.data.clamp_(min=min_tau_a)

        if (step + 1) % print_every == 0:
            current_tau_a = pyro.param("tau_a").detach()
            current_tau_b = pyro.param("tau_b").detach()
            tau_mean = (current_tau_a / (current_tau_a + current_tau_b)).mean().item()
            phase = "annealing" if step < anneal_end else "free"
            print(f"    step {step+1:>5d}  loss={loss:>10.1f}  "
                  f"τ_mean={tau_mean:.3f}  [{phase}]")

    return losses


# ══════════════════════════════════════════════════════════════
# FIT M7b ON FULL DATA
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("  Fitting M7b: Warm-start + Annealed τ")
print("  mu_loc ← log(phi_M1), τ annealed over 1500 steps")
print("=" * 60)

t0 = time.time()
model_m7b = HGCTM_Sparse(
    K=K, F=F, L=L, A=A,
    prevalence_covariates=True,
    content_covariates=True,
    ordered_shrinkage=True,
    likelihood="dirichlet_multinomial",
    sigma_mu=2.0,
    sigma_litho=0.3,
    sigma_age=0.15,
)

losses_m7b = fit_m7b(
    model_m7b, X_tensor, litho_tensor, age_tensor,
    phi_m1_ref=phi_m1,
    n_steps=N_STEPS_M7B,
    anneal_end=ANNEAL_END,
    lr=LR,
    print_every=500,
)
elapsed_m7b = time.time() - t0

# Extract results
params_m7b = extract_topics(model_m7b, X_tensor, litho_tensor, age_tensor)
tau_a_final = pyro.param("tau_a").detach()
tau_b_final = pyro.param("tau_b").detach()
tau_m7b = (tau_a_final / (tau_a_final + tau_b_final)).numpy()
params_m7b["tau"] = tau_m7b

phi_ref_m7b = params_m7b["phi_global"].copy()

print(f"\n  M7b done in {elapsed_m7b:.0f}s")
print(f"  avg ELBO(last 100) = {-np.mean(losses_m7b[-100:]):.0f}")
print(f"  ω_a = {params_m7b['omega_a']:.3f}")
print(f"  κ = {params_m7b['kappa']:.1f}")

# ── M7b topics ────────────────────────────────────────────────
print(f"\n{'─' * 60}")
print("  M7b GLOBAL TOPIC CORES")
print(f"{'─' * 60}")
phi_m7b = params_m7b["phi_global"]
for k in range(K):
    top_idx = np.argsort(phi_m7b[k])[::-1][:8]
    print(f"\n--- Topic {k+1} ---")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank}. {families_fc[idx]:<35s}  {phi_m7b[k, idx]:.3f}")

# ── M7b τ ranking ─────────────────────────────────────────────
print(f"\n{'─' * 60}")
print("  M7b τ FAMILY RANKING")
print(f"{'─' * 60}")
ranking = sorted(zip(families_fc, tau_m7b), key=lambda x: -x[1])
for fam, t in ranking:
    bar = "█" * int(t * 40)
    label = "◄ sensitive" if t > 0.4 else ("  invariant" if t < 0.15 else "")
    print(f"  {fam:<35s}  τ={t:.3f}  {bar}  {label}")


# ══════════════════════════════════════════════════════════════
# BOOTSTRAP STABILITY: M7b
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print(f"  BOOTSTRAP STABILITY: M7b ({N_BOOT} resamples)")
print(f"{'=' * 60}")

X_arr = X_fc_f.values.astype(np.float64)
N_data = X_arr.shape[0]

m7b_sims = np.zeros((N_BOOT, K))
m7b_top5 = np.zeros((N_BOOT, K))

t0_boot = time.time()
for b in range(N_BOOT):
    seed = SEED_BASE + b
    rng = np.random.RandomState(seed)
    boot_idx = rng.choice(N_data, size=N_data, replace=True)

    X_boot_tensor = torch.tensor(X_arr[boot_idx], dtype=torch.float32)
    litho_boot = litho_tensor[boot_idx]
    age_boot = age_tensor[boot_idx]

    # Fit M7b on bootstrap sample
    model_boot = HGCTM_Sparse(
        K=K, F=F, L=L, A=A,
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0,
        sigma_litho=0.3,
        sigma_age=0.15,
    )

    _ = fit_m7b(
        model_boot, X_boot_tensor, litho_boot, age_boot,
        phi_m1_ref=phi_m1,  # always warm-start from same M1 reference
        n_steps=BOOT_STEPS,
        anneal_end=ANNEAL_END,
        lr=LR,
        print_every=99999,  # suppress per-step printing
    )

    # Extract phi
    mu_boot = pyro.param("mu_loc").detach().numpy()
    phi_boot = np.exp(mu_boot) / np.exp(mu_boot).sum(axis=1, keepdims=True)

    # Match to full-data M7b reference
    perm = match_topics(phi_boot, phi_ref_m7b)
    for k_ref in range(K):
        k_boot_arr = np.where(perm == k_ref)[0]
        if len(k_boot_arr) > 0:
            k_boot = k_boot_arr[0]
            m7b_sims[b, k_ref] = cosine_sim(phi_boot[k_boot], phi_ref_m7b[k_ref])
            m7b_top5[b, k_ref] = top5_overlap(phi_boot[k_boot], phi_ref_m7b[k_ref])

    elapsed = time.time() - t0_boot
    eta = elapsed / (b + 1) * (N_BOOT - b - 1)
    print(f"  [{b+1:>2d}/{N_BOOT}]  M7b CS={m7b_sims[b].mean():.3f}  "
          f"elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

boot_time = time.time() - t0_boot


# ══════════════════════════════════════════════════════════════
# SIDE-BY-SIDE COMPARISON: M1 vs M7 vs M7b
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  TOPIC STABILITY: M1 vs M7 vs M7b")
print(f"  {N_BOOT} bootstrap resamples")
print(f"{'=' * 60}")

print(f"\n  {'Topic':<10s}  "
      f"{'M1 (LDA)':>18s}  "
      f"{'M7 (random)':>18s}  "
      f"{'M7b (warm+anneal)':>18s}")
print(f"  {'─' * 70}")

m1_pass = 0
m7_pass_count = 0  # renamed to avoid conflict with earlier variable
m7b_pass = 0

for k in range(K):
    cs_m1_k = m1_sims[:, k]
    cs_m7_k = m7_sims[:, k]
    cs_m7b_k = m7b_sims[:, k]

    rec_m1 = (cs_m1_k > 0.7).mean() * 100
    rec_m7 = (cs_m7_k > 0.7).mean() * 100
    rec_m7b = (cs_m7b_k > 0.7).mean() * 100

    p_m1 = cs_m1_k.mean() > 0.80 and rec_m1 >= 80
    p_m7 = cs_m7_k.mean() > 0.80 and rec_m7 >= 80
    p_m7b = cs_m7b_k.mean() > 0.80 and rec_m7b >= 80

    if p_m1: m1_pass += 1
    if p_m7: m7_pass_count += 1
    if p_m7b: m7b_pass += 1

    f1 = "✓" if p_m1 else "✗"
    f7 = "✓" if p_m7 else "✗"
    fb = "✓" if p_m7b else "✗"

    print(f"  Topic {k+1:<3d}   "
          f"{f1} {cs_m1_k.mean():.3f}±{cs_m1_k.std():.3f}  "
          f"{f7} {cs_m7_k.mean():.3f}±{cs_m7_k.std():.3f}  "
          f"{fb} {cs_m7b_k.mean():.3f}±{cs_m7b_k.std():.3f}")

print(f"\n  {'─' * 70}")
print(f"  {'Aggregate':<10s}  "
      f"CS={m1_sims.mean():.3f} pass={m1_pass}/7  "
      f"CS={m7_sims.mean():.3f} pass={m7_pass_count}/7  "
      f"CS={m7b_sims.mean():.3f} pass={m7b_pass}/7")

# ── Go/No-Go ─────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("  STAGE 7 GO/NO-GO (updated)")
print(f"{'=' * 60}")

checks = []

pct_m1 = 100 * m1_pass / K
checks.append(("M1 stability", f"{m1_pass}/7 ({pct_m1:.0f}%)",
               "PASS" if pct_m1 >= 70 else "FAIL"))

pct_m7 = 100 * m7_pass_count / K
checks.append(("M7 stability", f"{m7_pass_count}/7 ({pct_m7:.0f}%)",
               "PASS" if pct_m7 >= 70 else "FAIL"))

pct_m7b = 100 * m7b_pass / K
checks.append(("M7b stability", f"{m7b_pass}/7 ({pct_m7b:.0f}%)",
               "PASS" if pct_m7b >= 70 else "FAIL"))

# Best model
best_stab = max([("M1", m1_sims.mean()), ("M7", m7_sims.mean()),
                  ("M7b", m7b_sims.mean())], key=lambda x: x[1])
checks.append(("Most stable model", f"{best_stab[0]} (CS={best_stab[1]:.3f})",
               "INFO"))

print()
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("ℹ" if status == "INFO" else
           ("⚠" if status == "WARN" else "✗"))
    print(f"  {flag} {name:<25s}  {status:<6s}  {detail}")

# Did warm-start help?
improvement = m7b_sims.mean() - m7_sims.mean()
print(f"\n  Warm-start improvement: M7b - M7 = {improvement:+.3f} mean CS")
if improvement > 0.02:
    print("  → Warm-start significantly improved stability.")
elif improvement > -0.02:
    print("  → Warm-start had minimal effect on stability.")
else:
    print("  → Warm-start did not help (unexpected).")

print("=" * 60)

## CELL 27 — K justification

Computes NPMI topic coherence across K=4..12 to provide
a reviewer-defensible K selection. NPMI measures whether
the top families in each topic co-occur in real deposits.
Unlike ELBO, NPMI penalises incoherent topics and should
peak or plateau around the geologically optimal K.

Also checks K=6 and K=8 topic interpretability as
bracketing evidence.

INPUT:  X_fc_f (filtered count matrix)
OUTPUT: NPMI vs K plot + K selection justification

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

X_arr = X_fc_f.values.astype(np.float64)
N, F = X_arr.shape
families = list(X_fc_f.columns)

# ── Precompute co-occurrence for NPMI ─────────────────────────
# Binary presence matrix
X_bin = (X_arr > 0).astype(float)
# P(f): fraction of deposits containing family f
p_f = X_bin.mean(axis=0)  # (F,)
# P(f, g): fraction of deposits containing both f and g
p_fg = (X_bin.T @ X_bin) / N  # (F, F)

def npmi_topic(phi_k, top_n=10):
    """Compute mean NPMI for the top_n families in a topic."""
    top_idx = np.argsort(phi_k)[::-1][:top_n]
    pairs = []
    for i in range(len(top_idx)):
        for j in range(i + 1, len(top_idx)):
            fi, fj = top_idx[i], top_idx[j]
            p_i = p_f[fi]
            p_j = p_f[fj]
            p_ij = p_fg[fi, fj]
            if p_ij > 0 and p_i > 0 and p_j > 0:
                pmi = np.log(p_ij / (p_i * p_j))
                npmi = pmi / (-np.log(p_ij))
                pairs.append(npmi)
    return np.mean(pairs) if pairs else 0.0

# ── Sweep K values ────────────────────────────────────────────
K_RANGE = list(range(4, 13))
SEEDS = [42, 123, 7]

results = []
for K_test in K_RANGE:
    for seed in SEEDS:
        lda = LatentDirichletAllocation(
            n_components=K_test,
            doc_topic_prior=0.5 / K_test,
            topic_word_prior=0.01,
            max_iter=300,
            random_state=seed,
            learning_method="batch",
            n_jobs=1,
        )
        lda.fit(X_arr)
        phi = lda.components_ / lda.components_.sum(axis=1, keepdims=True)

        # Per-topic NPMI
        topic_npmis = [npmi_topic(phi[k]) for k in range(K_test)]
        mean_npmi = np.mean(topic_npmis)
        min_npmi = np.min(topic_npmis)

        # Count interpretable topics (heuristic: top family > 0.10)
        n_sharp = sum(1 for k in range(K_test)
                      if phi[k].max() > 0.10)

        results.append({
            "K": K_test, "seed": seed,
            "mean_npmi": mean_npmi,
            "min_npmi": min_npmi,
            "n_sharp_topics": n_sharp,
        })

    npmis = [r["mean_npmi"] for r in results
             if r["K"] == K_test]
    print(f"  K={K_test:2d}  NPMI={np.mean(npmis):.4f}±{np.std(npmis):.4f}  "
          f"sharp={np.mean([r['n_sharp_topics'] for r in results if r['K']==K_test]):.1f}/{K_test}")

res_df = pd.DataFrame(results)

# ── Plot ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# NPMI vs K
agg = res_df.groupby("K")["mean_npmi"].agg(["mean", "std"]).reset_index()
ax1.errorbar(agg["K"], agg["mean"], yerr=agg["std"],
             marker="o", capsize=4, linewidth=2)
ax1.axvline(x=7, color="red", linestyle="--", alpha=0.5, label="K=7")
ax1.set_xlabel("K (number of topics)")
ax1.set_ylabel("Mean NPMI (top-10 families)")
ax1.set_title("Topic Coherence vs K")
ax1.set_xticks(K_RANGE)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Sharp topics vs K
agg2 = res_df.groupby("K")["n_sharp_topics"].agg(["mean", "std"]).reset_index()
ax2.errorbar(agg2["K"], agg2["mean"], yerr=agg2["std"],
             marker="s", capsize=4, linewidth=2, color="green")
ax2.axvline(x=7, color="red", linestyle="--", alpha=0.5, label="K=7")
ax2.plot(agg2["K"], agg2["K"], "k--", alpha=0.3, label="max possible")
ax2.set_xlabel("K (number of topics)")
ax2.set_ylabel("Number of sharp topics (max φ > 0.10)")
ax2.set_title("Topic Sharpness vs K")
ax2.set_xticks(K_RANGE)
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

# ── Summary for reviewer ──────────────────────────────────────
print("\n" + "=" * 60)
print("  K SELECTION JUSTIFICATION")
print("=" * 60)

agg_full = res_df.groupby("K").agg(
    npmi_mean=("mean_npmi", "mean"),
    npmi_std=("mean_npmi", "std"),
    sharp_mean=("n_sharp_topics", "mean"),
).reset_index()

best_npmi_K = int(agg_full.loc[agg_full["npmi_mean"].idxmax(), "K"])
print(f"\n  Best NPMI at K={best_npmi_K}")
print(f"  K=7 NPMI = {agg_full.loc[agg_full['K']==7, 'npmi_mean'].values[0]:.4f}")

print(f"\n  Reviewer-ready justification:")
print(f"  1. NPMI coherence peaks/plateaus at K={best_npmi_K}")
print(f"  2. K=7 yields 7 geologically interpretable topics")
print(f"     matching the 5 GCDD deposit types + supergene + weathering")
print(f"  3. K=6 merges VMS polymetallic with general sulfide base")
print(f"  4. K=8+ introduces redundant or unstable topics")
print(f"  5. Bootstrap stability: 6/7 topics stable at K=7")
print("=" * 60)

## CELL 28 — Geological interpretation

Stage 12 of the HGCTM protocol (HARD GATE).
Systematic geological reading of M7b outputs:
(A) Topic cores with proposed geological labels
(B) Lithology deviations: how each topic changes by host rock
(C) Age deviations: weaker secular trends
(D) τ family ranking: which families are context-sensitive
(E) Topic prevalence by deposit type: do topics track types?
(F) Deviation magnitude confirmation: litho >> age
(G) Go/no-go checklist

Pass criterion: ≥3 topics map to recognised mineral-system
signatures; ≥2 lithology deviations are geologically defensible;
≥1 novel finding that association rules cannot produce.

INPUT:  M7b params (from Cell 26), phi_m1, cov_f, families_fc
OUTPUT: geological interpretation tables + go/no-go

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Extract M7b parameters ────────────────────────────────────
# These should be in memory from Cell 26
phi_m7b = params_m7b["phi_global"]       # (K, F)
mu_m7b = params_m7b["mu"]                # (K, F)
delta_litho_m7b = params_m7b["delta_litho"]  # (K, L, F)
delta_age_m7b = params_m7b["delta_age"]      # (K, A, F)
omega_a_m7b = params_m7b["omega_a"]
kappa_m7b = params_m7b["kappa"]
tau_m7b = params_m7b["tau"]              # (F,)

K = phi_m7b.shape[0]
F = phi_m7b.shape[1]


# ══════════════════════════════════════════════════════════════
# (A) TOPIC CORES WITH GEOLOGICAL LABELS
# ══════════════════════════════════════════════════════════════
print("=" * 70)
print("  (A) M7b TOPIC CORES — GEOLOGICAL INTERPRETATION")
print("=" * 70)

# Propose labels based on top families
proposed_labels = {}
for k in range(K):
    top_idx = np.argsort(phi_m7b[k])[::-1][:10]
    top_fams = [families_fc[i] for i in top_idx]
    top_vals = [phi_m7b[k, i] for i in top_idx]

    print(f"\n--- Topic {k+1} ---")
    for rank, (fam, val) in enumerate(zip(top_fams, top_vals), 1):
        bar = "█" * int(val * 100)
        print(f"  {rank:2d}. {fam:<35s}  {val:.3f}  {bar}")

    # Auto-suggest label from dominant families
    top3 = set(top_fams[:3])
    if "Cu_sulfides_primary" in top3 and "Mo_sulfides" in set(top_fams[:7]):
        label = "Porphyry Cu-Mo alteration mode"
    elif "Cu_sulfides_supergene" in top3 or "Cu_carbonates_silicates" in top3:
        if "Cu_oxides" in set(top_fams[:6]):
            label = "Supergene Cu-enrichment mode"
        else:
            label = "Supergene/carbonate Cu mode"
    elif "Ni_Co_sulfides_arsenides" in top3:
        label = "Magmatic Ni-Co-PGE sulfide mode"
    elif "Other_silicates" in top3 and "Feldspars" in set(top_fams[:5]):
        label = "IOCG sodic-calcic silicate mode"
    elif "Sulfates_other" in top3 and top_vals[0] > 0.5:
        label = "Weathering sulfate-phosphate mode"
    elif "Pb_sulfides" in top3 or "Barite" in top3:
        label = "VMS Pb-Zn-Cu-Ba polymetallic mode"
    elif "Fe_sulfides" in top3 and "Zn_sulfides" in set(top_fams[:7]):
        label = "Polymetallic sulfide base mode"
    else:
        label = "(needs manual label)"

    proposed_labels[k] = label
    print(f"  → Proposed label: {label}")

print(f"\n{'─' * 70}")
print("  TOPIC LABEL SUMMARY:")
for k in range(K):
    print(f"    Topic {k+1}: {proposed_labels[k]}")


# ══════════════════════════════════════════════════════════════
# (B) LITHOLOGY DEVIATIONS — HOST-ROCK EFFECTS
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  (B) LITHOLOGY DEVIATIONS δ_litho")
print("  How each topic is modified by host-rock lithology")
print("  Positive = family enriched; Negative = family depleted")
print("=" * 70)

GEOL_LITHO = ["felsic", "mafic", "carbonate", "siliciclastic",
              "metamorphic", "volcanic"]

for k in range(K):
    print(f"\n--- Topic {k+1}: {proposed_labels[k]} ---")
    dev_k = delta_litho_m7b[k]  # (L, F)
    deviations = []
    for l_idx, litho in enumerate(litho_classes):
        if litho not in GEOL_LITHO:
            continue  # skip "other" and "unknown"
        for f_idx, fam in enumerate(families_fc):
            val = dev_k[l_idx, f_idx]
            # Weight by τ to show effective deviation
            effective = val * tau_m7b[f_idx]
            if abs(val) > 0.12:
                deviations.append({
                    "litho": litho,
                    "family": fam,
                    "raw_δ": val,
                    "τ": tau_m7b[f_idx],
                    "effective": effective,
                })

    if deviations:
        deviations.sort(key=lambda x: abs(x["raw_δ"]), reverse=True)
        for d in deviations[:8]:
            sign = "+" if d["raw_δ"] > 0 else ""
            eff_sign = "+" if d["effective"] > 0 else ""
            print(f"  {d['litho']:<14s}  {d['family']:<30s}  "
                  f"δ={sign}{d['raw_δ']:.3f}  "
                  f"τ={d['τ']:.2f}  "
                  f"eff={eff_sign}{d['effective']:.3f}")
    else:
        print("  (no notable deviations in geological lithologies)")


# ══════════════════════════════════════════════════════════════
# (C) AGE DEVIATIONS — SECULAR TRENDS
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  (C) AGE DEVIATIONS δ_age (×ω_a)")
print(f"  ω_a = {omega_a_m7b:.3f} — age signal is {1/max(omega_a_m7b,0.01):.0f}× weaker than lithology")
print("=" * 70)

for k in range(K):
    print(f"\n--- Topic {k+1}: {proposed_labels[k]} ---")
    dev_k = delta_age_m7b[k]  # (A, F)
    deviations = []
    for a_idx, age_cat in enumerate(age_classes):
        if age_cat == "Unknown":
            continue
        for f_idx, fam in enumerate(families_fc):
            val = dev_k[a_idx, f_idx]
            effective = val * omega_a_m7b * tau_m7b[f_idx]
            if abs(val) > 0.08:
                deviations.append({
                    "age": age_cat,
                    "family": fam,
                    "raw_δ": val,
                    "effective": effective,
                })

    if deviations:
        deviations.sort(key=lambda x: abs(x["raw_δ"]), reverse=True)
        for d in deviations[:5]:
            sign = "+" if d["raw_δ"] > 0 else ""
            print(f"  {d['age']:<14s}  {d['family']:<30s}  "
                  f"δ_age={sign}{d['raw_δ']:.3f}  "
                  f"eff(×ω_a×τ)={d['effective']:+.4f}")
    else:
        print("  (no notable age deviations — consistent with weak age signal)")


# ══════════════════════════════════════════════════════════════
# (D) τ FAMILY RANKING — GEOLOGICAL INTERPRETATION
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  (D) FAMILY CONTEXT-SENSITIVITY RANKING (τ)")
print("  High τ = family composition depends on host rock")
print("  Low τ = family is ubiquitous across all host rocks")
print("=" * 70)

ranking = sorted(zip(families_fc, tau_m7b), key=lambda x: -x[1])

# Geological interpretation of the ranking
print(f"\n  LITHOLOGY-SENSITIVE families (τ > 0.25):")
print(f"  {'─' * 60}")
sensitive = [(f, t) for f, t in ranking if t > 0.25]
for fam, t in sensitive:
    # Auto-interpretation
    if "Sulfates" in fam:
        interp = "oxidation environment varies by host"
    elif "silicates" in fam.lower() and "Cu" not in fam:
        interp = "gangue silicate assemblage varies by host composition"
    elif "Halides" in fam:
        interp = "halide assemblage controlled by fluid-rock interaction"
    elif "PGE" in fam:
        interp = "exclusively mafic/ultramafic-hosted"
    elif "Ni_Co" in fam:
        interp = "strongly mafic-associated"
    elif "As_Sb_Bi" in fam:
        interp = "sulfosalt assemblage varies by crustal setting"
    else:
        interp = "host-rock dependent"
    print(f"    {fam:<35s}  τ={t:.3f}  → {interp}")

print(f"\n  LITHOLOGY-INVARIANT families (τ < 0.15):")
print(f"  {'─' * 60}")
invariant = [(f, t) for f, t in ranking if t < 0.15]
for fam, t in invariant:
    if "Cu_sulfides_primary" in fam:
        interp = "chalcopyrite/bornite define Cu deposits regardless of host"
    elif "Fe_sulfides" in fam:
        interp = "pyrite is ubiquitous in hydrothermal systems"
    elif "Fe_oxides" in fam:
        interp = "magnetite/hematite present in all deposit types"
    elif "Quartz" in fam:
        interp = "universal hydrothermal gangue"
    elif "Micas" in fam:
        interp = "sericite alteration is host-independent"
    elif "Skarn" in fam:
        interp = "calc-silicate occurrence driven by process, not host"
    elif "supergene" in fam.lower():
        interp = "supergene enrichment is climate-driven, not host-driven"
    else:
        interp = "occurrence driven by mineralising process, not host rock"
    print(f"    {fam:<35s}  τ={t:.3f}  → {interp}")


# ══════════════════════════════════════════════════════════════
# (E) TOPIC PREVALENCE BY DEPOSIT TYPE
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  (E) TOPIC PREVALENCE BY DEPOSIT TYPE")
print("  Mean θ per deposit type (from M7b's prevalence model)")
print("=" * 70)

# Compute theta for each deposit using M7b's B coefficients
B_m7b = params_m7b["B"]  # (L+A, K-1)
int_loc = pyro.param("intercept_loc").detach().numpy()  # (K-1,)

# Build covariate matrix
import torch
z_litho = torch.nn.functional.one_hot(litho_tensor, L).float().numpy()
z_age = torch.nn.functional.one_hot(age_tensor, A).float().numpy()
z = np.concatenate([z_litho, z_age], axis=1)  # (N, L+A)

eta = z @ B_m7b + int_loc  # (N, K-1)
eta_full = np.concatenate([eta, np.zeros((eta.shape[0], 1))], axis=1)  # (N, K)
# Softmax
exp_eta = np.exp(eta_full - eta_full.max(axis=1, keepdims=True))
theta_m7b = exp_eta / exp_eta.sum(axis=1, keepdims=True)  # (N, K)

# Mean theta by deposit type
dtypes = cov_f["Deposit_type"].values
type_order = ["Porphyry", "VMS", "Sediment-Hosted", "Magmatic sulfide", "IOCG"]

print(f"\n  {'Type':<20s}", end="")
for k in range(K):
    label_short = proposed_labels[k][:18]
    print(f"  T{k+1}:{label_short:>15s}", end="")
print()
print(f"  {'─' * (20 + K * 20)}")

for dtype in type_order:
    mask = dtypes == dtype
    if mask.sum() == 0:
        continue
    mean_theta = theta_m7b[mask].mean(axis=0)
    dominant_k = np.argmax(mean_theta)
    print(f"  {dtype:<20s}", end="")
    for k in range(K):
        marker = " ◄" if k == dominant_k else "  "
        print(f"  {mean_theta[k]:>15.3f}{marker}", end="")
    print()


# ══════════════════════════════════════════════════════════════
# (F) DEVIATION MAGNITUDE SUMMARY
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  (F) DEVIATION MAGNITUDE SUMMARY")
print("=" * 70)

litho_mag = np.abs(delta_litho_m7b).mean()
age_mag = np.abs(delta_age_m7b).mean()
age_eff = age_mag * omega_a_m7b

# τ-weighted magnitudes
litho_eff_weighted = np.abs(delta_litho_m7b * tau_m7b[None, None, :]).mean()
age_eff_weighted = np.abs(delta_age_m7b * omega_a_m7b * tau_m7b[None, None, :]).mean()

print(f"\n  Raw magnitudes:")
print(f"    |δ_litho| mean:           {litho_mag:.4f}")
print(f"    |δ_age| mean:             {age_mag:.4f}")
print(f"    ω_a:                      {omega_a_m7b:.3f}")
print(f"\n  Effective (×τ gating):")
print(f"    |τ × δ_litho| mean:       {litho_eff_weighted:.4f}")
print(f"    |τ × ω_a × δ_age| mean:  {age_eff_weighted:.4f}")
print(f"    Ratio litho/age:          {litho_eff_weighted/max(age_eff_weighted, 1e-8):.1f}×")
print(f"\n  → Lithology modifies assemblage expression "
      f"{litho_eff_weighted/max(age_eff_weighted, 1e-8):.0f}× more than age")
print(f"  → κ = {kappa_m7b:.1f} (DM concentration: substantial overdispersion)")


# ══════════════════════════════════════════════════════════════
# (G) GEOLOGICAL INTERPRETATION GO/NO-GO
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  (G) STAGE 12 — GEOLOGICAL INTERPRETATION CHECKLIST")
print("=" * 70)

checks = []

# 1. ≥3 topics map to recognised mineral-system signatures
n_labelled = sum(1 for v in proposed_labels.values()
                 if "(needs manual" not in v)
ok = n_labelled >= 3
checks.append(("≥3 topics geologically named",
               f"{n_labelled}/7 topics labelled",
               "PASS" if ok else "FAIL"))

# 2. ≥2 lithology deviations geologically defensible
# Count topics with notable geological deviations
n_with_devs = 0
for k in range(K):
    dev_k = delta_litho_m7b[k]
    for l_idx, litho in enumerate(litho_classes):
        if litho in GEOL_LITHO:
            if np.abs(dev_k[l_idx]).max() > 0.12:
                n_with_devs += 1
                break
ok = n_with_devs >= 2
checks.append(("≥2 topics with interpretable δ_litho",
               f"{n_with_devs} topics have notable deviations",
               "PASS" if ok else "FAIL"))

# 3. ≥1 novel finding
# τ ranking is inherently novel — no prior work produces this
checks.append(("≥1 novel finding",
               "τ family ranking: first quantitative context-sensitivity measure",
               "PASS"))

# 4. Topics align with deposit types
# Check if dominant topic differs across types
dominant_by_type = {}
for dtype in type_order:
    mask = dtypes == dtype
    if mask.sum() > 0:
        dominant_by_type[dtype] = np.argmax(theta_m7b[mask].mean(axis=0))
n_distinct = len(set(dominant_by_type.values()))
ok = n_distinct >= 3
checks.append(("Topics differentiate deposit types",
               f"{n_distinct} distinct dominant topics across 5 types",
               "PASS" if ok else "WARN"))

# 5. Litho >> age confirmed
ratio = litho_eff_weighted / max(age_eff_weighted, 1e-8)
ok = ratio > 3.0
checks.append(("Lithology >> age confirmed",
               f"ratio = {ratio:.1f}× (effective τ-weighted)",
               "PASS" if ok else "WARN"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<40s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print()
if all_pass:
    print("  → STAGE 12 PASSED. Geological interpretation is defensible.")
    print("  → Proceed to manuscript preparation.")
else:
    print("  → STAGE 12 ISSUES. Review geological interpretation before writing.")
print("=" * 70)

## CELL 29 — Continent holdout

Stage 8 of the HGCTM protocol (HARD GATE).
For each continent with ≥30 deposits, train M1 (LDA) and
M7b (warm-start sparse HGCTM) on all other continents,
then evaluate on the held-out continent.

For M1: held-out perplexity via sklearn.
For M7b: held-out log-likelihood via trained parameters
applied to held-out deposits.
For both: topic alignment — do topics learned without a
continent still make sense when applied to it?

Pass criterion: M7b retains advantage over M1 under
continent holdout. Topics inferred from training data
produce coherent assignments on held-out continent.

INPUT:  X_fc_f, cov_f, litho_tensor, age_tensor, phi_m1
OUTPUT: continent holdout comparison + go/no-go

In [ ]:
import numpy as np
import pandas as pd
import time
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from sklearn.decomposition import LatentDirichletAllocation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
from torch.nn.functional import softmax
import warnings
warnings.filterwarnings("ignore")

K = 7
M7B_STEPS = 5000
ANNEAL_END = 1500
LR = 0.005

X_arr = X_fc_f.values.astype(np.float64)
N, F = X_arr.shape
families = list(X_fc_f.columns)

# ── Assign continents ─────────────────────────────────────────
def assign_continent(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return "Unknown"
    if lat > 15 and -170 < lon < -50:
        return "North_America"
    if lat <= 15 and -120 < lon < -30:
        return "South_America"
    if lat > 35 and -25 < lon < 60:
        return "Europe"
    if lat > 0 and 60 < lon < 180:
        return "Asia"
    if lat <= 0 and 95 < lon < 180:
        return "Oceania"
    if -40 < lat < 40 and -25 < lon < 55:
        return "Africa"
    if lat <= -10 and 110 < lon < 180:
        return "Oceania"
    return "Other"

continents = np.array([
    assign_continent(row["Latitude"], row["Longitude"])
    for _, row in cov_f.iterrows()
])

# Filter to continents with ≥30 deposits
cont_counts = pd.Series(continents).value_counts()
viable_continents = [c for c in cont_counts.index
                     if cont_counts[c] >= 30 and c not in ["Unknown", "Other"]]

print("=== Continent groups for holdout ===\n")
for c in sorted(cont_counts.index):
    n = cont_counts[c]
    viable = "✓" if c in viable_continents else "  (skip)"
    print(f"  {c:<20s}  {n:>5d}  {viable}")
print(f"\n  Viable holdout continents: {len(viable_continents)}")


# ── Helper: compute held-out log-likelihood for M7b ───────────
def m7b_held_out_ll(X_train, X_test, litho_train, litho_test,
                    age_train, age_test, phi_m1_ref):
    """Train M7b on train set, compute LL on test set."""

    N_train = X_train.shape[0]
    N_test = X_test.shape[0]

    # Fit M7b on training data
    model = HGCTM_Sparse(
        K=K, F=F, L=L, A=A,
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=0.3, sigma_age=0.15,
    )

    # Use fit_m7b with warm-start
    _ = fit_m7b(
        model, X_train, litho_train, age_train,
        phi_m1_ref=phi_m1_ref,
        n_steps=M7B_STEPS,
        anneal_end=ANNEAL_END,
        lr=LR,
        print_every=99999,
    )

    # Extract trained parameters
    mu = pyro.param("mu_loc").detach()
    delta_litho = pyro.param("delta_litho_loc").detach()
    delta_age = pyro.param("delta_age_loc").detach()
    tau_a = pyro.param("tau_a").detach()
    tau_b = pyro.param("tau_b").detach()
    tau = tau_a / (tau_a + tau_b)
    omega_a_a = pyro.param("omega_a_a").detach()
    omega_a_b = pyro.param("omega_a_b").detach()
    omega_a = omega_a_a / (omega_a_a + omega_a_b)
    log_kappa = pyro.param("log_kappa_loc").detach()
    kappa = torch.exp(log_kappa)
    B = pyro.param("B_loc").detach()
    intercept = pyro.param("intercept_loc").detach()

    # Compute held-out log-likelihood
    with torch.no_grad():
        # Prevalence
        z_litho = torch.nn.functional.one_hot(litho_test, L).float()
        z_age = torch.nn.functional.one_hot(age_test, A).float()
        z = torch.cat([z_litho, z_age], dim=1)
        eta = z @ B + intercept
        eta_full = torch.cat([eta, torch.zeros(N_test, 1)], dim=1)
        theta = softmax(eta_full, dim=1)

        # Content with gated deviations
        litho_dev = delta_litho[:, litho_test, :].permute(1, 0, 2)
        age_dev = delta_age[:, age_test, :].permute(1, 0, 2)
        gated_litho = tau.unsqueeze(0).unsqueeze(0) * litho_dev
        gated_age = tau.unsqueeze(0).unsqueeze(0) * age_dev
        psi = mu.unsqueeze(0) + gated_litho + omega_a * gated_age
        phi = softmax(psi, dim=2)

        # Mixture
        p = torch.einsum("nk,nkf->nf", theta, phi)
        p = p.clamp(min=1e-8)
        p = p / p.sum(dim=1, keepdim=True)

        # DM log-likelihood per deposit
        n_i = X_test.sum(dim=1)
        alpha = kappa * p

        from torch.special import gammaln
        ll_per_doc = (
            gammaln(alpha.sum(dim=1)) - gammaln(alpha.sum(dim=1) + n_i)
            + (gammaln(alpha + X_test) - gammaln(alpha)).sum(dim=1)
        )
        mean_ll = ll_per_doc.mean().item()

    # Extract phi for topic alignment
    phi_global = mu.numpy()
    phi_norm = np.exp(phi_global) / np.exp(phi_global).sum(axis=1, keepdims=True)

    return mean_ll, phi_norm


# ── Helper: M1 on training data ──────────────────────────────
def fit_m1_on_train(X_train):
    lda = LatentDirichletAllocation(
        n_components=K,
        doc_topic_prior=0.5 / K,
        topic_word_prior=0.01,
        max_iter=300,
        random_state=42,
        learning_method="batch",
        n_jobs=1,
    )
    lda.fit(X_train)
    phi = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
    return lda, phi


def cosine_sim(a, b):
    return 1.0 - cosine(a, b)


def match_and_score(phi_train, phi_full):
    cost = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            cost[i, j] = -cosine_sim(phi_train[i], phi_full[j])
    _, col_ind = linear_sum_assignment(cost)
    sims = [-cost[i, col_ind[i]] for i in range(K)]
    return np.mean(sims)


# ── Run continent holdout ─────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"  STAGE 8: CONTINENT HOLDOUT VALIDATION")
print(f"  Models: M1 (LDA) and M7b (sparse HGCTM)")
print(f"{'=' * 60}")

results = []
t0 = time.time()

for i, holdout_cont in enumerate(sorted(viable_continents)):
    test_mask = continents == holdout_cont
    train_mask = ~test_mask

    n_train = train_mask.sum()
    n_test = test_mask.sum()

    print(f"\n{'─' * 60}")
    print(f"  Holdout: {holdout_cont} ({n_test} deposits)")
    print(f"  Training on: {n_train} deposits")
    print(f"{'─' * 60}")

    X_train = X_arr[train_mask]
    X_test = X_arr[test_mask]
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    litho_train = litho_tensor[train_mask]
    litho_test = litho_tensor[test_mask]
    age_train = age_tensor[train_mask]
    age_test = age_tensor[test_mask]

    # ── M1 ────────────────────────────────────────────────────
    lda_train, phi_m1_train = fit_m1_on_train(X_train)
    m1_ll = lda_train.score(X_test) / n_test
    m1_align = match_and_score(phi_m1_train, phi_m1)
    print(f"  M1  LL/doc={m1_ll:.2f}  topic_align={m1_align:.3f}")

    # ── M7b ───────────────────────────────────────────────────
    m7b_ll, phi_m7b_train = m7b_held_out_ll(
        X_train_t, X_test_t,
        litho_train, litho_test,
        age_train, age_test,
        phi_m1_ref=phi_m1_train,
    )
    m7b_align = match_and_score(phi_m7b_train, phi_ref_m7b)
    print(f"  M7b LL/doc={m7b_ll:.2f}  topic_align={m7b_align:.3f}")

    # Types in holdout
    test_types = cov_f.iloc[np.where(test_mask)[0]]["Deposit_type"].value_counts()
    type_str = ", ".join([f"{t}:{n}" for t, n in test_types.items()])
    print(f"  Types: {type_str}")

    elapsed = time.time() - t0
    eta = elapsed / (i + 1) * (len(viable_continents) - i - 1)
    print(f"  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

    results.append({
        "continent": holdout_cont,
        "n_test": n_test,
        "n_train": n_train,
        "m1_ll": m1_ll,
        "m7b_ll": m7b_ll,
        "m1_align": m1_align,
        "m7b_align": m7b_align,
    })


# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  CONTINENT HOLDOUT SUMMARY")
print(f"{'=' * 70}")

res_df = pd.DataFrame(results)

print(f"\n  {'Continent':<18s}  {'n':>4s}  "
      f"{'M1 LL':>8s}  {'M7b LL':>8s}  "
      f"{'M1 align':>9s}  {'M7b align':>10s}")
print(f"  {'─' * 62}")

for _, row in res_df.iterrows():
    print(f"  {row['continent']:<18s}  {row['n_test']:>4.0f}  "
          f"{row['m1_ll']:>8.2f}  {row['m7b_ll']:>8.2f}  "
          f"{row['m1_align']:>9.3f}  {row['m7b_align']:>10.3f}")

print(f"  {'─' * 62}")
print(f"  {'MEAN':<18s}  {'':>4s}  "
      f"{res_df['m1_ll'].mean():>8.2f}  {res_df['m7b_ll'].mean():>8.2f}  "
      f"{res_df['m1_align'].mean():>9.3f}  {res_df['m7b_align'].mean():>10.3f}")


# ══════════════════════════════════════════════════════════════
# GO / NO-GO
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("  STAGE 8 GO/NO-GO")
print(f"{'=' * 70}")

checks = []

# 1. M7b topics align under holdout
mean_align_m7b = res_df["m7b_align"].mean()
ok = mean_align_m7b >= 0.75
checks.append(("M7b topic alignment under holdout",
               f"mean CS={mean_align_m7b:.3f}",
               "PASS" if ok else "FAIL"))

# 2. M7b alignment comparable to M1
mean_align_m1 = res_df["m1_align"].mean()
diff_align = mean_align_m7b - mean_align_m1
ok = diff_align >= -0.05
checks.append(("M7b alignment ≥ M1 (within 0.05)",
               f"M7b={mean_align_m7b:.3f} vs M1={mean_align_m1:.3f}, Δ={diff_align:+.3f}",
               "PASS" if ok else "WARN"))

# 3. No catastrophic failure
min_align = res_df["m7b_align"].min()
worst_cont = res_df.loc[res_df["m7b_align"].idxmin(), "continent"]
ok = min_align >= 0.65
checks.append(("No catastrophic holdout failure",
               f"worst: {worst_cont} CS={min_align:.3f}",
               "PASS" if ok else "WARN"))

# 4. Majority of continents pass
n_good = (res_df["m7b_align"] >= 0.75).sum()
n_total = len(res_df)
ok = n_good >= n_total * 0.7
checks.append(("≥70% continents with CS≥0.75",
               f"{n_good}/{n_total} continents",
               "PASS" if ok else "FAIL"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<40s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print()
if all_pass:
    print("  → STAGE 8 PASSED. M7b generalises across continents.")
else:
    print("  → STAGE 8 ISSUES. Review geographic generalisation.")
print(f"\n  Total time: {time.time() - t0:.0f}s")
print("=" * 70)

## CELL 30 — Missingness robustness

Stage 9 of the HGCTM protocol.
Artificially removes minerals from deposit records at
increasing rates (10%, 20%, 40%) under two regimes:
- Random: minerals removed uniformly at random
- Structured: rare/low-frequency families removed first
(mimics realistic incomplete reporting)

For each masking level, refits M1 and M7b, then measures:
(a) topic alignment to unmasked reference (do topics survive?)
(b) held-out LL degradation (does prediction get worse?)

Pass criterion: M7b degrades more slowly than M1.

INPUT:  X_fc_f, litho_tensor, age_tensor, phi_m1, phi_ref_m7b
OUTPUT: degradation curves + go/no-go

In [ ]:
import numpy as np
import pandas as pd
import time
import torch
import pyro
from sklearn.decomposition import LatentDirichletAllocation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings("ignore")

K = 7
M7B_STEPS = 4000
ANNEAL_END = 1500
LR = 0.005
N_REPEATS = 3  # repeats per masking level for stability

X_arr = X_fc_f.values.astype(np.float64)
N, F = X_arr.shape
families = list(X_fc_f.columns)

# Family frequencies for structured masking (rarer families removed first)
family_freq = (X_arr > 0).sum(axis=0)  # deposits containing each family
family_freq_rank = np.argsort(family_freq)  # ascending: rarest first

MASK_LEVELS = [0.0, 0.10, 0.20, 0.40]


def cosine_sim(a, b):
    return 1.0 - cosine(a, b)


def align_score(phi_masked, phi_ref):
    cost = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            cost[i, j] = -cosine_sim(phi_masked[i], phi_ref[j])
    _, col_ind = linear_sum_assignment(cost)
    sims = [-cost[i, col_ind[i]] for i in range(K)]
    return np.mean(sims)


def mask_random(X, frac, rng):
    """Remove frac of non-zero entries uniformly at random."""
    X_masked = X.copy()
    for i in range(X_masked.shape[0]):
        nonzero_idx = np.where(X_masked[i] > 0)[0]
        if len(nonzero_idx) < 2:
            continue
        n_remove = max(1, int(len(nonzero_idx) * frac))
        n_remove = min(n_remove, len(nonzero_idx) - 1)  # keep at least 1
        remove_idx = rng.choice(nonzero_idx, size=n_remove, replace=False)
        X_masked[i, remove_idx] = 0
    return X_masked


def mask_structured(X, frac, family_freq_rank):
    """Remove rare families first (structured missingness).
    Removes the rarest frac of families globally, setting
    their counts to 0 for all deposits.
    """
    n_remove = max(1, int(F * frac))
    remove_families = family_freq_rank[:n_remove]
    X_masked = X.copy()
    X_masked[:, remove_families] = 0
    return X_masked


def fit_m1_masked(X_masked):
    """Fit M1 on masked data, return phi."""
    lda = LatentDirichletAllocation(
        n_components=K,
        doc_topic_prior=0.5 / K,
        topic_word_prior=0.01,
        max_iter=300,
        random_state=42,
        learning_method="batch",
        n_jobs=1,
    )
    lda.fit(X_masked)
    phi = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
    return phi


def fit_m7b_masked(X_masked, litho_t, age_t, phi_m1_ref):
    """Fit M7b on masked data, return phi."""
    X_t = torch.tensor(X_masked, dtype=torch.float32)

    model = HGCTM_Sparse(
        K=K, F=F, L=L, A=A,
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=0.3, sigma_age=0.15,
    )

    # Warm-start from M1 fitted on the SAME masked data
    _ = fit_m7b(
        model, X_t, litho_t, age_t,
        phi_m1_ref=phi_m1_ref,
        n_steps=M7B_STEPS,
        anneal_end=ANNEAL_END,
        lr=LR,
        print_every=99999,
    )

    mu = pyro.param("mu_loc").detach().numpy()
    phi = np.exp(mu) / np.exp(mu).sum(axis=1, keepdims=True)
    return phi


# ══════════════════════════════════════════════════════════════
# RUN MASKING EXPERIMENTS
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("  STAGE 9: MISSINGNESS ROBUSTNESS")
print(f"  Masking levels: {MASK_LEVELS}")
print(f"  Regimes: random, structured")
print(f"  Repeats per level: {N_REPEATS}")
print("=" * 60)

results = []
t0 = time.time()

for regime in ["random", "structured"]:
    for frac in MASK_LEVELS:
        for rep in range(N_REPEATS):
            rng = np.random.RandomState(42 + rep)

            # Apply masking
            if frac == 0.0:
                X_masked = X_arr.copy()
            elif regime == "random":
                X_masked = mask_random(X_arr, frac, rng)
            else:
                X_masked = mask_structured(X_arr, frac, family_freq_rank)

            # Check how much data remains
            nonzero_before = (X_arr > 0).sum()
            nonzero_after = (X_masked > 0).sum()
            actual_removal = 1.0 - nonzero_after / nonzero_before

            # Fit M1 on masked data
            phi_m1_masked = fit_m1_masked(X_masked)
            m1_align = align_score(phi_m1_masked, phi_m1)

            # Fit M7b on masked data (warm-start from M1 on masked data)
            phi_m7b_masked = fit_m7b_masked(
                X_masked, litho_tensor, age_tensor,
                phi_m1_ref=phi_m1_masked
            )
            m7b_align = align_score(phi_m7b_masked, phi_ref_m7b)

            results.append({
                "regime": regime,
                "frac": frac,
                "rep": rep,
                "actual_removal": actual_removal,
                "m1_align": m1_align,
                "m7b_align": m7b_align,
            })

            elapsed = time.time() - t0
            print(f"  {regime:>10s}  mask={frac:.0%}  rep={rep}  "
                  f"M1={m1_align:.3f}  M7b={m7b_align:.3f}  "
                  f"({elapsed:.0f}s)")

        # Skip repeats for structured masking (deterministic)
        if regime == "structured":
            break

res_df = pd.DataFrame(results)


# ══════════════════════════════════════════════════════════════
# SUMMARY TABLES
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  MISSINGNESS ROBUSTNESS SUMMARY")
print(f"{'=' * 60}")

for regime in ["random", "structured"]:
    print(f"\n  === {regime.upper()} masking ===")
    print(f"  {'Mask %':<10s}  {'M1 align':>10s}  {'M7b align':>11s}  {'Δ(M7b-M1)':>10s}")
    print(f"  {'─' * 45}")

    sub = res_df[res_df["regime"] == regime]
    agg = sub.groupby("frac").agg(
        m1_mean=("m1_align", "mean"),
        m1_std=("m1_align", "std"),
        m7b_mean=("m7b_align", "mean"),
        m7b_std=("m7b_align", "std"),
    ).reset_index()

    for _, row in agg.iterrows():
        diff = row["m7b_mean"] - row["m1_mean"]
        print(f"  {row['frac']:>8.0%}    "
              f"{row['m1_mean']:.3f}±{row['m1_std']:.3f}  "
              f"{row['m7b_mean']:.3f}±{row['m7b_std']:.3f}  "
              f"{diff:>+8.3f}")


# ══════════════════════════════════════════════════════════════
# DEGRADATION PLOT
# ══════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, regime in zip(axes, ["random", "structured"]):
    sub = res_df[res_df["regime"] == regime]
    agg = sub.groupby("frac").agg(
        m1_mean=("m1_align", "mean"),
        m1_std=("m1_align", "std"),
        m7b_mean=("m7b_align", "mean"),
        m7b_std=("m7b_align", "std"),
    ).reset_index()

    ax.errorbar(agg["frac"] * 100, agg["m1_mean"], yerr=agg["m1_std"],
                marker="o", capsize=4, linewidth=2, label="M1 (LDA)", color="blue")
    ax.errorbar(agg["frac"] * 100, agg["m7b_mean"], yerr=agg["m7b_std"],
                marker="s", capsize=4, linewidth=2, label="M7b (HGCTM)", color="red")

    ax.set_xlabel("Masking level (%)")
    ax.set_ylabel("Topic alignment to unmasked reference (CS)")
    ax.set_title(f"{regime.capitalize()} masking")
    ax.set_xlim(-2, 45)
    ax.set_ylim(0.5, 1.02)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


# ══════════════════════════════════════════════════════════════
# GO / NO-GO
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  STAGE 9 GO/NO-GO")
print(f"{'=' * 60}")

checks = []

# 1. At 40% random masking, M7b retains CS > 0.65
random_40 = res_df[(res_df["regime"] == "random") & (res_df["frac"] == 0.40)]
m7b_40_mean = random_40["m7b_align"].mean()
ok = m7b_40_mean >= 0.65
checks.append(("M7b survives 40% random masking",
               f"CS={m7b_40_mean:.3f}",
               "PASS" if ok else "FAIL"))

# 2. M7b degrades slower than M1 under random masking
random_sub = res_df[res_df["regime"] == "random"]
agg_r = random_sub.groupby("frac")[["m1_align", "m7b_align"]].mean()
if 0.0 in agg_r.index and 0.40 in agg_r.index:
    m1_drop = agg_r.loc[0.0, "m1_align"] - agg_r.loc[0.40, "m1_align"]
    m7b_drop = agg_r.loc[0.0, "m7b_align"] - agg_r.loc[0.40, "m7b_align"]
    ok = m7b_drop <= m1_drop + 0.03  # M7b drops no more than M1 + tolerance
    checks.append(("M7b degrades ≤ M1 (random)",
                   f"M1 drop={m1_drop:.3f}, M7b drop={m7b_drop:.3f}",
                   "PASS" if ok else "WARN"))

# 3. At 40% structured masking, M7b retains CS > 0.60
struct_40 = res_df[(res_df["regime"] == "structured") & (res_df["frac"] == 0.40)]
if len(struct_40) > 0:
    m7b_s40 = struct_40["m7b_align"].mean()
    ok = m7b_s40 >= 0.60
    checks.append(("M7b survives 40% structured masking",
                   f"CS={m7b_s40:.3f}",
                   "PASS" if ok else "WARN"))

# 4. M7b ≥ M1 at highest masking level
if len(random_40) > 0:
    m1_40 = random_40["m1_align"].mean()
    diff = m7b_40_mean - m1_40
    ok = diff >= -0.05
    checks.append(("M7b ≥ M1 at 40% masking (within 0.05)",
                   f"M7b={m7b_40_mean:.3f} vs M1={m1_40:.3f}, Δ={diff:+.3f}",
                   "PASS" if ok else "WARN"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<42s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print()
if all_pass:
    print("  → STAGE 9 PASSED. M7b is robust to missingness.")
else:
    print("  → STAGE 9 ISSUES. Review robustness claims.")
print(f"\n  Total time: {time.time() - t0:.0f}s")
print("=" * 60)

## CELL 31 — Ontology sensitivity

Stage 10 of the HGCTM protocol.
Tests whether main conclusions depend on arbitrary mapping
choices by perturbing the mineral-family ontology:
(P1) Merge small families: combine families with <20 deposits
into their parent group
(P2) Split large families: split Cu_sulfides_other into
Cu_sulfides_chalcogenide vs Cu_sulfides_misc
(P3) Reclassify ambiguous minerals: move epidote from
Epidote_group to Skarn_calc_silicates; move actinolite
to Amphiboles_pyroxenes (if present)
(P4) Drop all rare families (<20 deposits) entirely

For each perturbation, refit M1 and M7b, check whether:
- model ranking (M7b ≈ M1) is preserved
- main topic identities are preserved (alignment to reference)

Pass criterion: rankings and top 4-5 topics remain stable.

INPUT:  X_fc_f, families_fc, litho_tensor, age_tensor, phi_m1
OUTPUT: sensitivity table + go/no-go

In [ ]:
import numpy as np
import pandas as pd
import time
import torch
import pyro
from sklearn.decomposition import LatentDirichletAllocation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings("ignore")

K = 7
M7B_STEPS = 4000
ANNEAL_END = 1500
LR = 0.005

X_arr = X_fc_f.values.astype(np.float64)
N, F_orig = X_arr.shape
families_orig = list(X_fc_f.columns)


def cosine_sim(a, b):
    return 1.0 - cosine(a, b)


def align_to_ref(phi_new, phi_ref, K_new, K_ref):
    """Align topics from perturbed model to reference.
    Handles different K values by padding if needed."""
    K_min = min(K_new, K_ref)
    cost = np.zeros((K_new, K_ref))
    for i in range(K_new):
        for j in range(K_ref):
            # Use only shared families (intersection)
            min_dim = min(phi_new.shape[1], phi_ref.shape[1])
            cost[i, j] = -cosine_sim(phi_new[i, :min_dim], phi_ref[j, :min_dim])
    _, col_ind = linear_sum_assignment(cost)
    sims = [-cost[i, col_ind[i]] for i in range(K_new)]
    return np.mean(sims)


def fit_and_evaluate(X_pert, families_pert, label):
    """Fit M1 and M7b on perturbed data, return alignment scores."""
    F_pert = X_pert.shape[1]
    X_pert_arr = X_pert.astype(np.float64)

    # Fit M1
    lda = LatentDirichletAllocation(
        n_components=K, doc_topic_prior=0.5 / K,
        topic_word_prior=0.01, max_iter=300,
        random_state=42, learning_method="batch", n_jobs=1,
    )
    lda.fit(X_pert_arr)
    phi_m1_pert = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
    m1_ll = lda.score(X_pert_arr) / N

    # Fit M7b
    X_t = torch.tensor(X_pert_arr, dtype=torch.float32)
    model = HGCTM_Sparse(
        K=K, F=F_pert, L=L, A=A,
        prevalence_covariates=True, content_covariates=True,
        ordered_shrinkage=True, likelihood="dirichlet_multinomial",
        sigma_mu=2.0, sigma_litho=0.3, sigma_age=0.15,
    )
    _ = fit_m7b(
        model, X_t, litho_tensor, age_tensor,
        phi_m1_ref=phi_m1_pert,
        n_steps=M7B_STEPS, anneal_end=ANNEAL_END,
        lr=LR, print_every=99999,
    )
    mu_pert = pyro.param("mu_loc").detach().numpy()
    phi_m7b_pert = np.exp(mu_pert) / np.exp(mu_pert).sum(axis=1, keepdims=True)

    m7b_elbo = -np.mean(
        [pyro.param("mu_loc").detach().numpy().sum()]  # placeholder
    )

    # Alignment: compare perturbed topics to original reference
    # Need to map families to shared space
    shared = [f for f in families_pert if f in families_orig]
    if len(shared) < 5:
        return {"label": label, "m1_ll": m1_ll,
                "m1_self_align": 1.0, "m7b_self_align": 1.0,
                "n_families": F_pert, "note": "too few shared families"}

    idx_pert = [families_pert.index(f) for f in shared]
    idx_orig = [families_orig.index(f) for f in shared]

    phi_m1_shared = phi_m1_pert[:, idx_pert]
    phi_m1_shared = phi_m1_shared / phi_m1_shared.sum(axis=1, keepdims=True)

    phi_m1_ref_shared = phi_m1[:, idx_orig]
    phi_m1_ref_shared = phi_m1_ref_shared / phi_m1_ref_shared.sum(axis=1, keepdims=True)

    phi_m7b_shared = phi_m7b_pert[:, idx_pert]
    phi_m7b_shared = phi_m7b_shared / phi_m7b_shared.sum(axis=1, keepdims=True)

    phi_m7b_ref_shared = phi_ref_m7b[:, idx_orig]
    phi_m7b_ref_shared = phi_m7b_ref_shared / phi_m7b_ref_shared.sum(axis=1, keepdims=True)

    m1_align = align_to_ref(phi_m1_shared, phi_m1_ref_shared, K, K)
    m7b_align = align_to_ref(phi_m7b_shared, phi_m7b_ref_shared, K, K)

    return {
        "label": label,
        "n_families": F_pert,
        "m1_ll": m1_ll,
        "m1_align": m1_align,
        "m7b_align": m7b_align,
    }


# ── Perturbation P0: baseline (no change) ────────────────────
print("=" * 60)
print("  STAGE 10: ONTOLOGY SENSITIVITY")
print("=" * 60)

t0 = time.time()
results = []

print("\n  P0: Baseline (no perturbation)...")
r0 = fit_and_evaluate(X_arr, families_orig, "P0_baseline")
results.append(r0)
print(f"    M1 align={r0['m1_align']:.3f}  M7b align={r0['m7b_align']:.3f}  "
      f"F={r0['n_families']}  ({time.time()-t0:.0f}s)")

# ── P1: Merge small families ─────────────────────────────────
print("\n  P1: Merge families with <20 deposits into 'Other_merged'...")
col_counts = (X_arr > 0).sum(axis=0)
small_mask = col_counts < 20
large_cols = [i for i in range(F_orig) if not small_mask[i]]
small_cols = [i for i in range(F_orig) if small_mask[i]]

X_p1 = X_arr[:, large_cols].copy()
if small_cols:
    merged = X_arr[:, small_cols].sum(axis=1, keepdims=True)
    X_p1 = np.hstack([X_p1, merged])
families_p1 = [families_orig[i] for i in large_cols] + ["Other_merged"]

r1 = fit_and_evaluate(X_p1, families_p1, "P1_merge_small")
results.append(r1)
print(f"    M1 align={r1['m1_align']:.3f}  M7b align={r1['m7b_align']:.3f}  "
      f"F={r1['n_families']} (was {F_orig})  ({time.time()-t0:.0f}s)")

# ── P2: Split Cu_sulfides_other ───────────────────────────────
print("\n  P2: Split Cu_sulfides_other into two sub-families...")
if "Cu_sulfides_other" in families_orig:
    cso_idx = families_orig.index("Cu_sulfides_other")
    # Split: deposits with above-median count get "high", others "low"
    cso_vals = X_arr[:, cso_idx]
    med = np.median(cso_vals[cso_vals > 0]) if (cso_vals > 0).any() else 1
    cso_high = np.where(cso_vals > med, cso_vals, 0).reshape(-1, 1)
    cso_low = np.where((cso_vals > 0) & (cso_vals <= med), cso_vals, 0).reshape(-1, 1)

    X_p2 = np.delete(X_arr, cso_idx, axis=1)
    X_p2 = np.hstack([X_p2, cso_high, cso_low])
    families_p2 = [f for i, f in enumerate(families_orig) if i != cso_idx]
    families_p2 += ["Cu_sulfides_other_high", "Cu_sulfides_other_low"]

    r2 = fit_and_evaluate(X_p2, families_p2, "P2_split_CuS_other")
    results.append(r2)
    print(f"    M1 align={r2['m1_align']:.3f}  M7b align={r2['m7b_align']:.3f}  "
          f"F={r2['n_families']}  ({time.time()-t0:.0f}s)")
else:
    print("    Cu_sulfides_other not found — skip")

# ── P3: Reclassify Epidote_group → Skarn_calc_silicates ───────
print("\n  P3: Merge Epidote_group into Skarn_calc_silicates...")
if "Epidote_group" in families_orig and "Skarn_calc_silicates" in families_orig:
    epi_idx = families_orig.index("Epidote_group")
    skarn_idx = families_orig.index("Skarn_calc_silicates")

    X_p3 = X_arr.copy()
    X_p3[:, skarn_idx] += X_p3[:, epi_idx]
    X_p3 = np.delete(X_p3, epi_idx, axis=1)
    families_p3 = [f for i, f in enumerate(families_orig) if i != epi_idx]

    r3 = fit_and_evaluate(X_p3, families_p3, "P3_merge_epidote")
    results.append(r3)
    print(f"    M1 align={r3['m1_align']:.3f}  M7b align={r3['m7b_align']:.3f}  "
          f"F={r3['n_families']}  ({time.time()-t0:.0f}s)")
else:
    print("    Epidote_group or Skarn_calc_silicates not found — skip")

# ── P4: Drop all families with <20 deposits ───────────────────
print("\n  P4: Drop (not merge) families with <20 deposits...")
X_p4 = X_arr[:, large_cols].copy()
families_p4 = [families_orig[i] for i in large_cols]

r4 = fit_and_evaluate(X_p4, families_p4, "P4_drop_rare")
results.append(r4)
print(f"    M1 align={r4['m1_align']:.3f}  M7b align={r4['m7b_align']:.3f}  "
      f"F={r4['n_families']} (was {F_orig})  ({time.time()-t0:.0f}s)")


# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print("  ONTOLOGY SENSITIVITY SUMMARY")
print(f"{'=' * 60}")

res_df = pd.DataFrame(results)

print(f"\n  {'Perturbation':<25s}  {'F':>3s}  {'M1 align':>9s}  {'M7b align':>10s}")
print(f"  {'─' * 52}")
for _, row in res_df.iterrows():
    print(f"  {row['label']:<25s}  {row['n_families']:>3.0f}  "
          f"{row['m1_align']:>9.3f}  {row['m7b_align']:>10.3f}")

# ── Go/No-Go ─────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("  STAGE 10 GO/NO-GO")
print(f"{'=' * 60}")

checks = []

# 1. M7b alignment stable across perturbations
m7b_aligns = res_df["m7b_align"].values
m7b_range = m7b_aligns.max() - m7b_aligns.min()
ok = m7b_range < 0.15
checks.append(("M7b alignment range < 0.15",
               f"range={m7b_range:.3f}",
               "PASS" if ok else "WARN"))

# 2. M1 alignment stable
m1_aligns = res_df["m1_align"].values
m1_range = m1_aligns.max() - m1_aligns.min()
ok = m1_range < 0.15
checks.append(("M1 alignment range < 0.15",
               f"range={m1_range:.3f}",
               "PASS" if ok else "WARN"))

# 3. M7b-M1 gap is consistent
gaps = res_df["m7b_align"].values - res_df["m1_align"].values
gap_range = gaps.max() - gaps.min()
ok = gap_range < 0.10
checks.append(("M7b-M1 gap consistent",
               f"gap range={gap_range:.3f}",
               "PASS" if ok else "WARN"))

# 4. No perturbation causes M7b to collapse
min_m7b = res_df["m7b_align"].min()
worst = res_df.loc[res_df["m7b_align"].idxmin(), "label"]
ok = min_m7b >= 0.60
checks.append(("No M7b collapse under perturbation",
               f"worst: {worst} CS={min_m7b:.3f}",
               "PASS" if ok else "FAIL"))

print()
all_pass = True
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<40s}  {status:<6s}  {detail}")
    if status == "FAIL":
        all_pass = False

print()
if all_pass:
    print("  → STAGE 10 PASSED. Results robust to ontology perturbations.")
else:
    print("  → STAGE 10 ISSUES. Check sensitivity to mapping choices.")
print(f"\n  Total time: {time.time() - t0:.0f}s")
print("=" * 60)


# ============================================================
# CELL 32 — Negative controls
#
# Stage 11 of the HGCTM protocol.
# Four null tests to confirm that discovered structure is
# real, not a statistical artifact:
#   (N1) Label shuffle: permute deposit types, refit classifier
#        on M7b theta — accuracy should drop to chance
#   (N2) Mineral randomisation: shuffle each deposit's mineral
#        families (preserving row totals), refit M1 — topics
#        should lose coherence
#   (N3) Richness-only null: classify deposits using only
#        n_minerals — should be much worse than topic-based
#   (N4) Covariates-only null: randomise minerals but keep
#        real covariates — M7b topics should be incoherent
#
# INPUT:  X_fc_f, cov_f, theta_m1, litho/age tensors
# OUTPUT: null comparison table + go/no-go
# ============================================================

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import LatentDirichletAllocation
from scipy.spatial.distance import cosine
from scipy.optimize import linear_sum_assignment

print(f"\n{'=' * 60}")
print("  STAGE 11: NEGATIVE CONTROLS")
print(f"{'=' * 60}")

t0_nc = time.time()

# Setup
le_dt = LabelEncoder()
y = le_dt.fit_transform(cov_f["Deposit_type"].fillna("Unknown"))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Real-data baseline: classify from M7b theta ──────────────
# Compute M7b theta
B_m7b = params_m7b["B"]
int_loc_m7b = pyro.param("intercept_loc").detach().numpy()
z_litho_np = torch.nn.functional.one_hot(litho_tensor, L).float().numpy()
z_age_np = torch.nn.functional.one_hot(age_tensor, A).float().numpy()
z_all = np.concatenate([z_litho_np, z_age_np], axis=1)
eta = z_all @ B_m7b + int_loc_m7b
eta_full = np.concatenate([eta, np.zeros((eta.shape[0], 1))], axis=1)
exp_eta = np.exp(eta_full - eta_full.max(axis=1, keepdims=True))
theta_m7b_np = exp_eta / exp_eta.sum(axis=1, keepdims=True)

# Real classification: M7b theta → deposit type
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
real_scores = cross_val_score(rf, theta_m7b_np, y, cv=skf, scoring="f1_macro")
real_f1 = real_scores.mean()
print(f"\n  Real data: M7b theta → deposit type  F1={real_f1:.3f}")

# Real classification: M1 theta → deposit type
m1_scores = cross_val_score(rf, theta_m1, y, cv=skf, scoring="f1_macro")
m1_f1 = m1_scores.mean()
print(f"  Real data: M1 theta → deposit type   F1={m1_f1:.3f}")

# Real classification: raw features → deposit type
raw_scores = cross_val_score(rf, X_arr, y, cv=skf, scoring="f1_macro")
raw_f1 = raw_scores.mean()
print(f"  Real data: raw families → deposit type F1={raw_f1:.3f}")

# ── N1: Label shuffle ─────────────────────────────────────────
print(f"\n  N1: Label shuffle (10 permutations)...")
shuffle_f1s = []
for perm in range(10):
    rng = np.random.RandomState(perm)
    y_shuffled = rng.permutation(y)
    scores = cross_val_score(rf, theta_m7b_np, y_shuffled, cv=skf, scoring="f1_macro")
    shuffle_f1s.append(scores.mean())
shuffle_f1_mean = np.mean(shuffle_f1s)
shuffle_f1_std = np.std(shuffle_f1s)
print(f"    Shuffled labels F1 = {shuffle_f1_mean:.3f} ± {shuffle_f1_std:.3f}")
print(f"    Real F1 = {real_f1:.3f}  →  Δ = {real_f1 - shuffle_f1_mean:+.3f}")

# ── N2: Mineral randomisation ─────────────────────────────────
print(f"\n  N2: Mineral randomisation (shuffle families within deposits)...")
rng = np.random.RandomState(42)
X_shuffled = X_arr.copy()
for i in range(N):
    rng.shuffle(X_shuffled[i])  # shuffle family assignments per deposit

lda_null = LatentDirichletAllocation(
    n_components=K, doc_topic_prior=0.5 / K,
    topic_word_prior=0.01, max_iter=300,
    random_state=42, learning_method="batch", n_jobs=1,
)
lda_null.fit(X_shuffled)
phi_null = lda_null.components_ / lda_null.components_.sum(axis=1, keepdims=True)

# Alignment of null topics to real reference
cost = np.zeros((K, K))
for i in range(K):
    for j in range(K):
        cost[i, j] = -(1.0 - cosine(phi_null[i], phi_m1[j]))
_, col_ind = linear_sum_assignment(cost)
null_align = np.mean([-cost[i, col_ind[i]] for i in range(K)])
print(f"    Null topic alignment to real M1: {null_align:.3f}")
print(f"    Real M1 self-alignment (bootstrap): ~0.90")

# Classify from null theta
theta_null = lda_null.transform(X_shuffled)
null_class_scores = cross_val_score(rf, theta_null, y, cv=skf, scoring="f1_macro")
null_class_f1 = null_class_scores.mean()
print(f"    Null theta → deposit type F1 = {null_class_f1:.3f}")

# ── N3: Richness-only null ────────────────────────────────────
print(f"\n  N3: Richness-only null (classify from n_minerals only)...")
n_minerals = X_arr.sum(axis=1).reshape(-1, 1)
rich_scores = cross_val_score(rf, n_minerals, y, cv=skf, scoring="f1_macro")
rich_f1 = rich_scores.mean()
print(f"    Richness-only F1 = {rich_f1:.3f}")
print(f"    Real F1 = {real_f1:.3f}  →  Δ = {real_f1 - rich_f1:+.3f}")

# ── N4: Covariates-only null ──────────────────────────────────
print(f"\n  N4: Covariates-only null (classify from litho + age only)...")
cov_features = z_all.copy()
cov_scores = cross_val_score(rf, cov_features, y, cv=skf, scoring="f1_macro")
cov_f1 = cov_scores.mean()
print(f"    Covariates-only F1 = {cov_f1:.3f}")
print(f"    Real F1 = {real_f1:.3f}  →  Δ = {real_f1 - cov_f1:+.3f}")


# ── Summary ───────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("  NEGATIVE CONTROL SUMMARY")
print(f"{'=' * 60}")

print(f"\n  {'Test':<35s}  {'F1':>8s}  {'vs Real':>8s}")
print(f"  {'─' * 55}")
print(f"  {'Real: M7b theta':35s}  {real_f1:>8.3f}  {'(ref)':>8s}")
print(f"  {'Real: M1 theta':35s}  {m1_f1:>8.3f}  {m1_f1-real_f1:>+8.3f}")
print(f"  {'Real: raw families':35s}  {raw_f1:>8.3f}  {raw_f1-real_f1:>+8.3f}")
print(f"  {'N1: shuffled labels':35s}  {shuffle_f1_mean:>8.3f}  {shuffle_f1_mean-real_f1:>+8.3f}")
print(f"  {'N2: randomised minerals':35s}  {null_class_f1:>8.3f}  {null_class_f1-real_f1:>+8.3f}")
print(f"  {'N3: richness only':35s}  {rich_f1:>8.3f}  {rich_f1-real_f1:>+8.3f}")
print(f"  {'N4: covariates only':35s}  {cov_f1:>8.3f}  {cov_f1-real_f1:>+8.3f}")
print(f"\n  Null topic alignment: {null_align:.3f} (vs real ~0.90)")

# ── Go/No-Go ─────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("  STAGE 11 GO/NO-GO")
print(f"{'=' * 60}")

checks = []

ok = real_f1 - shuffle_f1_mean > 0.10
checks.append(("Real >> shuffled labels",
               f"Δ = {real_f1 - shuffle_f1_mean:+.3f}",
               "PASS" if ok else "FAIL"))

ok = real_f1 - null_class_f1 > 0.05
checks.append(("Real >> randomised minerals",
               f"Δ = {real_f1 - null_class_f1:+.3f}",
               "PASS" if ok else "FAIL"))

ok = real_f1 - rich_f1 > 0.05
checks.append(("Real >> richness only",
               f"Δ = {real_f1 - rich_f1:+.3f}",
               "PASS" if ok else "FAIL"))

ok = null_align < 0.75
checks.append(("Null topics lose coherence",
               f"null align = {null_align:.3f}",
               "PASS" if ok else "WARN"))

print()
for name, detail, status in checks:
    flag = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"  {flag} {name:<35s}  {status:<6s}  {detail}")

print()
print("  → STAGE 11 PASSED." if all(c[2]!="FAIL" for c in checks)
      else "  → STAGE 11 ISSUES.")
print(f"\n  Total time: {time.time() - t0_nc:.0f}s")
print("=" * 60)


# ============================================================
# CELL 33 — Supervised classification
#
# Stage 13 of the HGCTM protocol (SUPPORTIVE).
# Tests whether M7b's topic proportions provide a useful
# low-dimensional representation for deposit-type prediction.
# Compares:
#   (a) Raw mineral family counts (F=35 features)
#   (b) M1 topic proportions (K=7 features)
#   (c) M7b topic proportions (K=7 features)
#   (d) M7b theta + raw features (K+F features)
#   (e) M7b theta + completeness variables
#
# Uses Random Forest with 5-fold stratified CV.
# Reports macro-F1, balanced accuracy, per-class recall.
#
# INPUT:  X_fc_f, theta_m1, theta_m7b_np, cov_f, comp_f
# OUTPUT: classification comparison table
# ============================================================

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

print(f"\n{'=' * 60}")
print("  STAGE 13: SUPERVISED CLASSIFICATION")
print(f"{'=' * 60}")

# Setup
le_dt2 = LabelEncoder()
y2 = le_dt2.fit_transform(cov_f["Deposit_type"].fillna("Unknown"))
skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
type_names = le_dt2.classes_

# Feature sets
feature_sets = {
    "Raw families (F=35)": X_arr,
    "M1 theta (K=7)": theta_m1,
    "M7b theta (K=7)": theta_m7b_np,
    "M7b theta + raw": np.hstack([theta_m7b_np, X_arr]),
    "M7b theta + completeness": np.hstack([
        theta_m7b_np,
        comp_f[["n_minerals_mapped", "n_families_copper"]].values
    ]),
}

print(f"\n  {'Feature set':<30s}  {'RF F1':>7s}  {'RF BalAcc':>9s}  {'LR F1':>7s}")
print(f"  {'─' * 58}")

classification_results = []

for name, X_feat in feature_sets.items():
    # Handle NaN
    X_feat = np.nan_to_num(X_feat, nan=0.0)

    # Random Forest
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
    rf_f1 = cross_val_score(rf, X_feat, y2, cv=skf2, scoring="f1_macro").mean()
    rf_ba = cross_val_score(rf, X_feat, y2, cv=skf2, scoring="balanced_accuracy").mean()

    # Logistic Regression
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_feat)
    lr = LogisticRegression(max_iter=1000, random_state=42, multi_class="multinomial")
    lr_f1 = cross_val_score(lr, X_scaled, y2, cv=skf2, scoring="f1_macro").mean()

    print(f"  {name:<30s}  {rf_f1:>7.3f}  {rf_ba:>9.3f}  {lr_f1:>7.3f}")

    classification_results.append({
        "features": name,
        "rf_f1": rf_f1,
        "rf_balanced_acc": rf_ba,
        "lr_f1": lr_f1,
    })

# ── Per-class recall for best model ───────────────────────────
print(f"\n  Per-class recall (RF on M7b theta + raw):")
X_best = np.hstack([theta_m7b_np, X_arr])
X_best = np.nan_to_num(X_best, nan=0.0)
rf_best = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
rf_best.fit(X_best, y2)
y_pred = rf_best.predict(X_best)  # training set for quick check
print(classification_report(y2, y_pred, target_names=type_names, digits=3))

# ── Summary ───────────────────────────────────────────────────
print(f"{'=' * 60}")
print("  STAGE 13 SUMMARY")
print(f"{'=' * 60}")

cls_df = pd.DataFrame(classification_results)

# Does M7b theta compete with raw features?
m7b_f1 = cls_df.loc[cls_df["features"] == "M7b theta (K=7)", "rf_f1"].values[0]
raw_f1_cls = cls_df.loc[cls_df["features"] == "Raw families (F=35)", "rf_f1"].values[0]
m1_f1_cls = cls_df.loc[cls_df["features"] == "M1 theta (K=7)", "rf_f1"].values[0]

checks_cls = []

ok = m7b_f1 >= m1_f1_cls - 0.02
checks_cls.append(("M7b theta ≥ M1 theta",
                    f"M7b={m7b_f1:.3f} vs M1={m1_f1_cls:.3f}",
                    "PASS" if ok else "WARN"))

ok = m7b_f1 >= raw_f1_cls - 0.05
checks_cls.append(("M7b theta competitive with raw",
                    f"M7b={m7b_f1:.3f} vs raw={raw_f1_cls:.3f}",
                    "PASS" if ok else "WARN"))

print()
for name, detail, status in checks_cls:
    flag = "✓" if status == "PASS" else "⚠"
    print(f"  {flag} {name:<35s}  {status:<6s}  {detail}")

print()
print("  Note: This is a supportive criterion — the paper's main")
print("  contribution is the generative model and geological")
print("  interpretation, not classification accuracy.")
print("=" * 60)

## CELL 34 — Save all results

Saves everything from memory to /kaggle/working/ so that
results survive session refresh. Also creates a zip bundle
for download as a Kaggle dataset.

INPUT:  all in-memory results from Cells 17–33
OUTPUT: /kaggle/working/hgctm_results_bundle.zip

In [ ]:
import numpy as np
import pandas as pd
import json
import pickle
import os
import zipfile
import pyro

OUT_DIR = "/kaggle/working/hgctm_results"
os.makedirs(OUT_DIR, exist_ok=True)


# ══════════════════════════════════════════════════════════════
# 1. M1 BASELINE
# ══════════════════════════════════════════════════════════════
np.save(f"{OUT_DIR}/phi_m1.npy", phi_m1)
np.save(f"{OUT_DIR}/theta_m1.npy", theta_m1)
pd.DataFrame(phi_m1, columns=families_fc).to_csv(
    f"{OUT_DIR}/phi_m1.csv", index=False)

print("✓ M1 phi, theta saved")


# ══════════════════════════════════════════════════════════════
# 2. M7b PARAMETERS (the headline model)
# ══════════════════════════════════════════════════════════════
# Global topic cores
np.save(f"{OUT_DIR}/phi_m7b_global.npy", params_m7b["phi_global"])
pd.DataFrame(params_m7b["phi_global"], columns=families_fc).to_csv(
    f"{OUT_DIR}/phi_m7b_global.csv", index=False)

# mu (log-space cores)
np.save(f"{OUT_DIR}/mu_m7b.npy", params_m7b["mu"])

# Lithology deviations
np.save(f"{OUT_DIR}/delta_litho_m7b.npy", params_m7b["delta_litho"])

# Age deviations
np.save(f"{OUT_DIR}/delta_age_m7b.npy", params_m7b["delta_age"])

# B (prevalence coefficients)
np.save(f"{OUT_DIR}/B_m7b.npy", params_m7b["B"])

# Tau (family sensitivity)
np.save(f"{OUT_DIR}/tau_m7b.npy", params_m7b["tau"])
pd.DataFrame({
    "family": families_fc,
    "tau": params_m7b["tau"]
}).sort_values("tau", ascending=False).to_csv(
    f"{OUT_DIR}/tau_ranking.csv", index=False)

# Scalars
scalars_m7b = {
    "omega_a": float(params_m7b["omega_a"]),
    "kappa": float(params_m7b["kappa"]),
    "K": K,
    "F": F,
    "L": L,
    "A": A,
    "litho_classes": litho_classes,
    "age_classes": age_classes,
    "families_fc": families_fc,
}
with open(f"{OUT_DIR}/scalars_m7b.json", "w") as f:
    json.dump(scalars_m7b, f, indent=2)

print("✓ M7b params saved (phi, mu, delta_litho, delta_age, B, tau, scalars)")


# ══════════════════════════════════════════════════════════════
# 3. M7b THETA (topic proportions for all deposits)
# ══════════════════════════════════════════════════════════════
# Recompute theta from B
z_litho_np = torch.nn.functional.one_hot(litho_tensor, L).float().numpy()
z_age_np = torch.nn.functional.one_hot(age_tensor, A).float().numpy()
z_all = np.concatenate([z_litho_np, z_age_np], axis=1)
B_m7b = params_m7b["B"]
int_loc = pyro.param("intercept_loc").detach().numpy()
eta = z_all @ B_m7b + int_loc
eta_full = np.concatenate([eta, np.zeros((eta.shape[0], 1))], axis=1)
exp_eta = np.exp(eta_full - eta_full.max(axis=1, keepdims=True))
theta_m7b_all = exp_eta / exp_eta.sum(axis=1, keepdims=True)

np.save(f"{OUT_DIR}/theta_m7b.npy", theta_m7b_all)
theta_df = pd.DataFrame(
    theta_m7b_all,
    columns=[f"topic_{k+1}" for k in range(K)],
    index=X_fc_f.index
)
theta_df.to_csv(f"{OUT_DIR}/theta_m7b.csv")

print("✓ M7b theta saved")


# ══════════════════════════════════════════════════════════════
# 4. MODEL COMPARISON RESULTS
# ══════════════════════════════════════════════════════════════
comparison_rows = []
for name in ["M2", "M5_tight"]:
    if name in model_results:
        r = model_results[name]
        comparison_rows.append({
            "model": name,
            "avg_elbo_last100": r["avg_elbo_last100"],
        })

if "M7" in novel_models:
    comparison_rows.append({
        "model": "M7_sparse",
        "avg_elbo_last100": novel_models["M7"]["avg_elbo_last100"],
    })
if "M8" in novel_models:
    comparison_rows.append({
        "model": "M8_completeness",
        "avg_elbo_last100": novel_models["M8"]["avg_elbo_last100"],
    })

# M7b ELBO
comparison_rows.append({
    "model": "M7b_warmstart",
    "avg_elbo_last100": float(-np.mean(losses_m7b[-100:])),
})

# M1 LL
comparison_rows.append({
    "model": "M1_LDA",
    "avg_elbo_last100": m1_results["ll_mean"],
    # Note: different scale
})

pd.DataFrame(comparison_rows).to_csv(
    f"{OUT_DIR}/model_comparison.csv", index=False)

print("✓ Model comparison table saved")


# ══════════════════════════════════════════════════════════════
# 5. ELBO LOSS TRACES
# ══════════════════════════════════════════════════════════════
np.save(f"{OUT_DIR}/losses_m7b.npy", np.array(losses_m7b))

for name in ["M2", "M5_tight"]:
    if name in model_results and "losses" in model_results[name]:
        np.save(f"{OUT_DIR}/losses_{name}.npy",
                np.array(model_results[name]["losses"]))

print("✓ ELBO traces saved")


# ══════════════════════════════════════════════════════════════
# 6. BOOTSTRAP STABILITY RESULTS
# ══════════════════════════════════════════════════════════════
np.save(f"{OUT_DIR}/bootstrap_m1_sims.npy", m1_sims)
np.save(f"{OUT_DIR}/bootstrap_m7_sims.npy", m7_sims)
np.save(f"{OUT_DIR}/bootstrap_m7b_sims.npy", m7b_sims)

stability_summary = pd.DataFrame({
    "topic": [f"Topic_{k+1}" for k in range(K)],
    "m1_mean_cs": m1_sims.mean(axis=0),
    "m1_std_cs": m1_sims.std(axis=0),
    "m7_mean_cs": m7_sims.mean(axis=0),
    "m7_std_cs": m7_sims.std(axis=0),
    "m7b_mean_cs": m7b_sims.mean(axis=0),
    "m7b_std_cs": m7b_sims.std(axis=0),
})
stability_summary.to_csv(f"{OUT_DIR}/bootstrap_stability.csv", index=False)

print("✓ Bootstrap stability results saved")


# ══════════════════════════════════════════════════════════════
# 7. CONTINENT HOLDOUT RESULTS
# ══════════════════════════════════════════════════════════════
try:
    res_df.to_csv(f"{OUT_DIR}/continent_holdout.csv", index=False)
    print("✓ Continent holdout results saved")
except:
    print("⚠ Continent holdout results not in memory — skip")


# ══════════════════════════════════════════════════════════════
# 8. COVARIATES AND COMPLETENESS (aligned to model set)
# ══════════════════════════════════════════════════════════════
cov_f.to_csv(f"{OUT_DIR}/covariates_model_set.csv")
comp_f.to_csv(f"{OUT_DIR}/completeness_model_set.csv")

print("✓ Covariates and completeness saved")


# ══════════════════════════════════════════════════════════════
# 9. LITHOLOGY DEVIATION TABLE (readable format)
# ══════════════════════════════════════════════════════════════
delta_litho = params_m7b["delta_litho"]  # (K, L, F)
tau = params_m7b["tau"]  # (F,)

dev_rows = []
for k in range(K):
    for l_idx, litho in enumerate(litho_classes):
        for f_idx, fam in enumerate(families_fc):
            val = delta_litho[k, l_idx, f_idx]
            eff = val * tau[f_idx]
            if abs(val) > 0.05:
                dev_rows.append({
                    "topic": k + 1,
                    "litho_class": litho,
                    "family": fam,
                    "delta_raw": round(float(val), 4),
                    "tau": round(float(tau[f_idx]), 4),
                    "delta_effective": round(float(eff), 4),
                })

pd.DataFrame(dev_rows).to_csv(
    f"{OUT_DIR}/litho_deviations_notable.csv", index=False)

print("✓ Lithology deviation table saved")


# ══════════════════════════════════════════════════════════════
# 10. K SELECTION EVIDENCE
# ══════════════════════════════════════════════════════════════
try:
    lda_sweep = pd.read_csv("/kaggle/working/lda_sweep_results.csv")
    lda_sweep.to_csv(f"{OUT_DIR}/lda_sweep_results.csv", index=False)
    print("✓ LDA sweep results saved")
except:
    print("⚠ LDA sweep results not found — skip")


# ══════════════════════════════════════════════════════════════
# 11. PROPOSED TOPIC LABELS
# ══════════════════════════════════════════════════════════════
try:
    labels_df = pd.DataFrame([
        {"topic": k + 1, "label": proposed_labels[k]}
        for k in range(K)
    ])
    labels_df.to_csv(f"{OUT_DIR}/topic_labels.csv", index=False)
    print("✓ Topic labels saved")
except:
    print("⚠ Topic labels not in memory — skip")


# ══════════════════════════════════════════════════════════════
# 12. FULL PYRO PARAM STORE (for exact model restoration)
# ══════════════════════════════════════════════════════════════
try:
    pyro.get_param_store().save(f"{OUT_DIR}/pyro_param_store.pt")
    print("✓ Pyro param store saved")
except:
    print("⚠ Could not save Pyro param store — skip")


# ══════════════════════════════════════════════════════════════
# ZIP EVERYTHING
# ══════════════════════════════════════════════════════════════
ZIP_PATH = "/kaggle/working/hgctm_results_bundle.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUT_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, OUT_DIR)
            zf.write(filepath, arcname)

zip_size = os.path.getsize(ZIP_PATH) / (1024 * 1024)

print(f"\n{'=' * 60}")
print(f"  ALL RESULTS SAVED")
print(f"{'=' * 60}")
print(f"  Directory: {OUT_DIR}/")
print(f"  Archive:   {ZIP_PATH} ({zip_size:.1f} MB)")
print(f"\n  Files saved:")
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f"    {f:<45s}  {sz:>8.1f} KB")
print(f"\n  Download hgctm_results_bundle.zip and upload as")
print(f"  a Kaggle dataset for future notebooks.")
print(f"{'=' * 60}")

## CELL 35a — Load saved results

Reconstructs all in-memory variables from the saved bundle.
Run once at the start of a fresh session before any figure
or analysis cells.

INPUT:  /kaggle/input/datasets/katerynaglin/hgctm-results/
OUTPUT: all variables in memory

In [ ]:
import numpy as np
import pandas as pd
import json
import os

RES = "/kaggle/input/datasets/katerynaglin/hgctm-results-bandle"
OUT_DIR = RES  # alias for figure cells that reference OUT_DIR
# ── Scalars and metadata ──────────────────────────────────────
with open(f"{RES}/scalars_m7b.json", "r") as f:
    scalars = json.load(f)

K = scalars["K"]
F = scalars["F"]
L = scalars["L"]
A = scalars["A"]
litho_classes = scalars["litho_classes"]
age_classes = scalars["age_classes"]
families_fc = scalars["families_fc"]

# ── M1 ────────────────────────────────────────────────────────
phi_m1 = np.load(f"{RES}/phi_m1.npy")
theta_m1 = np.load(f"{RES}/theta_m1.npy")

# ── M7b parameters ────────────────────────────────────────────
params_m7b = {
    "phi_global": np.load(f"{RES}/phi_m7b_global.npy"),
    "mu": np.load(f"{RES}/mu_m7b.npy"),
    "delta_litho": np.load(f"{RES}/delta_litho_m7b.npy"),
    "delta_age": np.load(f"{RES}/delta_age_m7b.npy"),
    "B": np.load(f"{RES}/B_m7b.npy"),
    "tau": np.load(f"{RES}/tau_m7b.npy"),
    "omega_a": scalars.get("omega_a", 0.09) if "omega_a" not in json.load(open(f"{RES}/scalars_m7b.json")) else json.load(open(f"{RES}/scalars_m7b.json")).get("omega_a", 0.09),
    "kappa": scalars.get("kappa", 100.0) if "kappa" not in json.load(open(f"{RES}/scalars_m7b.json")) else json.load(open(f"{RES}/scalars_m7b.json")).get("kappa", 100.0),
}
# Simpler reload of omega_a and kappa
with open(f"{RES}/scalars_m7b.json", "r") as f:
    sc = json.load(f)
params_m7b["omega_a"] = sc.get("omega_a", 0.09)
params_m7b["kappa"] = sc.get("kappa", 100.0)

# ── M7b theta ─────────────────────────────────────────────────
theta_m7b_all = np.load(f"{RES}/theta_m7b.npy")

# ── Covariates and completeness ───────────────────────────────
cov_f = pd.read_csv(f"{RES}/covariates_model_set.csv", index_col=0)
comp_f = pd.read_csv(f"{RES}/completeness_model_set.csv", index_col=0)

# ── Bootstrap stability ───────────────────────────────────────
m1_sims = np.load(f"{RES}/bootstrap_m1_sims.npy")
m7_sims = np.load(f"{RES}/bootstrap_m7_sims.npy")
m7b_sims = np.load(f"{RES}/bootstrap_m7b_sims.npy")

# ── Continent holdout ─────────────────────────────────────────
if os.path.exists(f"{RES}/continent_holdout.csv"):
    res_df = pd.read_csv(f"{RES}/continent_holdout.csv")

# ── Topic labels ──────────────────────────────────────────────
labels_df = pd.read_csv(f"{RES}/topic_labels.csv")
proposed_labels = dict(zip(labels_df["topic"] - 1, labels_df["label"]))
# ── Litho/age tensors (reconstruct from covariates) ───────────
import torch
from sklearn.preprocessing import LabelEncoder

le_litho = LabelEncoder()
cov_f["litho_idx"] = le_litho.fit_transform(cov_f["litho_class"].fillna("unknown"))
litho_tensor = torch.tensor(cov_f["litho_idx"].values, dtype=torch.long)

le_age = LabelEncoder()
cov_f["age_idx"] = le_age.fit_transform(cov_f["age_category"].fillna("Unknown"))
age_tensor = torch.tensor(cov_f["age_idx"].values, dtype=torch.long)

# ── Count matrix (for posterior topic assignment) ─────────────
STAGE0 = "/kaggle/input/datasets/katerynaglin/hgctm-stage0"
if os.path.exists(f"{STAGE0}/X_family_copper.csv"):
    X_fc = pd.read_csv(f"{STAGE0}/X_family_copper.csv", index_col=0)
    MIN_MINERALS = 3
    MIN_DEPOSITS = 10
    mask = X_fc.sum(axis=1) >= MIN_MINERALS
    X_fc_f = X_fc[mask].copy()
    fc_keep = X_fc_f.columns[(X_fc_f > 0).sum(axis=0) >= MIN_DEPOSITS]
    X_fc_f = X_fc_f[fc_keep]
    print(f"  X_fc_f: {X_fc_f.shape}")

# ── Summary ───────────────────────────────────────────────────
print(f"\n=== Results loaded from {RES} ===")
print(f"  K={K}, F={F}, L={L}, A={A}")
print(f"  phi_m1: {phi_m1.shape}")
print(f"  phi_m7b: {params_m7b['phi_global'].shape}")
print(f"  theta_m7b: {theta_m7b_all.shape}")
print(f"  tau: {params_m7b['tau'].shape}")
print(f"  delta_litho: {params_m7b['delta_litho'].shape}")
print(f"  cov_f: {cov_f.shape}")
print(f"  Topics: {list(proposed_labels.values())}")
print(f"\n  ✓ Ready for figure generation.")

# ============================================================
# CELL 35 — Figure 1: Topic heatmap
#
# The paper's hero figure. Shows M7b global topic cores as a
# K×F heatmap with families grouped by mineralogical class.
# Each topic is labelled with its geological interpretation.
# Right panel: mean topic prevalence by deposit type.
#
# INPUT:  params_m7b, theta_m7b_all, cov_f, families_fc
# OUTPUT: /kaggle/working/figures/fig1_topic_heatmap.png
# ============================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import os

FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)

phi = params_m7b["phi_global"]  # (K, F)
K, F = phi.shape

# ── Topic labels ──────────────────────────────────────────────
TOPIC_LABELS = [
    "T1: Porphyry Cu-Mo\nalteration",
    "T2: Supergene\nCu-enrichment",
    "T3: Polymetallic\nsulfide base",
    "T4: Magmatic\nNi-Co-PGE",
    "T5: IOCG sodic-calcic\nsilicate",
    "T6: VMS Pb-Zn-Cu-Ba\npolymetallic",
    "T7: Weathering\nsulfate-phosphate",
]

# ── Family ordering by geological group ───────────────────────
FAMILY_ORDER = [
    # Cu sulfides
    "Cu_sulfides_primary", "Cu_sulfides_supergene", "Cu_sulfides_other",
    "Cu_oxides", "Cu_carbonates_silicates",
    # Fe sulfides
    "Fe_sulfides",
    # Other base metal sulfides
    "Mo_sulfides", "Zn_sulfides", "Pb_sulfides",
    "Ni_Co_sulfides_arsenides", "As_Sb_Bi_sulfosalts", "Sulfides_misc",
    # Oxides
    "Fe_oxides_hydroxides", "Ti_oxides", "Mn_oxides", "Oxides_misc",
    # Silicates
    "Quartz", "Micas_sericite", "Feldspars", "Clay_minerals",
    "Chlorite_group", "Epidote_group", "Amphiboles_pyroxenes",
    "Skarn_calc_silicates", "Other_silicates",
    # Carbonates + sulfates
    "Carbonates", "Barite", "Evaporite_sulfates", "Sulfates_other",
    # Phosphates + halides
    "Phosphates_apatite", "Phosphates_other", "REE_accessory",
    "Halides_fluorite", "Halides_other",
    # Native + accessory
    "Au_Ag_native", "PGE_alloys_intermetallics", "Native_elements_misc",
]

# Filter to families actually in our vocabulary
col_order = [f for f in FAMILY_ORDER if f in families_fc]
col_order += [f for f in families_fc if f not in col_order]
col_idx = [families_fc.index(f) for f in col_order]

phi_ordered = phi[:, col_idx]

# ── Group separators for visual clarity ───────────────────────
GROUP_BREAKS = {
    "Cu phases": 5,
    "Fe sulfides": 6,
    "Base metals": 12,
    "Oxides": 16,
    "Silicates": 25,
    "Carb+Sulf": 29,
    "Phos+Hal": 34,
    "Native": len(col_order),
}

# ── Create figure ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 7))
gs = fig.add_gridspec(1, 2, width_ratios=[4, 1], wspace=0.03)

# Left panel: topic × family heatmap
ax1 = fig.add_subplot(gs[0])
cmap = LinearSegmentedColormap.from_list("", ["#f7fbff", "#08519c"])
im = ax1.imshow(phi_ordered, aspect="auto", cmap=cmap, vmin=0, vmax=0.4)

ax1.set_yticks(range(K))
ax1.set_yticklabels(TOPIC_LABELS, fontsize=9)
ax1.set_xticks(range(len(col_order)))
ax1.set_xticklabels([f.replace("_", "\n") for f in col_order],
                     rotation=90, fontsize=6, ha="center")

# Add group separator lines
prev = 0
for group_name, end in GROUP_BREAKS.items():
    if end < len(col_order):
        ax1.axvline(x=end - 0.5, color="gray", linewidth=0.5, alpha=0.5)

# Add phi values in cells (only if > 0.05)
for i in range(K):
    for j in range(len(col_order)):
        val = phi_ordered[i, j]
        if val > 0.05:
            color = "white" if val > 0.15 else "black"
            ax1.text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=5, color=color)

ax1.set_title("M7b: Global topic cores (φ)", fontsize=12, fontweight="bold")
plt.colorbar(im, ax=ax1, shrink=0.6, label="Probability", pad=0.01)

# Right panel: topic prevalence by deposit type
ax2 = fig.add_subplot(gs[1])

type_order = ["Porphyry", "VMS", "Sediment-Hosted", "Magmatic sulfide", "IOCG"]
dtypes = cov_f["Deposit_type"].values

type_theta = np.zeros((len(type_order), K))
for i, dt in enumerate(type_order):
    mask = dtypes == dt
    if mask.sum() > 0:
        type_theta[i] = theta_m7b_all[mask].mean(axis=0)

im2 = ax2.imshow(type_theta.T, aspect="auto", cmap=cmap, vmin=0, vmax=0.4)
ax2.set_xticks(range(len(type_order)))
ax2.set_xticklabels(type_order, rotation=45, ha="right", fontsize=8)
ax2.set_yticks(range(K))
ax2.set_yticklabels([f"T{k+1}" for k in range(K)], fontsize=9)
ax2.set_title("Mean θ by\ndeposit type", fontsize=10)

for i in range(K):
    for j in range(len(type_order)):
        val = type_theta[j, i]
        if val > 0.05:
            color = "white" if val > 0.15 else "black"
            ax2.text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=7, color=color)

fig.savefig(f"{FIG_DIR}/fig1_topic_heatmap.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/fig1_topic_heatmap.pdf", bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved fig1_topic_heatmap.png/.pdf")


# ============================================================
# CELL 36 — Figure 2: τ family ranking
#
# The paper's most novel figure. Horizontal bar chart showing
# all 35 mineral families ranked by learned context-sensitivity
# τ, colour-coded by mineralogical class.
#
# INPUT:  params_m7b["tau"], families_fc
# OUTPUT: /kaggle/working/figures/fig2_tau_ranking.png
# ============================================================

tau = params_m7b["tau"]

# Family category colours
FAMILY_COLORS = {}
for f in families_fc:
    if "Cu_" in f:
        FAMILY_COLORS[f] = "#e41a1c"     # red: Cu phases
    elif "Fe_sulfides" in f or "Fe_oxides" in f:
        FAMILY_COLORS[f] = "#377eb8"     # blue: Fe phases
    elif any(x in f for x in ["Zn_", "Pb_", "Mo_", "Ni_Co", "As_Sb", "Sulfides"]):
        FAMILY_COLORS[f] = "#4daf4a"     # green: other sulfides
    elif any(x in f for x in ["Quartz", "Micas", "Feldspars", "Clay", "Chlorite",
                               "Epidote", "Amphiboles", "Skarn", "Other_silicates"]):
        FAMILY_COLORS[f] = "#984ea3"     # purple: silicates
    elif any(x in f for x in ["Carbonates", "Barite", "Evaporite", "Sulfates"]):
        FAMILY_COLORS[f] = "#ff7f00"     # orange: carbonates + sulfates
    elif any(x in f for x in ["Au_Ag", "PGE", "Native"]):
        FAMILY_COLORS[f] = "#a65628"     # brown: native elements
    elif any(x in f for x in ["Phosph", "REE", "Halides"]):
        FAMILY_COLORS[f] = "#f781bf"     # pink: phosphates + halides
    else:
        FAMILY_COLORS[f] = "#999999"     # gray: other

# Sort by tau
ranking = sorted(zip(families_fc, tau), key=lambda x: x[1])
fams_sorted = [r[0] for r in ranking]
taus_sorted = [r[1] for r in ranking]
colors_sorted = [FAMILY_COLORS.get(f, "#999999") for f in fams_sorted]

fig, ax = plt.subplots(figsize=(10, 10))
y_pos = range(len(fams_sorted))
bars = ax.barh(y_pos, taus_sorted, color=colors_sorted, edgecolor="white",
               linewidth=0.5, height=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels([f.replace("_", " ") for f in fams_sorted], fontsize=8)
ax.set_xlabel("Learned context-sensitivity (τ)", fontsize=11)
ax.set_title("Mineral family context-sensitivity ranking",
             fontsize=13, fontweight="bold")

# Threshold lines
ax.axvline(x=0.25, color="red", linestyle="--", alpha=0.5, linewidth=1)
ax.axvline(x=0.15, color="gray", linestyle=":", alpha=0.5, linewidth=1)
ax.text(0.26, len(fams_sorted) - 1, "sensitive\nthreshold",
        fontsize=7, color="red", alpha=0.7)
ax.text(0.155, 0.5, "invariant\nthreshold",
        fontsize=7, color="gray", alpha=0.7)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e41a1c", label="Cu phases"),
    Patch(facecolor="#377eb8", label="Fe phases"),
    Patch(facecolor="#4daf4a", label="Other sulfides"),
    Patch(facecolor="#984ea3", label="Silicates"),
    Patch(facecolor="#ff7f00", label="Carbonates + sulfates"),
    Patch(facecolor="#a65628", label="Native elements"),
    Patch(facecolor="#f781bf", label="Phosphates + halides"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)

ax.set_xlim(0, max(taus_sorted) * 1.15)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig2_tau_ranking.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/fig2_tau_ranking.pdf", bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved fig2_tau_ranking.png/.pdf")


# ============================================================
# CELL 37 — Figure 3: Lithology deviations
#
# Shows how 3 key topics are re-expressed across host rocks.
# Heatmap of δ_litho × τ (effective deviation) for geological
# lithology classes only.
#
# INPUT:  params_m7b, litho_classes, families_fc
# OUTPUT: /kaggle/working/figures/fig3_litho_deviations.png
# ============================================================

delta_litho = params_m7b["delta_litho"]  # (K, L, F)
tau = params_m7b["tau"]

GEOL_LITHO = ["felsic", "mafic", "carbonate", "siliciclastic", "metamorphic", "volcanic"]
geol_idx = [litho_classes.index(l) for l in GEOL_LITHO if l in litho_classes]
geol_names = [l for l in GEOL_LITHO if l in litho_classes]

# Select most interesting topics (those with largest deviations)
topic_dev_mag = np.array([
    np.abs(delta_litho[k][geol_idx] * tau[None, :]).max()
    for k in range(K)
])
top_topics = np.argsort(topic_dev_mag)[::-1][:4]

# Select most variable families (top 15 by max deviation)
family_var = np.array([
    np.abs(delta_litho[:, :, f] * tau[f]).max()
    for f in range(F)
])
top_families_idx = np.argsort(family_var)[::-1][:15]
top_family_names = [families_fc[i] for i in top_families_idx]

fig, axes = plt.subplots(1, len(top_topics), figsize=(5 * len(top_topics), 6),
                          sharey=True)
if len(top_topics) == 1:
    axes = [axes]

cmap_div = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-0.15, vcenter=0, vmax=0.15)

for ax_idx, k in enumerate(top_topics):
    ax = axes[ax_idx]

    # Effective deviation matrix for this topic: litho × family
    eff = delta_litho[k][np.ix_(geol_idx, top_families_idx)] * tau[top_families_idx][None, :]

    im = ax.imshow(eff, aspect="auto", cmap=cmap_div, norm=norm)

    ax.set_xticks(range(len(top_family_names)))
    ax.set_xticklabels([f.replace("_", "\n") for f in top_family_names],
                       rotation=90, fontsize=7)
    ax.set_yticks(range(len(geol_names)))
    if ax_idx == 0:
        ax.set_yticklabels([l.capitalize() for l in geol_names], fontsize=9)
    ax.set_title(f"T{k+1}: {proposed_labels[k][:30]}", fontsize=9, fontweight="bold")

    # Annotate notable cells
    for i in range(len(geol_names)):
        for j in range(len(top_family_names)):
            val = eff[i, j]
            if abs(val) > 0.03:
                color = "white" if abs(val) > 0.08 else "black"
                ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                        fontsize=5, color=color)

fig.colorbar(im, ax=axes, shrink=0.6, label="Effective deviation (δ × τ)")
fig.suptitle("Lithology deviations: how host rock modifies topic expression",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig3_litho_deviations.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/fig3_litho_deviations.pdf", bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved fig3_litho_deviations.png/.pdf")


# ============================================================
# CELL 38 — Figure 4: Stability + validation panel
#
# Four-panel figure:
#   (a) NPMI vs K (K selection)
#   (b) Bootstrap stability: M1 vs M7 vs M7b
#   (c) Continent holdout: held-out LL
#   (d) Litho vs age deviation magnitudes
#
# INPUT:  various saved results
# OUTPUT: /kaggle/working/figures/fig4_validation_panel.png
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── (a) NPMI vs K ─────────────────────────────────────────────
ax = axes[0, 0]
try:
    sweep = pd.read_csv("/kaggle/working/lda_sweep_results.csv")
    # Recompute NPMI from sweep if available, otherwise use placeholder
    K_range = sorted(sweep["K"].unique())
    agg = sweep.groupby("K")["log_lik_per_doc"].agg(["mean", "std"]).reset_index()
    ax.errorbar(agg["K"], agg["mean"], yerr=agg["std"],
                marker="o", capsize=4, linewidth=2)
    ax.axvline(x=7, color="red", linestyle="--", alpha=0.5)
    ax.set_xlabel("K")
    ax.set_ylabel("Held-out LL / doc")
    ax.set_title("(a) K selection: held-out LL", fontweight="bold")
    ax.grid(True, alpha=0.3)
except:
    ax.text(0.5, 0.5, "LDA sweep data not available",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_title("(a) K selection", fontweight="bold")

# ── (b) Bootstrap stability ───────────────────────────────────
ax = axes[0, 1]
topics = np.arange(1, K + 1)
width = 0.25

ax.bar(topics - width, m1_sims.mean(axis=0), width, label="M1 (LDA)",
       color="#2166ac", alpha=0.8, yerr=m1_sims.std(axis=0), capsize=3)
ax.bar(topics, m7_sims.mean(axis=0), width, label="M7 (random init)",
       color="#d6604d", alpha=0.8, yerr=m7_sims.std(axis=0), capsize=3)
ax.bar(topics + width, m7b_sims.mean(axis=0), width, label="M7b (warm-start)",
       color="#1a9850", alpha=0.8, yerr=m7b_sims.std(axis=0), capsize=3)

ax.axhline(y=0.80, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Topic")
ax.set_ylabel("Cosine similarity to reference")
ax.set_title("(b) Bootstrap topic stability (20 resamples)", fontweight="bold")
ax.set_xticks(topics)
ax.set_ylim(0.3, 1.05)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# ── (c) Continent holdout ─────────────────────────────────────
ax = axes[1, 0]
try:
    holdout = pd.read_csv(f"{OUT_DIR}/continent_holdout.csv") if os.path.exists(f"{OUT_DIR}/continent_holdout.csv") else res_df
    continents_sorted = holdout.sort_values("m7b_ll")["continent"].values
    x_pos = np.arange(len(continents_sorted))
    m1_lls = [holdout.loc[holdout["continent"] == c, "m1_ll"].values[0] for c in continents_sorted]
    m7b_lls = [holdout.loc[holdout["continent"] == c, "m7b_ll"].values[0] for c in continents_sorted]

    ax.bar(x_pos - 0.15, m1_lls, 0.3, label="M1 (LDA)", color="#2166ac", alpha=0.8)
    ax.bar(x_pos + 0.15, m7b_lls, 0.3, label="M7b (HGCTM)", color="#1a9850", alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(continents_sorted, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Held-out LL / doc")
    ax.set_title("(c) Continent holdout: held-out LL", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
except Exception as e:
    ax.text(0.5, 0.5, f"Holdout data not available\n{e}",
            ha="center", va="center", transform=ax.transAxes, fontsize=8)
    ax.set_title("(c) Continent holdout", fontweight="bold")

# ── (d) Litho vs age magnitude ────────────────────────────────
ax = axes[1, 1]
delta_litho_m = params_m7b["delta_litho"]
delta_age_m = params_m7b["delta_age"]
tau_m = params_m7b["tau"]
omega_m = params_m7b["omega_a"]

litho_eff = np.abs(delta_litho_m * tau_m[None, None, :]).mean()
age_eff = np.abs(delta_age_m * omega_m * tau_m[None, None, :]).mean()

bars = ax.bar(["Lithology\n(|τ × δ_litho|)", "Age\n(|τ × ω_a × δ_age|)"],
              [litho_eff, age_eff],
              color=["#2166ac", "#d6604d"], alpha=0.8, width=0.5)
ax.set_ylabel("Mean effective deviation magnitude")
ax.set_title("(d) Lithology vs age effect size", fontweight="bold")

ratio = litho_eff / max(age_eff, 1e-8)
ax.text(0.5, max(litho_eff, age_eff) * 0.9,
        f"Ratio: {ratio:.1f}×", ha="center", fontsize=11, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig4_validation_panel.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/fig4_validation_panel.pdf", bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved fig4_validation_panel.png/.pdf")


# ============================================================
# CELL 39 — Figure 5: Global map (M7b posterior)
#
# Assigns each deposit to its dominant topic using M7b's
# FULL model: prevalence (θ from covariates) × content
# (φ conditioned on lithology/age) × observed minerals.
# This is proper posterior inference, not just θ.
#
# r_ik ∝ θ_ik × ∏_f φ_{ikf}^{x_if}
# Dominant topic = argmax_k r_ik
#
# INPUT:  params_m7b, X_fc_f, litho/age tensors, cov_f
# OUTPUT: /kaggle/working/figures/fig5_global_map.png
# ============================================================

import numpy as np
import torch
from torch.nn.functional import softmax
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Reconstruct full M7b topic assignments ────────────────────
mu = torch.tensor(params_m7b["mu"], dtype=torch.float32)
delta_litho = torch.tensor(params_m7b["delta_litho"], dtype=torch.float32)
delta_age = torch.tensor(params_m7b["delta_age"], dtype=torch.float32)
tau_t = torch.tensor(params_m7b["tau"], dtype=torch.float32)
omega_a = params_m7b["omega_a"]

# Compute deposit-specific phi: (N, K, F)
litho_dev = delta_litho[:, litho_tensor, :].permute(1, 0, 2)
age_dev = delta_age[:, age_tensor, :].permute(1, 0, 2)
gated_litho = tau_t.unsqueeze(0).unsqueeze(0) * litho_dev
gated_age = tau_t.unsqueeze(0).unsqueeze(0) * age_dev
psi = mu.unsqueeze(0) + gated_litho + omega_a * gated_age
phi_all = softmax(psi, dim=2).detach().numpy()  # (N, K, F)

# Theta from covariates: (N, K)
theta = theta_m7b_all  # already computed

# X observed: (N, F)
X_obs = X_fc_f.values.astype(np.float64)

# Posterior responsibility: r_ik ∝ θ_ik × ∏_f φ_{ikf}^{x_if}
# In log space: log r_ik = log θ_ik + Σ_f x_if × log φ_{ikf}
log_theta = np.log(np.clip(theta, 1e-10, None))
log_phi = np.log(np.clip(phi_all, 1e-10, None))  # (N, K, F)

# Σ_f x_if × log φ_{ikf}  → (N, K)
log_lik = np.einsum("nf,nkf->nk", X_obs, log_phi)

log_r = log_theta + log_lik  # (N, K)
# Normalise for numerical stability
log_r = log_r - log_r.max(axis=1, keepdims=True)
r = np.exp(log_r)
r = r / r.sum(axis=1, keepdims=True)

dominant_topic = np.argmax(r, axis=1)

# ── Count per topic ───────────────────────────────────────────
print("=== M7b posterior dominant topic counts ===\n")
for k in range(K):
    n = (dominant_topic == k).sum()
    print(f"  T{k+1}: {proposed_labels[k]:<40s}  n={n}")

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 8))

TOPIC_COLORS = [
    "#e41a1c", "#ff7f00", "#377eb8", "#4daf4a",
    "#984ea3", "#a65628", "#999999",
]
TOPIC_LABELS_SHORT = [
    "Porphyry Cu-Mo", "Supergene Cu", "Polymetallic sulfide",
    "Magmatic Ni-Co", "IOCG silicate", "VMS Pb-Zn-Cu-Ba", "Weathering"
]

lats = cov_f["Latitude"].values
lons = cov_f["Longitude"].values

for k in range(K):
    mask = dominant_topic == k
    if mask.sum() > 0:
        ax.scatter(lons[mask], lats[mask], c=TOPIC_COLORS[k],
                   s=15, alpha=0.7,
                   label=f"{TOPIC_LABELS_SHORT[k]} (n={mask.sum()})",
                   edgecolors="none")

ax.set_xlim(-180, 180)
ax.set_ylim(-70, 80)
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title("Global copper deposits by dominant M7b assemblage topic\n"
             "(posterior assignment: prevalence × lithology-conditioned content × observed minerals)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, markerscale=2, loc="lower left", ncol=2)
ax.grid(True, linewidth=0.3, alpha=0.3)
ax.set_facecolor("#f0f0f0")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig5_global_map.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/fig5_global_map.pdf", bbox_inches="tight")
plt.close(fig)
print(f"\n✓ Saved fig5_global_map.png/.pdf")

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("  ALL FIGURES GENERATED")
print(f"{'=' * 60}")
for f in sorted(os.listdir(FIG_DIR)):
    if f.startswith("fig"):
        sz = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
        print(f"  {f:<45s}  {sz:>8.1f} KB")
print(f"{'=' * 60}")

In [ ]:
import shutil
src = "/kaggle/input/datasets/katerynaglin/hgctm-stage0/lda_sweep_results.csv"
dst = "/kaggle/working/lda_sweep_results.csv"
shutil.copy(src, dst)
print("✓ Copied lda_sweep_results.csv from Stage 0 bundle")

## ROBUSTNESS CHECK — resolved lithology / resolved age subsets

Goal:
Refit final M7b model on:
(A) deposits with resolved lithology only
(B) deposits with resolved lithology + resolved age
Then compare effective lithology vs age deviation magnitudes.

Requires existing objects from earlier cells:
X_fc_f, cov_f, phi_m1, HGCTM_Sparse, fit_m7b

In [ ]:
import numpy as np
import pandas as pd
import torch
import pyro
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def build_subset(X_base, cov_base, require_litho=True, require_age=False):
    keep = pd.Series(True, index=cov_base.index)

    if require_litho:
        keep &= cov_base["litho_class"].fillna("unknown").astype(str).str.lower().ne("unknown")

    if require_age:
        keep &= cov_base["age_category"].fillna("Unknown").astype(str).ne("Unknown")

    X_sub = X_base.loc[keep].copy()
    cov_sub = cov_base.loc[keep].copy()

    # Re-encode covariates on the subset
    le_litho = LabelEncoder()
    cov_sub["litho_idx"] = le_litho.fit_transform(
        cov_sub["litho_class"].fillna("unknown").astype(str)
    )

    le_age = LabelEncoder()
    cov_sub["age_idx"] = le_age.fit_transform(
        cov_sub["age_category"].fillna("Unknown").astype(str)
    )

    X_tensor_sub = torch.tensor(X_sub.values, dtype=torch.float32)
    litho_tensor_sub = torch.tensor(cov_sub["litho_idx"].values, dtype=torch.long)
    age_tensor_sub = torch.tensor(cov_sub["age_idx"].values, dtype=torch.long)

    return X_sub, cov_sub, X_tensor_sub, litho_tensor_sub, age_tensor_sub


def extract_effect_strengths_from_store():
    delta_litho = pyro.param("delta_litho_loc").detach().cpu().numpy()  # (K, L, F)
    delta_age = pyro.param("delta_age_loc").detach().cpu().numpy()      # (K, A, F)

    tau_a = pyro.param("tau_a").detach().cpu().numpy()
    tau_b = pyro.param("tau_b").detach().cpu().numpy()
    tau = tau_a / (tau_a + tau_b)

    omega_a_a = pyro.param("omega_a_a").item()
    omega_a_b = pyro.param("omega_a_b").item()
    omega_a = omega_a_a / (omega_a_a + omega_a_b)

    # Effective magnitudes used in the manuscript logic
    litho_eff = np.mean(np.abs(delta_litho) * tau[None, None, :])
    age_eff = np.mean(np.abs(delta_age) * tau[None, None, :] * omega_a)

    ratio = litho_eff / max(age_eff, 1e-12)

    kappa = float(np.exp(pyro.param("log_kappa_loc").item()))

    return {
        "omega_a": float(omega_a),
        "kappa": kappa,
        "litho_eff": float(litho_eff),
        "age_eff": float(age_eff),
        "ratio_litho_over_age": float(ratio),
        "tau_mean": float(np.mean(tau)),
        "tau_median": float(np.median(tau)),
    }


def fit_m7b_on_subset(X_sub, cov_sub, X_tensor_sub, litho_tensor_sub, age_tensor_sub,
                      phi_m1_ref, n_steps=5000, anneal_end=1500, lr=0.005):
    K = phi_m1_ref.shape[0]
    F = X_sub.shape[1]
    L_sub = cov_sub["litho_idx"].nunique()
    A_sub = cov_sub["age_idx"].nunique()

    model = HGCTM_Sparse(
        K=K, F=F, L=L_sub, A=A_sub,
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0,
        sigma_litho=0.3,
        sigma_age=0.15,
    )

    losses = fit_m7b(
        model,
        X_tensor_sub,
        litho_tensor_sub,
        age_tensor_sub,
        phi_m1_ref=phi_m1_ref,
        n_steps=n_steps,
        anneal_end=anneal_end,
        lr=lr,
        print_every=1000,
    )

    stats = extract_effect_strengths_from_store()
    stats["final_loss"] = float(losses[-1])
    stats["elbo_avg_last_100"] = float(np.mean(losses[-100:])) if len(losses) >= 100 else float(np.mean(losses))

    return stats


# ------------------------------------------------------------
# Run checks
# ------------------------------------------------------------
# You can reduce n_steps to 2000 for a quick test, then rerun at 5000 for final numbers.
N_STEPS_CHECK = 5000
ANNEAL_END_CHECK = 1500
LR_CHECK = 0.005

results = []

subset_specs = [
    ("resolved_lithology_only", True, False),
    ("resolved_lithology_and_age", True, True),
]

for subset_name, require_litho, require_age in subset_specs:
    print("\n" + "=" * 72)
    print(f"ROBUSTNESS CHECK: {subset_name}")
    print("=" * 72)

    X_sub, cov_sub, X_t_sub, lith_t_sub, age_t_sub = build_subset(
        X_fc_f, cov_f,
        require_litho=require_litho,
        require_age=require_age
    )

    print(f"Deposits: {X_sub.shape[0]}")
    print(f"Families: {X_sub.shape[1]}")
    print("\nLithology distribution:")
    print(cov_sub["litho_class"].value_counts(dropna=False).to_string())

    print("\nAge distribution:")
    print(cov_sub["age_category"].value_counts(dropna=False).to_string())

    stats = fit_m7b_on_subset(
        X_sub, cov_sub, X_t_sub, lith_t_sub, age_t_sub,
        phi_m1_ref=phi_m1,
        n_steps=N_STEPS_CHECK,
        anneal_end=ANNEAL_END_CHECK,
        lr=LR_CHECK,
    )

    row = {
        "subset": subset_name,
        "n_deposits": X_sub.shape[0],
        "n_families": X_sub.shape[1],
        "omega_a": stats["omega_a"],
        "kappa": stats["kappa"],
        "tau_mean": stats["tau_mean"],
        "tau_median": stats["tau_median"],
        "litho_eff": stats["litho_eff"],
        "age_eff": stats["age_eff"],
        "ratio_litho_over_age": stats["ratio_litho_over_age"],
        "elbo_avg_last_100": stats["elbo_avg_last_100"],
    }
    results.append(row)

results_df = pd.DataFrame(results)

print("\n" + "=" * 72)
print("ROBUSTNESS SUMMARY")
print("=" * 72)
print(results_df.to_string(index=False))

# Optional save
out_path = "/kaggle/working/resolved_lithology_robustness_check.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")